# Kütüphaneler

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas.io.formats.excel as pife
import warnings
import pickle
import math
import os
import locale
from termcolor import colored
from pandas.errors import SettingWithCopyWarning
from scipy.stats import zscore
from datetime import datetime

# Ayarlar

In [2]:
scope = 12
skip_last_month = True
# target_start_term  = '2022.09'
# target_end_term  = '2023.09'
target_start_term, target_end_term = None, None

os.system('color')
locale.setlocale(locale.LC_ALL, 'tr_TR')
pd.options.display.float_format = '{:,.2f}'.format
pife.ExcelFormatter.header_style = None

warnings.filterwarnings('ignore', category = UserWarning)
warnings.filterwarnings('ignore', category = pd.errors.ParserWarning)
warnings.filterwarnings('ignore', category = SettingWithCopyWarning)

base_path = 'C:\\Users\\105617\\Desktop\\Workspace\\Panel\\'

pickle_folder_path = base_path + 'pickle' + '\\'
data_folder_path = base_path + 'data' + '\\'
output_folder_path = base_path + 'output' + '\\'
kr202_folder_path = data_folder_path + 'KR202' + '\\'
ms_folder_path = data_folder_path + 'Müşteri Sınıfı' + '\\'
proje_folder_path = data_folder_path + 'Proje Kredileri' + '\\'
memzuc_folder_path = data_folder_path + 'Memzuç' + '\\'
yz_folder_path = data_folder_path + 'Yapay Zeka' + '\\'
eds_folder_path = data_folder_path + 'Entegre Derece Skor' + '\\'
fg_folder_path = data_folder_path + 'Finansal Güçlük' + '\\'
kroa_folder_path = data_folder_path + 'KRÖA' + '\\'
lgd_folder_path = data_folder_path + 'LGD' + '\\'
bcek_folder_path = data_folder_path + 'Bankamız Çek' + '\\'
dbcek_folder_path = data_folder_path + 'Diğer Banka Çek' + '\\'
ktcs_folder_path = data_folder_path + 'KTÇS' + '\\'
senet_folder_path = data_folder_path + 'Senet' + '\\'

for p in [
    data_folder_path, output_folder_path, pickle_folder_path,
    kr202_folder_path, ms_folder_path, proje_folder_path, memzuc_folder_path,
    yz_folder_path, fg_folder_path, kroa_folder_path, lgd_folder_path,
    bcek_folder_path, dbcek_folder_path, ktcs_folder_path, senet_folder_path,
]:
    os.makedirs(p, exist_ok = True)
    
mali_file_name = 'Mali Tablolar.xlsx'
zombi_mali_file_name = 'Mali Tablolar - Zombi.xlsx'

proje_file_name = 'Proje Kredileri.csv'
kur_file_name = 'Kur Riski Çalışması.csv'
sektor_file_name = 'Sektör Risklilik Çalışması.csv'
nace_file_name = 'NACE Bilgileri.csv'

sube_file_name = 'Şube Bilgileri.xlsx'
mnvt_file_name = 'Müşteri No - VKN - TCKN.csv'

# Yardımcı Fonksiyonlar

## Tablo Manipülasyonu

In [3]:
def add_h_space(dfo, space=1):
    df = dfo.copy()
    w = df.shape[1]
    for _ in range(space):
        df.loc[len(df)] = [np.nan] * w
    return df

def add_v_space(dfo, space=1):
    df = dfo.copy()
    return pd.concat([df, pd.DataFrame([[np.nan] * space], columns=[np.nan] * space)], axis=1)
    

def h_stack(dfo, space=3):
    df = dfo.copy()
    df = df[0].reset_index(drop=True)
    dfln = [df]
    dfna = pd.DataFrame({np.nan: [np.nan]})
    for dftt in dfo[1:]:
        dft = dftt.copy()
        for i in range(space):
            dfln.append(dfna)
        dfln.append(dft.reset_index(drop=True))
    df = pd.concat(dfln, axis=1).reset_index(drop=True)
    return df


def v_stack(dfo, space=3):
    df = dfo.copy()
    df = df[0].reset_index(drop=True)
    dfln = [df]
    dfna = pd.DataFrame(columns=df.columns)

    dfna = add_h_space(dfna, space)
    for dftt in dfo[1:]:
        dft = dftt.copy()
        l = df.shape[1]
        lt = dft.shape[1]
        diff = l - lt

        if diff > 0:
            dft = add_v_space(dft, diff)
        elif diff < 0:
            df = add_v_space(df, -diff)

        dfc = pd.DataFrame([dft.columns], columns=df.columns)
        dft.columns = df.columns
        dfln.append(dfna)
        dfln.append(dfc)
        dfln.append(dft.reset_index(drop=True))

    df = pd.concat(dfln).reset_index(drop=True)
    return df


def insert_header(dfo, col):
    df = dfo.copy()
    header = [col] if type(col) is str else col
    return v_stack([pd.DataFrame(columns=header), df], 0)

def insert_corner(dfo):
    df = dfo.copy()
    return h_stack([pd.DataFrame(), v_stack([pd.DataFrame(), df], 0)], 1)

## Verim Tabloları

In [4]:
def fix_verim_table(df, columns=None):
    start = 0
    if columns is None:
        start = 3
        columns = df.iloc[2, 1:]
    df = df.iloc[start:, 1:]
    df.columns = columns
    df.columns.name = None
    df = df.reset_index(drop=True)
    return df

def quick_export_verim_csv(data, output_file_name, sep=';', header=False, index=False, open_file=False):
    cl = [str(x) for x in range(0, 16)]
    df = pd.DataFrame()
    df['10'] = data
    for c in ['0', '15']:
        df[c] = 'x'
    for c in cl:
        if c not in ['0', '10', '15']:
            df[c] = np.nan
    df = df[cl]

    output_file_path = output_folder_path + output_file_name + '.csv'
    df.to_csv(output_file_path, sep=sep, header=header, index=index)

    if open_file:
        os.startfile(output_file_path)
    

## Hızlı Çıktı

In [5]:
def quick_export(df_export, output_file_name, sheet_name=None, open_file=True):
    output_file_path = output_folder_path + output_file_name + '.xlsx'
    writer = pd.ExcelWriter(output_file_path, engine = 'xlsxwriter')
    if type(df_export) is list:
        if sheet_name is None:
            sheet_name = [x + 1 for x in range(len(df_export))]
        for df, sheet in zip(df_export, sheet_name):
            df.to_excel(writer, sheet_name = sheet, index = False)
    else:
        if sheet_name is None:
            sheet_name = output_file_name
        df_export.to_excel(writer, sheet_name = sheet_name, index = False)
    writer.close()
    if open_file:
        os.startfile(output_file_path)


def quick_export_csv(df, file_name, open_file=False):
    df.to_csv(output_folder_path + file_name  + '.csv', sep=';', encoding='ANSI', index=False)

## Now

In [6]:
def now():
    # return datetime.strftime(datetime.now(), '%Y-%m-%d_%H-%M-%S')
    return datetime.strftime(datetime.now(), '%m%d%H%M')

## Pickle

In [7]:
def save_pickle(file, name, sub=''):
    os.makedirs(pickle_folder_path + sub, exist_ok = True)
    with open(pickle_folder_path + sub + name, 'wb') as handle:
        pickle.dump(file, handle, protocol=pickle.HIGHEST_PROTOCOL)

def load_pickle(name, sub=''):
    with open(pickle_folder_path + sub + name, 'rb') as handle:
        return pickle.load(handle)

## Toplu Dosya İşleme

In [8]:
def scan_folder(folder_path, extension='.xlsx', offset=0):
    file_list, date_list = [], []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(extension) and not file_name.startswith('~$'):
            s = file_name.split('.')
            y = s[0][-4:]
            m = s[1]
            d = s[2][:2]
            file_list.append(file_name)
            date_list.append(y + '.' + m + '.' + d)

    sorted_list = sorted(zip(date_list, file_list))

    if target_end_term is not None:
        sorted_list = sorted_list[:[x[0][:7] for x in sorted_list].index(target_end_term) + 1 + offset]
    elif skip_last_month:
        sorted_list = sorted_list[:-1]

    if target_start_term is not None:
        sorted_list = sorted_list[[x[0][:7] for x in sorted_list].index(target_end_term) + offset:]
    else:
        sorted_list = sorted_list[-scope:]

    return file_list, date_list, sorted_list

In [9]:
def batch_import_files(sorted_list, data_folder_path, pickle_sub, pickle_name_base, fix_verim=False, csv=None, sheet_list=None):
    sorted_date_list, df_raw_list = [], []
    pickle_sub += '\\'
    pickle_name_base += '_'
    l = len(sorted_list)
    for i, (date, name) in enumerate(sorted_list):
        sorted_date_list.append(date[:7])
        pickle_name = pickle_name_base + date
        if os.path.exists(pickle_folder_path + pickle_sub + pickle_name):
            df = load_pickle(pickle_name, pickle_sub)
            ps = f'({round((i + 1) / l * 100)}%)'
            ps = ' ' * (len('(100%)') - len(ps)) + ps
            print(colored(name, 'light_green'), 'önbellekten okundu', colored(ps, 'light_green'))
        else:
            print(colored(name, 'light_green'), 'okunuyor')
            if csv is not None:
                sep, encoding, low_memory, index_col = ';', 'ANSI', False, True
                if type(csv) is list:
                    try:
                        sep = csv[0]
                        encoding = csv[1]
                        low_memory = csv[2]
                        index_col = csv[3]
                    except:
                        pass
                if index_col:
                    df = pd.read_csv(data_folder_path + name, sep=sep, encoding=encoding, low_memory=low_memory)
                else:
                    df = pd.read_csv(data_folder_path + name, sep=sep, encoding=encoding, low_memory=low_memory, index_col=index_col)

            else:
                xls = pd.ExcelFile(data_folder_path + name)
                df = pd.DataFrame()
                if sheet_list is None:
                    sheet_list = xls.sheet_names
                for sheet in sheet_list:
                    print('\t', colored(sheet, 'light_cyan'), 'okunuyor')
                    dfs = pd.read_excel(xls, sheet_name=sheet)
                    if fix_verim:
                        dfs = fix_verim_table(dfs)
                    df = pd.concat([df, dfs], ignore_index=True) if len(sheet_list) > 1 else dfs
                xls.close()
            
                print(colored(name, 'light_green'), 'kaydediliyor')
                save_pickle(df, pickle_name, pickle_sub)
                ps = f'({round((i + 1) / l * 100)}%)'
                ps = ' ' * (len('(100%)') - len(ps)) + ps
                print(colored(name, 'light_green'), 'kaydedildi', colored(ps, 'light_green'))

        df_raw_list.append(df.copy())
    print('Tüm dosyalar başarıyla okundu')

    return sorted_date_list, df_raw_list

In [10]:
def batch_process_files(df_raw_list, sorted_list, sorted_date_list, fix_df_function, column_list=None, fill_string=None, rename_exclude_columns = 'Müşteri No'):
    df_all = []
    df = pd.DataFrame()
    l = len(df_raw_list)
    for i, dfi in enumerate(df_raw_list):
        df = dfi.copy()
        
        if fix_df_function is not None:
            df = fix_df_function(df, i)

        if i == len(df_raw_list) - 1:
            df_last = df.copy().reset_index(drop=True)

        if column_list is not None:  
            df = df[column_list]
     
        if type(rename_exclude_columns) is str:
            df.columns = [c + ' ' + sorted_date_list[i] if c != rename_exclude_columns else c for c in df.columns]
        else:
            df.columns = [c + ' ' + sorted_date_list[i] if c not in rename_exclude_columns else c for c in df.columns]
        df = df.sort_values('Müşteri No').reset_index(drop=True)

        if fill_string is not None:
            df = df.fillna(fill_string)

        df_all.append(df)        
        ps = f'({round((i + 1) / l * 100)}%)'
        ps = ' ' * (len('(100%)') - len(ps)) + ps
        print(colored(sorted_list[i][1], 'light_green'), 'işlendi', colored(ps, 'light_green'))

    print('Tüm dosyalar başarıyla oluşturuldu')

    return df_all, df_last

# Tekil Dosyalar

## Mali Tablolar

In [11]:
if os.path.exists(pickle_folder_path + 'df_mali'):
    df_mali = load_pickle('df_mali')
else:
    xls = pd.ExcelFile(data_folder_path + mali_file_name)
    df_mali = pd.read_excel(xls)
    save_pickle(df_mali, 'df_mali')
    xls.close()
df_mali_backup = df_mali.copy()

## Zombi Firmalar Mali Tablolar

In [12]:
if os.path.exists(pickle_folder_path + 'df_zombi_mali'):
    df_zombi_mali = load_pickle('df_zombi_mali')
else:
    xls = pd.ExcelFile(data_folder_path + zombi_mali_file_name)
    df_zombi_mali = pd.read_excel(xls)
    save_pickle(df_zombi_mali, 'df_zombi_mali')
    xls.close()
df_zombi_mali_backup = df_zombi_mali.copy()

## Kur Riski Çalışması

In [13]:
if os.path.exists(pickle_folder_path + 'df_kur'):
    df_kur = load_pickle('df_kur')
else:
    df_kur = pd.read_csv(data_folder_path + kur_file_name, sep=';', encoding='utf-8')
    save_pickle(df_kur, 'df_kur')

## Sektör Risklilik Çalışması

In [14]:
if os.path.exists(pickle_folder_path + 'df_sektor'):
    df_sektor = load_pickle('df_sektor')
else:
    df_sektor = pd.read_csv(data_folder_path + sektor_file_name, sep=';', encoding='ANSI')
    save_pickle(df_sektor, 'df_sektor')

## Şube Bilgileri

In [15]:
if os.path.exists(pickle_folder_path + 'df_sube'):
    df_sube = load_pickle('df_sube')
else:
    xls = pd.ExcelFile(data_folder_path + sube_file_name)
    df_sube = pd.read_excel(xls)
    save_pickle(df_sube, 'df_sube')
    xls.close()

## NACE Bilgileri

In [16]:
if os.path.exists(pickle_folder_path + 'df_nace'):
    df_nace = load_pickle('df_nace')
else:
    df_nace = pd.read_csv(data_folder_path + nace_file_name, encoding='ANSI')
    for c in df_nace.columns:
        df_nace[c] = df_nace[c].apply(lambda x: x.strip())
    save_pickle(df_nace, 'df_nace')

## Müşteri No - VKN - TCKN

In [17]:
if os.path.exists(pickle_folder_path + 'df_mnvt'):
    df_mnvt = load_pickle('df_mnvt')
else:
    df_mnvt = pd.read_csv(data_folder_path + mnvt_file_name, sep=';')
    save_pickle(df_mnvt, 'df_mnvt')

df_mnvt = df_mnvt.loc[(df_mnvt['TCKN'].notnull()) | (df_mnvt['VKN'].notnull())]

# Veri İçe Alma

## KR202

In [18]:
kr202_file_list, kr202_date_list, kr202_sorted_list = scan_folder(kr202_folder_path)
kr202_sorted_date_list, kr202_df_raw_list = batch_import_files(kr202_sorted_list, kr202_folder_path, 'KR202', 'kr202')

last_date = kr202_sorted_list[-1][0][:7]

KR202 - 2022.12.31.xlsx önbellekten okundu   (8%)
KR202 - 2023.01.31.xlsx önbellekten okundu  (17%)
KR202 - 2023.02.28.xlsx önbellekten okundu  (25%)
KR202 - 2023.03.31.xlsx önbellekten okundu  (33%)
KR202 - 2023.04.30.xlsx önbellekten okundu  (42%)
KR202 - 2023.05.31.xlsx önbellekten okundu  (50%)
KR202 - 2023.06.30.xlsx önbellekten okundu  (58%)
KR202 - 2023.07.31.xlsx önbellekten okundu  (67%)
KR202 - 2023.08.31.xlsx önbellekten okundu  (75%)
KR202 - 2023.09.30.xlsx önbellekten okundu  (83%)
KR202 - 2023.10.31.xlsx önbellekten okundu  (92%)
KR202 - 2023.11.30.xlsx önbellekten okundu (100%)
Tüm dosyalar başarıyla okundu


In [19]:
# # for df, date in zip(kr202_df_raw_list, kr202_sorted_date_list):
# #     cl = list(df.loc[df['Musteri No'].notnull(), 'Musteri No'])
# #     quick_export_verim_csv(cl, 'Müşteri Listesi - ' + date)

# df = kr202_df_raw_list[-1]
# date = '2023.11'
# cl = list(df.loc[df['Musteri No'].notnull(), 'Musteri No'])
# quick_export_verim_csv(cl, 'Müşteri Listesi - ' + date)

In [20]:
kr202_column_list = [
    'Müşteri No',
    'Şube Kodu',
    # 'Şube Türü',
    'İlgili Bölüm',
    'Yetki Kodu',
    'NACE Kodu',
    'NACE Harf',
    'Tahsis Sektör Adı',
    'Nakdi TL Reeskontlu',
    'Nakdi YP Reeskontlu',
    'Nakdi Reeskontlu Toplam',
    'G.Nakdi TL Bakiye',
    'G.Nakdi YP Bakiye',
    'G.Nakdi Toplam',
    'Toplam Reeskontlu Risk',
]

kr202_initial_column_list = [x for x in kr202_column_list if x != 'NACE Harf']

kr202_include_list = [
    'Müşteri No', 'Adı', 'Şube Kodu', 'Şube Adı', 'Şube İl', 'Şube İlçe',
    'Şube Türü', 'Yetki Kodu', 'İlgili Bölüm', 'Risk Grubu Kodu', 'Risk Grubu Adı',
    'NACE Kodu', 'Tahsis Sektör Adı', 'Tahsis Alt Sektör Adı',
    'Nakdi TL Bakiye', 'Nakdi YP Bakiye', 'Nakdi Toplam',
    'G.Nakdi TL Bakiye', 'G.Nakdi YP Bakiye', 'G.Nakdi Toplam', 'Toplam Risk',
    'Nakdi TL Reeskontlu', 'Nakdi YP Reeskontlu', 'Nakdi Reeskontlu Toplam', 'Toplam Reeskontlu Risk'
]

kr202_float_list = [
    'Müşteri No', 'Şube Kodu', 'Yetki Kodu',
    'Nakdi TL Bakiye', 'Nakdi YP Bakiye', 'Nakdi Toplam',
    'G.Nakdi TL Bakiye', 'G.Nakdi YP Bakiye', 'G.Nakdi Toplam', 'Toplam Risk',
    'Nakdi TL Reeskontlu', 'Nakdi YP Reeskontlu', 'Nakdi Reeskontlu Toplam', 'Toplam Reeskontlu Risk'
]

kr202_column_rename_dict = {
	'MUSTERI_NO': 'Müşteri No',
    'Musteri No': 'Müşteri No',
	'ADI': 'Adı',
    'Adi': 'Adı',
	'SANTRAL_SUBE_KODU': 'Şube Kodu',
    'SANTRAL SUBE KODU': 'Şube Kodu',
    'Santral Şube Kodu': 'Şube Kodu',
	'SANTRAL_SUBE_ADI': 'Şube Adı',
    'Santral Şube Adı': 'Şube Adı',
	'SANTRAL_SUBE_ILI': 'Şube İl',
    'SANTRAL_SUBE_IL': 'Şube İl',
    'SANTRAL ŞUBE İL': 'Şube İl',
    'Santral  Şube İl': 'Şube İl',
	'SANTRAL_SUBE_ILCE': 'Şube İlçe',
    'SANTRAL_SUBE_ILCE': 'Şube İlçe',
    'SANTRAL ŞUBE İLÇE': 'Şube İlçe',
    'Santral Şube İlçe': 'Şube İlçe',
	'SANTRAL_SUBE_TURU': 'Şube Türü',
    'Santral Şube Türü': 'Şube Türü',
	'YETKI_KODU': 'Yetki Kodu',
	'BOLUM_ADI': 'İlgili Bölüm',
	'RISK_GRUBU_KODU': 'Risk Grubu Kodu',
	'RISK_GRUBU_ADI': 'Risk Grubu Adı',
	'NACE_KODU': 'NACE Kodu',
    'Nace Kodu': 'NACE Kodu',
	'TAHSIS_SEKTOR_ADI': 'Tahsis Sektör Adı',
    'SEKTÖR': 'Tahsis Sektör Adı',
	'TAHSIS_ALT_SEKTOR_ADI': 'Tahsis Alt Sektör Adı',
    'Tahsis Alt Sektör AdI': 'Tahsis Alt Sektör Adı',
    'ALT SEKTÖR': 'Tahsis Alt Sektör Adı',
    'NAKDI_TL_BAK': 'Nakdi TL Bakiye',
	'NAKDI_YP_BAK': 'Nakdi YP Bakiye',
	'NAKDI_TOPLAM': 'Nakdi Toplam',
	'GNAKDI_TL_BAK': 'G.Nakdi TL Bakiye',
	'GNAKDI_YP_BAK': 'G.Nakdi YP Bakiye',
	'GNAKDI_TOPLAM': 'G.Nakdi Toplam',
	'NAKDI_TL_REESKONT': 'Nakdi TL Reeskontlu',
	'NAKDI_YP_REESKONT': 'Nakdi YP Reeskontlu',
	'NAKDI_REESKONTLU_TOPLAM': 'Nakdi Reeskontlu Toplam',
}

kr202_sube_rename_dict = {
    'KKTC Şubeleri': 'Kıbrıs',
    'Karma Şubeler': 'Karma',
    'Kurumsal Şubeler': 'Kurumsal',
    'Ticari İhtisas Şubeleri': 'Ticari',
    'Y.dışı': 'Yurt Dışı',
    'Serbest Bölge Şubeleri': 'Serbest Bölge',
    'Özel Bankacılık Şubeleri': 'Özel Bankacılık',
}

kr202_bolum_rename_dict = {
    'KKTB': 'Kurumsal Krediler Tahsis Bölümü',
    'Kurumsal Tahsis': 'Kurumsal Krediler Tahsis Bölümü',
    'Kurumsal Krediler Tahsis': 'Kurumsal Krediler Tahsis Bölümü',
    'Kurumsal Kredi Krediler Tahsis': 'Kurumsal Krediler Tahsis Bölümü',
    'Kurumsal Kredi Krediler Tahsis Bölümü': 'Kurumsal Krediler Tahsis Bölümü',
    'TKTB': 'Ticari Krediler Tahsis Bölümü',
    'Ticari Tahsis': 'Ticari Krediler Tahsis Bölümü',
    'Ticari Krediler Tahsis': 'Ticari Krediler Tahsis Bölümü',
    'Ticari Kredi Krediler Tahsis': 'Ticari Krediler Tahsis Bölümü',
    'Ticari Kredi Krediler Tahsis Bölümü': 'Ticari Krediler Tahsis Bölümü',
    'Perakende Tahsis': 'Perakende Krediler Tahsis Bölümü',
    'Krediler İzleme': 'Krediler İzleme Bölümü',
    'Takip': 'Ticari ve Kurumsal Krediler Takip Bölümü',
    'Proje Finansmanı': 'Proje Finansmanı Bölümü'
}

kr202_tsa_rename_dict = {
    'İNŞAAT': 'İNŞAAT VE TAAHHÜT',
    'İNŞAAT TAAHHÜT': 'İNŞAAT VE TAAHHÜT',
    'ULAŞTIRMA DEPOLAMA': 'ULAŞTIRMA VE DEPOLAMA',
    'OTOMOTİV ULAŞIM ARAÇLARI': 'OTOMOTİV VE ULAŞIM ARAÇLARI',
    'DİĞER HİZMET': 'DİĞER HİZMET FAALİYETLERİ',
    'ELEKTRONİK BİLİŞİM': 'ELEKTRONİK VE BİLİŞİM',
    'GEMİCİLİK TERSANECİLİK': 'GEMİCİLİK VE TERSANECİLİK',
    'SAVUNMA SANAYİ': 'SAVUNMA SANAYİİ',
    'PPP (Kamu Garantili KÖİ) (HASTANE VE OTOYOLLAR)': 'PPP (HASTANE VE OTOYOLLAR)',
    'RAFİNERİ VE YAKIT TİCARETİ (AKARYAKIT, PETROL, KÖMÜR, GAZ)': 'RAFİNERİ VE YAKIT TİCARETİ',
    'MAKİNE VE TEÇHİZAT SANAYİİ VE TİCARETİ (savaş araçları imalatı hariç)': 'MAKİNE VE TEÇHİZAT SANAYİİ VE TİCARETİ',
    'BELEDİYELER VE OSBler': 'BELEDİYELER VE OSBLER',
    'BELEDİYELER VE OSB ler': 'BELEDİYELER VE OSBLER',
    'BELEDİYELER VE OSBLER': 'BELEDİYELER VE OSBLER',
    'BELEDİYELER VE OSB LER': 'BELEDİYELER VE OSBLER',
    'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMler)': 'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMLER)',
    'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVM ler)': 'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMLER)',
    'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMLER)': 'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMLER)',
    'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVM LER)': 'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMLER)',
    'nan': np.nan,
    '-': np.nan,
}


def fix_kr202_df(df, i):
    df.columns = [x.strip() for x in df.columns]
    for x in kr202_column_rename_dict:
        try:
            df = df.rename(columns={x: kr202_column_rename_dict[x]})
        except:
            pass
    
    for x in kr202_float_list:
        if x in df.columns and df[x].dtype == 'O':
            df[x] = df[x].apply(lambda x: float(str(x).replace(',', '.')))


    if 'Toplam Risk' not in df.columns:
        df['Toplam Risk'] = df['Nakdi Toplam'] + df['G.Nakdi Toplam']
    if 'Toplam Reeskontlu Risk' not in df.columns:
        df['Toplam Reeskontlu Risk'] = df['Nakdi Reeskontlu Toplam'] + df['G.Nakdi Toplam']

    if i == len(kr202_df_raw_list) - 1:
        df = df.loc[:, df.columns.isin(kr202_include_list)]
    else:
        df = df[kr202_initial_column_list]

    df = df.loc[df['Müşteri No'].notnull()]
    df['İlgili Bölüm'] = df['İlgili Bölüm'].map(lambda x: kr202_bolum_rename_dict.get(x, x))
    df['Tahsis Sektör Adı'] = df['Tahsis Sektör Adı'].fillna('-').apply(lambda x: x.strip().replace('\'', '').replace('  ', ' '))
    df['Tahsis Sektör Adı'] = df['Tahsis Sektör Adı'].map(lambda x: kr202_tsa_rename_dict.get(x, x))
    df['NACE Kodu'] = df['NACE Kodu'].apply(lambda x: x.strip() if type(x) is str else np.nan)
    df['NACE Harf'] = df['NACE Kodu'].apply(lambda x: x[0] if type(x) is str else np.nan)

    if 'Şube Türü' in df.columns:
        current_sube_tur_list = df['Şube Türü'].unique()
        if 'Kurumsal' in current_sube_tur_list and 'Kurumsal Şubeler' in current_sube_tur_list:
            df['Şube Türü'] = df['Şube Türü'].map(lambda x: {'Kurumsal': 'KKTB'}.get(x, x))
        df['Şube Türü'] = df['Şube Türü'].map(lambda x: kr202_sube_rename_dict.get(x, x))

    return df

    
df_kr202_all, df_kr202_last = batch_process_files(kr202_df_raw_list, kr202_sorted_list, kr202_sorted_date_list, fix_kr202_df, kr202_column_list)


customer_list = []
for df in df_kr202_all:
    customer_list.append(list(df['Müşteri No']))

df_kr202_last_original_backup = df_kr202_last.copy()

KR202 - 2022.12.31.xlsx işlendi   (8%)
KR202 - 2023.01.31.xlsx işlendi  (17%)
KR202 - 2023.02.28.xlsx işlendi  (25%)
KR202 - 2023.03.31.xlsx işlendi  (33%)
KR202 - 2023.04.30.xlsx işlendi  (42%)
KR202 - 2023.05.31.xlsx işlendi  (50%)
KR202 - 2023.06.30.xlsx işlendi  (58%)
KR202 - 2023.07.31.xlsx işlendi  (67%)
KR202 - 2023.08.31.xlsx işlendi  (75%)
KR202 - 2023.09.30.xlsx işlendi  (83%)
KR202 - 2023.10.31.xlsx işlendi  (92%)
KR202 - 2023.11.30.xlsx işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Müşteri Sınıfı

In [21]:
ms_file_list, ms_date_list, ms_sorted_list = scan_folder(ms_folder_path, '.csv')
ms_sorted_date_list, ms_df_raw_list = batch_import_files(ms_sorted_list, ms_folder_path, 'Müşteri Sınıfı', 'ms', csv=[';'])

MS - 2022.12.31.csv önbellekten okundu   (8%)
MS - 2023.01.31.csv okunuyor
MS - 2023.02.28.csv okunuyor
MS - 2023.03.31.csv okunuyor
MS - 2023.04.30.csv okunuyor
MS - 2023.05.31.csv okunuyor
MS - 2023.06.30.csv okunuyor
MS - 2023.07.31.csv okunuyor
MS - 2023.08.31.csv okunuyor
MS - 2023.09.30.csv okunuyor
MS - 2023.10.31.csv okunuyor
MS - 2023.11.30.csv okunuyor
Tüm dosyalar başarıyla okundu


In [22]:
ms_column_list = ['MUSTERI_NO', 'MUSTERI_SINIFI']

renamed_ms_column_list = ['Müşteri No', 'Müşteri Sınıfı']

def fix_ms_df(df, i):
    df = df.loc[df['MUSTERI_NO'].isin(customer_list[i]), ms_column_list]
    df.columns = renamed_ms_column_list
    df['Müşteri Sınıfı'] = df['Müşteri Sınıfı'].apply(lambda x: int(x[-1]))
    return df


df_ms_all, df_ms_last = batch_process_files(ms_df_raw_list, ms_sorted_list, ms_sorted_date_list, fix_ms_df)

MS - 2022.12.31.csv işlendi   (8%)
MS - 2023.01.31.csv işlendi  (17%)
MS - 2023.02.28.csv işlendi  (25%)
MS - 2023.03.31.csv işlendi  (33%)
MS - 2023.04.30.csv işlendi  (42%)
MS - 2023.05.31.csv işlendi  (50%)
MS - 2023.06.30.csv işlendi  (58%)
MS - 2023.07.31.csv işlendi  (67%)
MS - 2023.08.31.csv işlendi  (75%)
MS - 2023.09.30.csv işlendi  (83%)
MS - 2023.10.31.csv işlendi  (92%)
MS - 2023.11.30.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Proje

In [23]:
proje_file_list, proje_date_list, proje_sorted_list = scan_folder(proje_folder_path, '.csv')
proje_sorted_date_list, proje_df_raw_list = batch_import_files(proje_sorted_list, proje_folder_path, 'Proje Kredileri', 'proje', csv=[';', 'ANSI', False, False])

Proje - 2023.12.31.csv okunuyor
Tüm dosyalar başarıyla okundu


In [24]:
proje_column_list = [' MUS_NO ']

renamed_proje_column_list = ['Müşteri No']

def fix_proje_df(df, i):
    df = df.loc[df[' MUS_NO '].isin(customer_list[i]), proje_column_list]
    df.columns = renamed_proje_column_list
    df = df.drop_duplicates(keep='first')
    df['Proje Kredisi'] = 1
    return df


df_proje_all, df_proje_last = batch_process_files(proje_df_raw_list, proje_sorted_list, proje_sorted_date_list, fix_proje_df)

Proje - 2023.12.31.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Memzuç

In [25]:
memzuc_file_list, memzuc_date_list, memzuc_sorted_list = scan_folder(memzuc_folder_path, '.csv', -1)
memzuc_sorted_date_list, memzuc_df_raw_list = batch_import_files(memzuc_sorted_list, memzuc_folder_path, 'Memzuç', 'memzuc', csv=[';'])

Memzuç - 2023.11.01.csv okunuyor
Tüm dosyalar başarıyla okundu


In [26]:
memzuc_column_list = [
    'MUS_NO', 'BANKA_SAY',
    'SEKTOR_LIMIT', 'SEKTOR_RISK', 'BANKAMIZ_LIMIT', 'BANKAMIZ_RISK',
    'SEKTOR_LIMIT_NAKDI', 'SEKTOR_RISK_NAKDI', 'BANKAMIZ_LIMIT_NAKDI', 'BANKAMIZ_RISK_NAKDI',
    'SEKTOR_LIMIT_GNAKDI', 'SEKTOR_RISK_GNAKDI', 'BANKAMIZ_LIMIT_GNAKDI', 'BANKAMIZ_RISK_GNAKDI'
]

renamed_memzuc_column_list = [
    'Müşteri No', 'Banka Sayısı',
    'Memzuç Limit', 'Memzuç Risk', 'Bankamız Limit', 'Bankamız Risk',
    'Memzuç Nakdi Limit', 'Memzuç Nakdi Risk', 'Bankamız Nakdi Limit', 'Bankamız Nakdi Risk',
    'Memzuç G.Nakdi Limit', 'Memzuç G.Nakdi Risk', 'Bankamız G.Nakdi Limit', 'Bankamız G.Nakdi Risk'
]

def fix_memzuc_df(df, i):
    df = df.loc[df['MUS_NO'].isin(customer_list[i]), memzuc_column_list]
    df.columns = renamed_memzuc_column_list
    df['Sektör Limit Doluluk Oranı (%)'] = df['Memzuç Risk'] / df['Memzuç Limit']
    df['Bankamız Limit Doluluk Oranı (%)'] = df['Bankamız Risk'] / df['Bankamız Limit']
    df['Bankamız Memzuç Limit Payı (%)'] = df['Bankamız Limit'] / df['Memzuç Limit']
    df['Bankamız Memzuç Risk Payı (%)'] = df['Bankamız Risk'] / df['Memzuç Risk']
    df['Sektör Nakdi Limit Doluluk Oranı (%)'] = df['Memzuç Nakdi Risk'] / df['Memzuç Nakdi Limit']
    df['Bankamız Nakdi Limit Doluluk Oranı (%)'] = df['Bankamız Nakdi Risk'] / df['Bankamız Nakdi Limit']
    df['Bankamız Memzuç Nakdi Limit Payı (%)'] = df['Bankamız Nakdi Limit'] / df['Memzuç Nakdi Limit']
    df['Bankamız Memzuç Nakdi Risk Payı (%)'] = df['Bankamız Nakdi Risk'] / df['Memzuç Nakdi Risk']
    df.loc[:, [c for c in df.columns if c not in renamed_memzuc_column_list]] = df.loc[:, [c for c in df.columns if c not in renamed_memzuc_column_list]].clip(0, 100)   
    return df

df_memzuc_all, df_memzuc_last = batch_process_files(memzuc_df_raw_list, memzuc_sorted_list, memzuc_sorted_date_list, fix_memzuc_df)

Memzuç - 2023.11.01.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Yapay Zeka

In [27]:
yz_file_list, yz_date_list, yz_sorted_list = scan_folder(yz_folder_path)
yz_sorted_date_list, yz_df_raw_list = batch_import_files(yz_sorted_list, yz_folder_path, 'Yapay Zeka', 'yz')

Yapay Zeka - 2023.12.31.xlsx önbellekten okundu (100%)
Tüm dosyalar başarıyla okundu


In [28]:
yz_column_list = ['MUSTERI_NUMARASI', 'YZ_RISK_SINIFI', 'KM_RISK_SINIFI', 'IZLEME_RISK_SINIFI', 'MODEL', 'GECIKME_GUN_SAYISI']

yz_column_rename_dict = {
    'MUSTERI_NUMARASI': 'Müşteri No',
    'YZ_RISK_SINIFI': 'Yapay Zeka Risk Sınıfı',
    'KM_RISK_SINIFI': 'Karar Motoru Risk Sınıfı',
    'IZLEME_RISK_SINIFI': 'İzleme Risk Sınıfı',
    'MODEL': 'Gecikme',
    'GECIKME_GUN_SAYISI': 'Gecikme Gün'
}

yz_gecikme_map_dict = {
    'Erken Uyarı': 0,
    'Tahsilat': 1
}


def fix_yz_df(df, i):
    df = df.loc[df['MUSTERI_NUMARASI'].isin(customer_list[i]), yz_column_list]
    df = df.rename(columns = yz_column_rename_dict)
    df['Gecikme'] = df['Gecikme'].map(yz_gecikme_map_dict)
    return df


df_yz_all, df_yz_last = batch_process_files(yz_df_raw_list, yz_sorted_list, yz_sorted_date_list, fix_yz_df)

Yapay Zeka - 2023.12.31.xlsx işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Entegre Derece Skor

In [29]:
eds_file_list, eds_date_list, eds_sorted_list = scan_folder(eds_folder_path, '.csv')
eds_sorted_date_list, eds_df_raw_list = batch_import_files(eds_sorted_list, eds_folder_path, 'Entegre Derece Skor', 'eds', csv=[';'])

Entegre Derece Skor - 2023.12.01.csv okunuyor
Tüm dosyalar başarıyla okundu


In [30]:
eds_column_list = ['MUSTERI_NO', 'CUSTOMER_TYPE', 'FINAL_RATING']
renamed_eds_column_list = ['Müşteri No', 'Değerlendirmeye Esas Segment', 'Entegre Derece Skor']

des_map_dict = {
    'TARIM': 'İŞLETME',
    'ISLETME': 'İŞLETME',
    'KOBI': 'KOBİ',
    'TİCARİ': 'TİCARİ',
    'KURUMSAL': 'KURUMSAL',
}

def fix_eds_df(df, i):
    df = df.loc[df['MUSTERI_NO'].isin(customer_list[i]), eds_column_list]
    df.columns = renamed_eds_column_list
    df['Değerlendirmeye Esas Segment'] = df['Değerlendirmeye Esas Segment'].map(des_map_dict)
    return df

df_eds_all, df_eds_last = batch_process_files(eds_df_raw_list, eds_sorted_list, eds_sorted_date_list, fix_eds_df)

Entegre Derece Skor - 2023.12.01.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Finansal Güçlük

In [31]:
fg_file_list, fg_date_list, fg_sorted_list = scan_folder(fg_folder_path)
fg_sorted_date_list, fg_df_raw_list = batch_import_files(fg_sorted_list, fg_folder_path, 'Finansal Güçlük', 'fg', True)

Finansal Güçlük - 2023.12.01.xlsx önbellekten okundu (100%)
Tüm dosyalar başarıyla okundu


In [32]:
fg_column_list = [
    'Müşteri\nNumarası',
    'Çek\nYasaklılık',
    'Son 2 Yıldaki\nMemzuç Dönemlerinden \nHerhangi Birinde Diğer \nBanka Takip > 250 TL\n',
    'Ticari Sicil\nPaylaşım Sistemi\nİflas/Konkordato/\nİflas Erteleme',
    'Son 1 Yıldaki Memzuç\nDönemlerinden Herhangi \nBirinde Diğer Banka \nTazmin > 1000 TL',
    'Diğer Banka \nCanlı Alacaklardan \nYeniden Yapılandırma > 0',
    'Diğer Banka Canlı Alacaklardan \nYeniden Yapılandırma/\nDiğer Banka Toplam \nNakdi Risk Oranı (%)\n',
    'Bankamız\nKarşılıksız\nÇek',
    'Diğer Banka\nKarşılıksız\nÇek',
    'Bankamız ve\nDiğer Banka\nKarşılıksız Çek',
    'Diğer Banka\nTahakkuk\n>20.000',
    'Diğer Toplam Tahakkuk /\nDiğer Toplam Nakdi Risk\n> 0,01'
]

fg_new_column_list = [
    'Müşteri\nNumarası',
    'Çek\nYasaklılık',
    'Karşılıksız Çek',
    'Diğer Banka Tahakkuk',
    'Son 2 Yıldaki\nMemzuç Dönemlerinden \nHerhangi Birinde Diğer \nBanka Takip > 250 TL\n',
    'Ticari Sicil\nPaylaşım Sistemi\nİflas/Konkordato/\nİflas Erteleme',
    'Son 1 Yıldaki Memzuç\nDönemlerinden Herhangi \nBirinde Diğer Banka \nTazmin > 1000 TL',
    'Diğer Banka Yapılandırma',
    'Diğer Banka Canlı Alacaklardan \nYeniden Yapılandırma/\nDiğer Banka Toplam \nNakdi Risk Oranı (%)\n'
]

renamed_fg_column_list = [
    'Müşteri No',
    'Çek Yasaklılık',
    'Karşılıksız Çek',
    'Diğer Banka Tahakkuk',
    'Diğer Banka Takip',
    'TSPS İflas vb',
    'Diğer Banka Tazmin',
    'Diğer Banka Yapılandırma',
    'Diğer Banka Yapılandırma Oranı (%)'
]


def check_karsiliksiz_cek(df):
    if df['Bankamız\nKarşılıksız\nÇek'] == 1 or df['Diğer Banka\nKarşılıksız\nÇek'] == 1 or df['Bankamız ve\nDiğer Banka\nKarşılıksız Çek'] == 1:
        return 1
    return 0

def check_tahakkuk(df):
    if df['Diğer Banka\nTahakkuk\n>20.000'] == 1 or df['Diğer Toplam Tahakkuk /\nDiğer Toplam Nakdi Risk\n> 0,01'] == 1:
        return 1
    return 0
    
def check_yapilandirma(df):
    if df['Diğer Banka \nCanlı Alacaklardan \nYeniden Yapılandırma > 0'] == 1:
        return 1
    return 0


def fix_fg_df(df, i):
    df = df.loc[df['Müşteri\nNumarası'].isin(customer_list[i]), fg_column_list]

    df = df.replace('VAR', 1).replace(['YOK', np.nan], 0)

    df['Karşılıksız Çek'] = df.apply(check_karsiliksiz_cek, axis = 1)
    df['Diğer Banka Tahakkuk'] = df.apply(check_tahakkuk, axis = 1)
    df['Diğer Banka Yapılandırma'] = df.apply(check_yapilandirma, axis = 1)

    df = df[fg_new_column_list]
    df.columns = renamed_fg_column_list

    df['FG Kriter Sayısı'] = 0
    for col in df.columns[1:-2]:
        df[col] = df[col].fillna(0)
        df['FG Kriter Sayısı'] += df[col]

    df['Finansal Güçlük'] = df['FG Kriter Sayısı'].apply(lambda x: 1 if x > 0 else 0)

    return df


df_fg_all, df_fg_last = batch_process_files(fg_df_raw_list, fg_sorted_list, fg_sorted_date_list, fix_fg_df)

Finansal Güçlük - 2023.12.01.xlsx işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## KRÖA

In [33]:
kroa_file_list, kroa_date_list, kroa_sorted_list = scan_folder(kroa_folder_path, '.csv')
kroa_sorted_date_list, kroa_df_raw_list = batch_import_files(kroa_sorted_list, kroa_folder_path, 'KRÖA', 'kroa', csv=[';'])

KRÖA - 2023.12.31.csv okunuyor
Tüm dosyalar başarıyla okundu


In [34]:
kroa_column_list = ['Müşteri No', 'KRÖA']

def fix_kroa_df(df, i):
    dfn = pd.DataFrame()
    dfn['Müşteri No'] = df[' MUSTERI_NO'].unique()
    dfn = dfn.loc[dfn['Müşteri No'].isin(customer_list[i])]
    dfn['KRÖA'] = 1
    return dfn

df_kroa_all, df_kroa_last = batch_process_files(kroa_df_raw_list, kroa_sorted_list, kroa_sorted_date_list, fix_kroa_df, kroa_column_list)

KRÖA - 2023.12.31.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## LGD

In [35]:
lgd_file_list, lgd_date_list, lgd_sorted_list = scan_folder(lgd_folder_path, '.csv')
lgd_sorted_date_list, lgd_df_raw_list = batch_import_files(lgd_sorted_list, lgd_folder_path, 'LGD', 'lgd', csv=[';'])

LGD - 2023.12.01.csv okunuyor
Tüm dosyalar başarıyla okundu


In [36]:
lgd_column_list = ['CUST_ID', 'LGD_IFRS']

renamed_lgd_column_list = ['Müşteri No', 'LGD']

def fix_lgd_df(df, i):
    df = df.loc[df['CUST_ID'].isin(customer_list[i]), lgd_column_list]
    df.columns = renamed_lgd_column_list
    df['LGD'] = df['LGD'].apply(lambda x: float(x.replace(',', '.')) if type(x) is str else x)
    return df

df_lgd_all, df_lgd_last = batch_process_files(lgd_df_raw_list, lgd_sorted_list, lgd_sorted_date_list, fix_lgd_df)

LGD - 2023.12.01.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Bankamız Çek

In [37]:
cek_durum_kod_dict = {
    'Vadesi Gelmemiş': 1,
    'Vadesi gelmemiş': 1,
    'Ödendi': 2,
    'Ödenmiş': 2,
    'Takastan İade': 3,
    'Takastan iade edilmiş': 3,
    'Karşılıksız': 4,
    'Muamelesiz İade': 5
}

# cek_kod_durum_dict = {
#     1: 'Vadesi Gelmemiş',
#     2: 'Ödenmiş',
#     3: 'Takastan İade',
#     4: 'Karşılıksız',
#     5: 'Muamelesiz İade'
# }

cek_int_columns = ['Çek No', 'Müşteri No', 'Hamil Müşteri No', 'Durum Kodu', 'Şube Kodu', 'Karşılıksız']
cek_float_columns = ['Tutar', 'Çek Tutarı']

In [38]:
bcek_file_list, bcek_date_list, bcek_sorted_list = scan_folder(bcek_folder_path)
bcek_sorted_date_list, bcek_df_raw_list = batch_import_files(bcek_sorted_list, bcek_folder_path, 'Bankamız Çek', 'bcek', True)

BÇEK - 2023.12.01.xlsx önbellekten okundu (100%)
Tüm dosyalar başarıyla okundu


In [39]:
bcek_column_list = [
    'Çek Numarası',
    'Keşideci Müşteri Numarası',
    'Durum Kodu',
    'Çek Tutarı',
    'Kesit Tarihi',
    'Takas Tarihi',
    'Tahsil/Teminat Kodu',
    'Teminata Alınma Kodu'
]

bcek_new_column_list = [
    'Çek Numarası',
    'Keşideci Müşteri Numarası',
    'Durum Kodu',
    'Döviz Kodu',
    'Tutar',
    'Çek Tutarı'
]

renamed_bcek_column_list = [
    'Çek No',
    'Müşteri No',
    'Durum Kodu',
    'Döviz Kodu',
    'Tutar',
    'Çek Tutarı'
]


def fix_bcek_ay_farki(df):
    before = 'Takas Tarihi'
    after = 'Kesit Tarihi'
    yd = df[after].year - df[before].year
    md = df[after].month - df[before].month
    md += yd * 12
    if md <= 11:
        return True
    else:
        return False

def fix_bcek_teminat(df):
    t1 = 'Tahsil/Teminat Kodu'
    t2 = 'Teminata Alınma Kodu'
    if df[t1] == 'T' or df[t2] == 'T':
        return True
    else:
        return False

def fix_bcek_df(df, i):
    df = df.loc[df['Keşideci Müşteri Numarası'].isin(customer_list[i]), bcek_column_list]

    df['Ay Farkı'] = df.apply(fix_bcek_ay_farki, axis=1)
    df['Teminat'] = df.apply(fix_bcek_teminat, axis=1)
    df = df.loc[(df['Ay Farkı']) & (df['Teminat'])].copy().reset_index(drop=True)
    
    df['Döviz Kodu'] = 'TRY'
    df['Tutar'] = df['Çek Tutarı']

    df = df[bcek_new_column_list]
    df.columns = renamed_bcek_column_list

    df['Karşılıksız'] = df['Durum Kodu'].apply(lambda x: 1 if x == 4 else 0)

    for c in cek_int_columns:
        if c in df.columns:
            df[c] = df[c].astype(int)
    for c in cek_float_columns:
        if c in df.columns:
            df[c] = df[c].astype(float)

    return df


bcek_df_list, df_bcek_last = batch_process_files(bcek_df_raw_list, bcek_sorted_list, bcek_sorted_date_list, fix_bcek_df, rename_exclude_columns=['Çek No', 'Müşteri No'])

BÇEK - 2023.12.01.xlsx işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Diğer Banka Çek

In [40]:
dbcek_file_list, dbcek_date_list, dbcek_sorted_list = scan_folder(dbcek_folder_path)
dbcek_sorted_date_list, dbcek_df_raw_list = batch_import_files(dbcek_sorted_list, dbcek_folder_path, 'Diğer Banka Çek', 'dbcek', True)

DBÇEK - 2023.12.01.xlsx önbellekten okundu (100%)
Tüm dosyalar başarıyla okundu


In [41]:
dbcek_column_list = [
    'Çek Numarası',
    'Keşideci Müşteri Numarası',
    'Keşideci TCKN/VKN',
    'Keşideci TCKN/VKN Seçeneği',
    'Durum',
    'İade Kodu',
    'Çek Tutarı',
    'Kesit Tarihi',
    'Takas Tarihi',
    'Teminat/Tahsil Kodu',
    'Teminata Alınma Kodu',
]

dbcek_new_column_list = [
    'Çek Numarası',
    'Keşideci Müşteri Numarası',
    'Durum Kodu',
    'Döviz Kodu',
    'Tutar',
    'Çek Tutarı',
]

renamed_dbcek_column_list = [
    'Çek No',
    'Müşteri No',
    'Durum Kodu',
    'Döviz Kodu',
    'Tutar',
    'Çek Tutarı',
]


def fix_dbcek_ay_farki(df):
    before = 'Takas Tarihi'
    after = 'Kesit Tarihi'
    yd = df[after].year - df[before].year
    md = df[after].month - df[before].month
    md += yd * 12
    if md <= 11:
        return True
    else:
        return False

def fix_dbcek_teminat(df):
    t1 = 'Teminat/Tahsil Kodu'
    t2 = 'Teminata Alınma Kodu'
    if df[t1] == 'T' or df[t2] == 'T':
        return True
    else:
        return False


def fix_dbcek_df(df, i):
    x = 'Keşideci Müşteri Numarası'
    df[x] = np.nan
    for c in ['VKN', 'TCKN']:
        df[c] = np.nan
        n = c[:1]
        df.loc[df['Keşideci TCKN/VKN Seçeneği'] == n, c] = df['Keşideci TCKN/VKN']
        dft = df_mnvt[['Müşteri No', c]].copy()
        dft.columns = [n, c]
        dft = dft.loc[(dft[c].notnull()) & (dft[c].isin(list(df[c].unique())))]
        df = pd.merge(df, dft, on=c, how='left')
        df.loc[df['Keşideci TCKN/VKN Seçeneği'] == n, x] = df[n]
    
    df = df.loc[(df[x].notnull()) & (df[x].isin(customer_list[i])), dbcek_column_list]

    df['Durum Kodu'] = df['Durum'].map(cek_durum_kod_dict)
    df['Ay Farkı'] = df.apply(fix_dbcek_ay_farki, axis=1)
    df['Teminat'] = df.apply(fix_dbcek_teminat, axis=1)
    df = df.loc[(df['Ay Farkı']) & (df['Teminat'])].copy().reset_index(drop=True)
    
    df['Döviz Kodu'] = 'TRY'
    df['Tutar'] = df['Çek Tutarı']

    df = df[dbcek_new_column_list]
    df.columns = renamed_dbcek_column_list

    df['Karşılıksız'] = df['Durum Kodu'].apply(lambda x: 1 if x == 4 else 0)

    for c in cek_int_columns:
        if c in df.columns:
            df[c] = df[c].astype(int)
    for c in cek_float_columns:
        if c in df.columns:
            df[c] = df[c].astype(float)

    return df


dbcek_df_list, df_dbcek_last = batch_process_files(dbcek_df_raw_list, dbcek_sorted_list, dbcek_sorted_date_list, fix_dbcek_df, rename_exclude_columns=['Çek No', 'Müşteri No'])

DBÇEK - 2023.12.01.xlsx işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## KTÇS

In [42]:
ktcs_file_list, ktcs_date_list, ktcs_sorted_list = scan_folder(ktcs_folder_path)
ktcs_sorted_date_list, ktcs_df_raw_list = batch_import_files(ktcs_sorted_list, ktcs_folder_path, 'KTÇS', 'ktcs', True, sheet_list=['YP Çekler'])

KTÇS - 2023.12.01.xlsx önbellekten okundu (100%)
Tüm dosyalar başarıyla okundu


In [43]:
ktcs_column_list = [
    'Çek Numarası (MICR)',
    'Müşteri Numarası',
    'Statü açıklama',
    'Döviz Kodu',
    'Çek Tutarı',
    'Çek Tutarı-TL'
]

renamed_ktcs_column_list = [
    'Çek No',
    'Müşteri No',
    'Durum Kodu',
    'Döviz Kodu',
    'Tutar',
    'Çek Tutarı'
]


def fix_ktcs_df(df, i):
    df = df.loc[df['Müşteri Numarası'].isin(customer_list[i]), ktcs_column_list]
    df['Statü açıklama'] = df['Statü açıklama'].map(cek_durum_kod_dict)

    df = df[ktcs_column_list]
    df.columns = renamed_ktcs_column_list

    df['Karşılıksız'] = df['Durum Kodu'].apply(lambda x: 1 if x == 4 else 0)

    return df


ktcs_df_list, df_ktcs_last = batch_process_files(ktcs_df_raw_list, ktcs_sorted_list, ktcs_sorted_date_list, fix_ktcs_df, rename_exclude_columns=['Çek No', 'Müşteri No'])

KTÇS - 2023.12.01.xlsx işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Senet

In [44]:
# senet_kod_durum_dict = {
    # 0: 'Girişi İptal',
    # 1: 'Açık',
    # 2: 'Ödendi',
    # 3: 'Protestolu',
    # 4: 'Muamelesiz İade',
    # 8: 'Tahsil-Teminat senedi arasında virman edilmiş',
# }

senet_int_columns = ['Müşteri No', 'Durum Kodu', 'Karşılıksız']
senet_float_columns = ['Tutar', 'Senet Tutarı']

In [45]:
senet_file_list, senet_date_list, senet_sorted_list = scan_folder(senet_folder_path, '.csv')
senet_sorted_date_list, senet_df_raw_list = batch_import_files(senet_sorted_list, senet_folder_path, 'Senet', 'senet', csv=[';'])

Senet - 2023.12.01.csv okunuyor
Tüm dosyalar başarıyla okundu


In [46]:
senet_column_list = [
    'ODS_MUSTERI_NO',
    'ODS_HESAP_NO',
    'ODS_DURUM_KODU',
    'ODS_DOVIZ_CINSI',
    'ODS_BAKIYE'
]

senet_new_column_list = [
    'Anahtar',
    'ODS_MUSTERI_NO',
    'ODS_DURUM_KODU',
    'ODS_DOVIZ_CINSI',
    'ODS_BAKIYE',
    'Senet Tutarı'
]

renamed_senet_column_list = [
    'Anahtar',
    'Müşteri No',
    'Durum Kodu',
    'Döviz Kodu',
    'Tutar',
    'Senet Tutarı'
]



def fix_senet_df(df, i):
    df = df.loc[(df['ODS_MUSTERI_NO'].isin(customer_list[i])) & (df['ODS_DURUM_KODU'] != ' '), senet_column_list]
    df['Anahtar'] = df.apply(lambda d: '{}-{}'.format(d['ODS_MUSTERI_NO'], d['ODS_HESAP_NO']), axis=1)
    
    df['Döviz Kuru'] = 1 ##################
    df['ODS_BAKIYE'] = df['ODS_BAKIYE'].apply(lambda x: float(x.replace(',', '.')))
    df['Senet Tutarı'] = df['ODS_BAKIYE'] * df['Döviz Kuru']

    df = df[senet_new_column_list]
    df.columns = renamed_senet_column_list

    df['Durum Kodu'] = df['Durum Kodu'].astype(int)
    df['Protestolu'] = df['Durum Kodu'].apply(lambda x: 1 if x == 3 else 0)

    for c in senet_int_columns:
        if c in df.columns:
            df[c] = df[c].astype(int)
    for c in senet_float_columns:
        if c in df.columns:
            df[c] = df[c].astype(float)

    return df


senet_df_list, df_senet_last = batch_process_files(senet_df_raw_list, senet_sorted_list, senet_sorted_date_list, fix_senet_df, rename_exclude_columns=['Çek No', 'Müşteri No'])

Senet - 2023.12.01.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


# Veri Birleştirme

## df_wide

In [47]:
sorted_date_list = yz_sorted_date_list

df = df_kr202_last_original_backup.copy()
df = pd.merge(df, df_ms_last, on='Müşteri No', how='left')
df = pd.merge(df, df_proje_last, on='Müşteri No', how='left')
df['Proje Kredisi'] = df['Proje Kredisi'].fillna(0)
df = df.loc[df['Müşteri Sınıfı'] <= 2]
df = df.loc[df['İlgili Bölüm'].notnull()] 
df_kr202_last = df.copy()

df = df_eds_last.copy()
df = pd.merge(df, df_yz_last, how='outer', on='Müşteri No')
df = pd.merge(df, df_kroa_last, how='outer', on='Müşteri No')
df = pd.merge(df, df_lgd_last, how='outer', on='Müşteri No')
df_izleme_last = df.copy()

sc, bc, ic = 'Şube Kodu', 'Bölge Adı', 'Şube İl'
df_wide = []
for df_kr202, df_p, df_eds, df_yz, df_kroa, df_lgd, df_fg, df_ms, d in zip(df_kr202_all, df_proje_all, df_eds_all, df_yz_all, df_kroa_all, df_lgd_all, df_fg_all, df_ms_all, sorted_date_list):
    dfw = df_kr202.copy()

    for df in [df_p, df_eds, df_yz, df_kroa, df_lgd, df_fg, df_ms]:
        dfw = pd.merge(dfw, df, on='Müşteri No', how='left')
    
    dfw = dfw.loc[dfw['Müşteri Sınıfı ' + d] <= 2] 
    dfw = dfw.loc[dfw['İlgili Bölüm ' + d].notnull()] 

    c, b, i = sc + ' ' + d, bc + ' ' + d, ic + ' ' + d
    dfs = df_sube[[sc, bc, ic]].copy().rename(columns={sc: c, bc: b, ic: i})
    dfw = pd.merge(dfw, dfs, how='left', on=c)
    df_wide.append(dfw.copy())


na_0_list = []

for c in df_kroa_last.columns:
    if c != 'Müşteri No':
        na_0_list.append(c)
        df_izleme_last[c] = df_izleme_last[c].fillna(0)

for c in df_fg_last.columns:
    if c != 'Müşteri No':
        na_0_list.append(c)
        df_fg_last[c] = df_fg_last[c].fillna(0)

for c in df_proje_last.columns:
    if c != 'Müşteri No':
        na_0_list.append(c)

for i, d in enumerate(sorted_date_list):
    c_list = [c + ' ' + d for c in na_0_list]
    df_wide[i][c_list] = df_wide[i].loc[:, c_list].fillna(0)

## df_mali

In [48]:
mali_columns = [
    'CST_NMR', 'FST_YR', 'FST_MO', 'DONEM', 'AKTIF', 'DONEN_DEGERLER', 'PARA_MEVCUDU', 'MENKUL_DEGERLER',
    'ZARAR', 'KISA_VADELI_BORCLAR', 'KV_FINANSAL_BORCLAR', 'UZUN_VADELI_BORCLAR', 'UV_FINANSAL_BORCLAR',
    'OZ_SERMAYE', 'KAR', 'NET_SATISLAR', 'FAALIYET_GELIRI', 'FINANSMAN_GIDERLERI', 'VFO_KAR',
    'DONEM_AMORTISMAN', 'TOP_FIN_BORC', 'TOP_BORCLAR', 'YENI_OZSERMAYE', 'EBITDA',
    'KVFB_NET_SATISLAR', 'BORCLANMA_ORANI', 'KALDIRAC_ORANI', 'FAIZ_YUKU_KARSILAMA'
]

mali_column_rename_dict = {
    'CST_NMR': 'Müşteri No',
    'KVFB_NET_SATISLAR': 'KVFB / Net Satışlar',
    'BORCLANMA_ORANI': 'Borçlanma Oranı',
    'KALDIRAC_ORANI': 'Kaldıraç Oranı',
    'FAIZ_YUKU_KARSILAMA': 'Faiz Yükü Karşılama Farkı',
}

def mali_cari_oran(df):
    if df['KISA_VADELI_BORCLAR'] == 0: return np.nan
    else: return df['DONEN_DEGERLER'] / df['KISA_VADELI_BORCLAR']

df = df_mali_backup.copy()
df = df.loc[df['RN'] == 1, mali_columns]
df = df.rename(columns=mali_column_rename_dict)
df['Cari Oran'] = df.apply(mali_cari_oran, axis=1)
df_mali_last = df.copy()

df_kur_last = df_kur.copy()
df_sektor_last = df_sektor.copy()

## df_cek

In [49]:
cek_wide_column_list = [
    'Müşteri Sınıfı',
    'Değerlendirmeye Esas Segment',
    'İlgili Bölüm',
    'Yetki Kodu',
    'Tahsis Sektör Adı',
    'NACE Harf',
    'Şube Kodu',
    'Bölge Adı',
    'Şube İl',
    'Proje Kredisi'
]

cek_df_list = []

for dfw, b, db, k, d in zip(df_wide, bcek_df_list, dbcek_df_list, ktcs_df_list, bcek_sorted_date_list):
    df = pd.concat([b, db, k], ignore_index=True)

    cek_dated_wide_column_list = ['Müşteri No'] + [c + ' ' + d for c in cek_wide_column_list]
    dft = dfw[cek_dated_wide_column_list].copy()
    df = pd.merge(df, dft, on='Müşteri No', how='left')
    cek_df_list.append(df.copy())


df.columns = [c[:-(len(d) + 1)] if c.endswith(d) else c for c in df.columns]
df_cek_last = df.copy()


dfc = df_cek_last[['Müşteri No', 'Karşılıksız']].copy()
dfc = dfc.loc[dfc['Karşılıksız'] == 1, ['Müşteri No']]

dfs = df_senet_last[['Müşteri No', 'Protestolu']].copy()
dfs = dfs.loc[dfs['Protestolu'] == 1, ['Müşteri No']]

dff = df_fg_last[['Müşteri No', 'Karşılıksız Çek', 'Çek Yasaklılık']].copy()
dff = dff.loc[(dff['Karşılıksız Çek'] == 1) | (dff['Çek Yasaklılık'] == 1), ['Müşteri No']]

df = pd.merge(dfc, dfs, on='Müşteri No', how='outer')
df = pd.merge(df, dff, on='Müşteri No', how='outer')

df = df.drop_duplicates(keep='first').reset_index(drop=True)
df = df.sort_values('Müşteri No').reset_index(drop=True)
df['Karşılıksız Çek/Senet'] = 1

df_cek_senet = df.copy()

## df_master

In [50]:
master_column_list = [
    'Müşteri No', 'Adı', 'Şube Kodu', 'Şube Adı', 'Şube İl', 'Şube İlçe',
    'Bölge Adı', 'Şube Türü', 'Yetki Kodu',
    'İlgili Bölüm', 'Müşteri Sınıfı', 'Risk Grubu Kodu', 'Risk Grubu Adı',
    'Tahsis Sektör Adı', 'Tahsis Alt Sektör Adı',
    'NACE Kodu', 'NACE Harf', 'Sektör Derecesi', 'Sektör Derece İzleme',
    'Nakdi TL Bakiye', 'Nakdi YP Bakiye', 'Nakdi Toplam',
    'G.Nakdi TL Bakiye', 'G.Nakdi YP Bakiye', 'G.Nakdi Toplam', 'Toplam Risk',
    'Nakdi TL Reeskontlu', 'Nakdi YP Reeskontlu', 'Nakdi Reeskontlu Toplam', 'Toplam Reeskontlu Risk',
    'Proje Kredisi',
]

dfm = df_kr202_last.copy()

dfm['NACE 6lı'] = dfm['NACE Kodu'].apply(lambda x: x[2:] if type(x) is str else np.nan)
dfm = pd.merge(dfm, df_sektor_last, on='NACE 6lı', how='left')
dfm = pd.merge(dfm, df_sube[['Şube Kodu', 'Bölge Adı']], on='Şube Kodu', how='left')

for df in [df_kur_last, df_izleme_last, df_fg_last, df_memzuc_last, df_mali_last]:
    dfm = pd.merge(dfm, df, on='Müşteri No', how='left')
    cl = list(df.columns)
    if 'Müşteri No' in cl:
        cl.remove('Müşteri No')
    master_column_list += cl

for c in list(df_fg_last.columns) + ['KRÖA']:
    dfm[c] = dfm[c].fillna(0)

dfm = dfm.sort_values('Nakdi Reeskontlu Toplam', ascending=False).reset_index(drop=True)

for c in master_column_list.copy():
    if c not in dfm.columns:
        master_column_list.remove(c)
        
dfm = dfm[master_column_list]

df_master = dfm.copy()

## Listeler

In [51]:
segment_list = ['KURUMSAL', 'TİCARİ', 'KOBİ', 'İŞLETME']
bolum_list = [
    'Kurumsal Krediler Tahsis Bölümü',
    'Ticari Krediler Tahsis Bölümü',
    'Perakende Krediler Tahsis Bölümü',
    'Proje Finansmanı Bölümü',
    'Krediler İzleme Bölümü',
    'Ticari ve Kurumsal Krediler Takip Bölümü'
]

sektor_list = list(df_kr202_last[['Tahsis Sektör Adı', 'Nakdi Reeskontlu Toplam']].groupby('Tahsis Sektör Adı').sum().sort_values('Nakdi Reeskontlu Toplam', ascending=False).index)
filtered_sektor_list = list(filter(lambda x: x not in ['PPP (Kamu Garantili KÖİ) (HASTANE VE OTOYOLLAR)', 'FİNANS'], sektor_list))
sorted_sektor_list = sektor_list.copy()
sorted_sektor_list.sort(key=locale.strxfrm)

sube_list = sorted(df_kr202_last['Şube Kodu'].unique())
sube_list = list(filter(lambda x: x >= 1000, sube_list))
sube_tur_list = sorted(df_sube.loc[(df_sube['Şube Kodu'].isin(sube_list)) & (df_sube['Şube Türü'].notnull()), 'Şube Türü'].unique())
bolge_list = sorted(df_sube.loc[(df_sube['Şube Kodu'].isin(sube_list)) & (df_sube['Bölge Adı'].notnull()) & (df_sube['Bölge Adı'] != ' ') & (df_sube['Bölge Adı'] != 0), 'Bölge Adı'].unique())
il_list = sorted(df_sube.loc[(df_sube['Şube Kodu'].isin(sube_list)) & (df_sube['Şube İl'].notnull()), 'Şube İl'].unique())
sube_kodu_adi_dict = dict(zip(df_sube['Şube Kodu'], df_sube['Şube Adı']))
sube_kodu_adi_dict = {**sube_kodu_adi_dict, **{'TOPLAM': 'TOPLAM'}}
sube_adi_kodu_dict = {v: k for k, v in sube_kodu_adi_dict.items()}

unique_dict = {}
for c in df_izleme_last.columns:
    if c != 'Müşteri No':
        unique_dict[c] = sorted(df_izleme_last.loc[df_izleme_last[c].notnull(), c].unique())
        
for c in ['Müşteri Sınıfı', 'Yetki Kodu', 'NACE Harf']:
    unique_dict[c] = sorted(df_kr202_last.loc[df_kr202_last[c].notnull(), c].unique())
    
unique_dict['Tahsis Sektör Adı'] = sektor_list
unique_dict['Bölge Adı'] = bolge_list
unique_dict['Şube Kodu'] = sube_list
nace_list = unique_dict['NACE Harf']

h_space, w_space = 25, 3
h_limit, l_space = 16, 7 

letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
letters = list(letters) + ['A' + x for x in letters]

## convert_sube_kodu_adi()

In [52]:
def convert_sube_kodu_adi(dfs):
    if type(dfs) is list:
        dfl = []
        for dfi in dfs:
            df = dfi.copy()
            c = df.columns[0]
            x = c.replace('Şube Kodu', 'Şube Adı')
            df[c] = df[c].map(sube_kodu_adi_dict)
            df = df.rename(columns={c: x})
            dfl.append(df.copy())
        return dfl
    else:
        df = dfs.copy()
        c = df.columns[0]
        x = c.replace('Şube Kodu', 'Şube Adı')
        df[c] = df[c].map(sube_kodu_adi_dict)
        df = df.rename(columns={c: x})
        return df

## df_nace

In [53]:
df = df_nace.copy()
df = df.loc[df['NACE_KODU'].isin(nace_list), ['NACE_KODU', 'NACE_ADI']]
df.columns = ['NACE Harf', 'NACE Adı']
df['NACE Ana Sektör'] = df['NACE Harf'] + ' - ' + df['NACE Adı']
df = df[['NACE Harf', 'NACE Ana Sektör']]
df.loc[len(df)] = ['TOPLAM', 'TOPLAM']
nace_kodu_adi_dict = df.set_index('NACE Harf').T.to_dict('records')[0]

In [66]:
dfm = df_master.copy()
tsa = 'Tahsis Sektör Adı'
df = pd.DataFrame()
df[tsa] = sorted(sektor_list)
df = add_v_space(df)
des = 'Değerlendirmeye Esas Segment'
c = 'LGD'
t_list = ['Genel Toplam', np.nan]
for s in segment_list:
    dft = dfm.loc[dfm[des] == s, [tsa, c]]
    t = dft[c].mean()
    t_list.append(t)
    dft = dft.groupby(tsa).mean().reset_index().rename(columns={c: s})
    df = pd.merge(df, dft, on=tsa, how='left')

df = add_v_space(df)
t_list.append(np.nan)


dft = dfm[[tsa, c]]
t = dft[c].mean()
t_list.append(t)
dft = dft.groupby(tsa).mean().reset_index().rename(columns={c: 'Portföy Geneli'})
df = pd.merge(df, dft, on=tsa, how='left')

df = add_h_space(df)
df.loc[len(df)] = t_list
df

,Tahsis Sektör Adı,NaN,KURUMSAL,TİCARİ,KOBİ,İŞLETME,NaN,Portföy Geneli
0,ALKOLLÜ İÇECEKLER,NaN,0.52,0.51,0.46,0.37,NaN,0.39
1,AĞAÇ ÜRÜNLERİ,NaN,0.55,0.53,0.46,0.34,NaN,0.37
2,BALIKÇILIK,NaN,0.54,0.51,0.47,0.37,NaN,0.41
3,BASIM YAYIM (MEDYA),NaN,0.49,0.52,0.48,0.35,NaN,0.37
4,BELEDİYELER VE OSBLER,NaN,0.50,0.52,0.42,0.32,NaN,0.35
5,DERİ VE DERİ ÜRÜNLERİ,NaN,0.55,0.52,0.47,0.34,NaN,0.38
6,DİĞER HİZMET FAALİYETLERİ,NaN,0.53,0.52,0.44,0.34,NaN,0.35
7,DİĞER İMALAT,NaN,0.54,0.53,0.46,0.33,NaN,0.36
8,ELEKTRONİK VE BİLİŞİM,NaN,0.56,0.53,0.49,0.35,NaN,0.39
9,ENERJİ,NaN,0.52,0.51,0.44,0.33,NaN,0.39


# Zombi Firmalar

## Firma Kümesi

In [78]:
# df_zombi = df_master.copy()
# zombi_nace_exclude_list = [
#     'FİNANS',
#     'PPP (Kamu Garantili KÖİ) (HASTANE VE OTOYOLLAR)',
#     'BELEDİYELER VE OSB\'LER',
#     'BELEDİYELER VE OSB LER'
# ]
# df_zombi = df_zombi.loc[
#     (df_zombi['NACE Kodu'].notnull()) &
#     (df_zombi['Proje Kredisi'] == 0) &
#     (df_zombi['Risk Grubu Kodu'] != 'BANKRG') &
#     (df_zombi['Tahsis Sektör Adı'].notnull()) &
#     (~df_zombi['Tahsis Sektör Adı'].isin(zombi_nace_exclude_list))
#     ]

# df_zombi['NACE 2li']  = df_zombi['NACE Kodu'].apply(lambda x: x[:4])
# firm_count_in_nace_limit = 5
# zombi_nace_empty_list = list(df_zombi['NACE 2li'].value_counts()[df_zombi['NACE 2li'].value_counts() < firm_count_in_nace_limit].index)
# df_zombi = df_zombi.loc[~df_zombi['NACE 2li'].isin(zombi_nace_empty_list)]

# df_zombi_mali = df_zombi_mali_backup.copy()
# df_zombi_mali = df_zombi_mali.rename(columns={'CST_NMR': 'Müşteri No'})
# df_zombi_mali = df_zombi_mali.drop_duplicates(subset=['Müşteri No', 'FST_YR'], keep='first')
# df_zombi = df_zombi.loc[df_zombi['Müşteri No'].isin(df_zombi_mali['Müşteri No'].unique())]

## Mali Veriler

In [79]:
# zombi_year_list = sorted(df_zombi_mali['FST_YR'].unique(), reverse=True)
# zombi_year_prefix_list = [str(x) for x in zombi_year_list[:-1]]
# zombi_mali_column_list = ['Müşteri No', 'FST_YR', 'FAALIYET_GELIRI', 'FINANSMAN_GIDERLERI', 'UZUN_VADELI_BORCLAR', 'KISA_VADELI_BORCLAR', 'AKTIF', 'NET_SATISLAR']
# df_zombi['FST_YR'] = np.nan

# for y in zombi_year_list:
#     dft = df_zombi_mali.copy()
#     dft = dft.loc[dft['FST_YR'] == y, zombi_mali_column_list]
#     dft.columns = ['Müşteri No'] + [str(y) + '_' + x for x in zombi_mali_column_list[1:]]
#     df_zombi = pd.merge(df_zombi, dft, on='Müşteri No', how='left')
#     df_zombi['FST_YR'] = df_zombi['FST_YR'].fillna(df_zombi[str(y) + '_FST_YR'])
#     df_zombi = df_zombi.drop(str(y) + '_FST_YR', axis=1)

# df_zombi = df_zombi.loc[df_zombi['FST_YR'].isin(zombi_year_list[:2])]

## Kriterler

In [80]:
# s = '_INTEREST_COVERAGE_RATIO (Faaliyet Geliri / Finansman Gideri)'

# for y in zombi_year_prefix_list:
#     c = y + s
#     df_zombi[c] = df_zombi[y + '_FAALIYET_GELIRI'] / df_zombi[y + '_FINANSMAN_GIDERLERI']
#     df_zombi[c] = df_zombi[c].replace([np.inf, -np.inf, 0], np.nan)
	
# s = '_KALDIRAC ((KV_BORC + UV_BORC) / AKTIF)'

# for y in zombi_year_prefix_list:
#     c = y + s
#     df_zombi[c] = (df_zombi[y + '_UZUN_VADELI_BORCLAR'] + df_zombi[y + '_KISA_VADELI_BORCLAR'] )/ df_zombi[y + '_AKTIF']
#     df_zombi[c] = df_zombi[c].replace([np.inf, -np.inf, 0], np.nan)

# df_zombi_medyan = df_zombi[['NACE 2li'] + [y + s for y in zombi_year_prefix_list]].copy().groupby('NACE 2li').median()
# df_zombi_medyan.columns = [x.replace(' ((KV_BORC + UV_BORC) / AKTIF)', '_MEDYAN') for x in df_zombi_medyan.columns]
# df_zombi = pd.merge(df_zombi, df_zombi_medyan, on='NACE 2li', how='left')

# s = '_NOMINAL_CIRO_ARTIS'
# x = '_NET_SATISLAR'

# for y in zombi_year_prefix_list:
#     c = y + s
#     df_zombi[c] = df_zombi[y + x] / df_zombi[str(int(y) - 1) + x]
#     df_zombi[c] = df_zombi[c].replace([np.inf, -np.inf, 0], np.nan)

In [81]:
# def detect_icr_zombie(df):
#     s = '_INTEREST_COVERAGE_RATIO (Faaliyet Geliri / Finansman Gideri)'
#     y = int(df['FST_YR'])
#     if df['2021' + s] < 1:
#         if (df['2022' + s] < 1 and y == 2022) or (df['2020' + s] < 1 and y == 2021):
#             return 1
#     return 0

# def detect_leverage_zombie(df):
#     s = '_KALDIRAC ((KV_BORC + UV_BORC) / AKTIF)'
#     s_m = '_KALDIRAC_MEDYAN'
#     y = int(df['FST_YR'])

#     if df['2021' + s] > df['2021' + s_m]:
#         if (df['2022' + s] > df['2022' + s_m] and y == 2022) or (df['2020' + s] > df['2020' + s_m] and y == 2021):
#             return 1
#     return 0

# def detect_sales_zombie(df):
#     s = '_NOMINAL_CIRO_ARTIS'
#     y = int(df['FST_YR'])

#     ufe_2022 = 1.2847 + 1
#     ufe_2021 = 0.4386 + 1
#     ufe_2020 = 0.1218 + 1

#     if df['2021' + s] < ufe_2021:
#         if (df['2022' + s] < ufe_2022 and y == 2022) or (df['2020' + s] < ufe_2020 and y == 2021):
#             return 1
#     return 0

# def detect_zombie_firm(df):
#     c_sales= 'CIRO_ZOMBI_BELIRTEC'
#     c_icr = 'ICR_ZOMBI_BELIRTEC'
#     c_leverage = 'KALDIRAC_ZOMBI_BELIRTEC'

#     if df[c_sales] == 1 and df[c_icr] == 1 and df[c_leverage] == 1:
#         return 1
#     return 0


## Analiz

In [82]:
# zombi_belirtec_list = ['ICR_ZOMBI_BELIRTEC', 'KALDIRAC_ZOMBI_BELIRTEC', 'CIRO_ZOMBI_BELIRTEC', '3_ZOMBI_BELIRTEC']
# zombi_belirtec_fun_list = [detect_icr_zombie, detect_leverage_zombie, detect_sales_zombie, detect_zombie_firm]

# for c, f in zip(zombi_belirtec_list, zombi_belirtec_fun_list):
#     df_zombi[c] = df_zombi.apply(f, axis=1)

# df_zombi['Belirteç Sayısı'] = df_zombi['ICR_ZOMBI_BELIRTEC'] + df_zombi['KALDIRAC_ZOMBI_BELIRTEC'] + df_zombi['CIRO_ZOMBI_BELIRTEC']

# Portföy Risklilik Analizi

## Kriter Fonksiyonları

In [83]:
# def fg_score(df):
#     return int(df['FG Kriter Sayısı'])

# def eds_score(df):
#     if df['Entegre Derece Skor'] == 12: return 10
#     elif df['Entegre Derece Skor'] == 11: return 8
#     elif df['Entegre Derece Skor'] == 10: return 6
#     elif df['Entegre Derece Skor'] == 9: return 4
#     elif df['Entegre Derece Skor'] == 8: return 3
#     elif df['Entegre Derece Skor'] == 7: return 2
#     elif df['Entegre Derece Skor'] == 6: return 1
#     return 0

# def yz_score(df):
#     if df['Yapay Zeka Risk Sınıfı'] == 5: return 10 + df['Gecikme'] * 6
#     elif df['Yapay Zeka Risk Sınıfı'] == 4: return 8 + df['Gecikme'] * 4
#     elif df['Yapay Zeka Risk Sınıfı'] == 3: return 6 + df['Gecikme'] * 2
#     elif df['Yapay Zeka Risk Sınıfı'] == 2: return 4 + df['Gecikme'] * 1
#     return 0

# def kur_riski_score(df):
#     if df['Kur Riski Çalışması Sınıfı'] == 'Yüksek Riskli': return 4
#     elif df['Kur Riski Çalışması Sınıfı'] == 'Yakından İzlenmeli': return 2
#     return 0

# def kvfb_score(df):
#     if df['KVFB / Net Satışlar'] > 2: return 2
#     elif df['KVFB / Net Satışlar'] > 1: return 1
#     return 0

# def borclanma_score(df):
#     if df['Borçlanma Oranı'] > 8 or df['YENI_OZSERMAYE'] < 0: return 2
#     elif df['Borçlanma Oranı'] > 4: return 1
#     return 0

# def kaldirac_score(df):
#     riskli_sektor_list = [
#         'İNŞAAT TAAHHÜT',
#         'BELEDİYELER VE OSB LER',
#         'BELEDİYELER VE OSB\'LER',
#         'MADENCİLİK',
#         'GEMİCİLİK TERSANECİLİK',
#         'TURİZM'
#     ]
#     if df['Tahsis Sektör Adı'] in riskli_sektor_list:
#         low, high = 4, 12
#     else:
#         low, high = 8, 14
#     if df['Kaldıraç Oranı'] > high or df['EBITDA'] < 0: return 2
#     elif df['Kaldıraç Oranı'] > low: return 1
#     return 0

# def faiz_yuku_score(df):
#     if df['Faiz Yükü Karşılama Farkı'] < -6: return 2
#     elif df['Faiz Yükü Karşılama Farkı'] < 0: return 1
#     return 0

# def cari_oran_score(df):
#     if df['Cari Oran'] < 1: return 2
#     elif df['Cari Oran'] < 2: return 1
#     return 0

# def sektor_score(df):
#     if df['Sektör Derecesi'] == 'C+': return 6
#     elif df['Sektör Derecesi'] == 'B-': return 3
#     return 0

# def gecikme_score(df):
#     if df['Gecikme Gün'] >= 21: return 7
#     elif df['Gecikme Gün'] >= 11: return 5
#     elif df['Gecikme Gün'] >= 1: return 3
#     return 0

# def diger_tahakkuk_score(df):
#     if df['Diğer Banka Tahakkuk']: return np.inf
#     return 0




# def memzuc_payi_score(df):
#     if df['Bankamız Memzuç Risk Payı (%)'] > 75 or df['Bankamız Memzuç Nakdi Risk Payı (%)'] > 75: return 4
#     if df['Bankamız Memzuç Risk Payı (%)'] > 50 or df['Bankamız Memzuç Nakdi Risk Payı (%)'] > 50: return 2
#     return 0

# def sektor_limit_score(df):
#     if df['Sektör Limit Doluluk Oranı (%)'] > 75 or df['Sektör Nakdi Limit Doluluk Oranı (%)'] > 75: return 4
#     if df['Sektör Limit Doluluk Oranı (%)'] > 50 or df['Sektör Nakdi Limit Doluluk Oranı (%)'] > 50: return 2
#     return 0

# def bankamiz_limit_score(df):
#     if df['Bankamız Limit Doluluk Oranı (%)'] > 75 or df['Bankamız Nakdi Limit Doluluk Oranı (%)'] > 75: return 4
#     if df['Bankamız Limit Doluluk Oranı (%)'] > 50 or df['Bankamız Nakdi Limit Doluluk Oranı (%)'] > 50: return 2
#     return 0

# def banka_sayisi_score(df):
#     if df['Banka Sayısı Değişimi'] < 0: return 4
#     elif df['Banka Sayısı Değişimi'] > 0: return 2
#     return 0

# def lgd_score(df):
#     if df['LGD'] > 0.55: return 4
#     elif df['LGD'] > 0.40: return 2
#     return 0


# def diger_takip_score(df):
#     if df['Diğer Banka Takip']: return 7
#     return 0


# def tsps_iflas_score(df):
#     if df['TSPS İflas vb']: return 3
#     return 0


# def diger_yapilandirma_score(df):
#     if df['Diğer Banka Yapılandırma']: return 2
#     return 0


# def diger_tazmin_score(df):
#     if df['Diğer Banka Tazmin']: return 5
#     return 0


# def cek_score(df):
#     if df['Karşılıksız Çek (BB+DB)'] or df['Karşılıksız Çek'] or df['Çek Yasaklılık']: return 3
#     return 0




# # def fg_extra_score(df):
# #     score = 0
# #     if df['Diğer Banka Takip']: score += 7
# #     if df['TSPS İflas vb']: score += 3
# #     if df['Diğer Banka Tahakkuk']: score += 1000  #?????
# #     if df['Diğer Banka Yapılandırma']: score += 2
# #     if df['Diğer Banka Tazmin']: score += 5
# #     if df['Karşılıksız Çek'] or df['Çek Yasaklılık']: score += 3    ##  Senet?
# #     return score

## Risk Kategori Fonksiyonları

In [84]:
# def categorize_pra(df, x):
#     ########## ????????????
#     if df[x] <= 2: return 'Çok Düşük'
#     elif df[x] <= 10: return 'Düşük'
#     elif df[x] <= 15: return 'Makul'
#     return 'Yüksek'
    
def categorize_pra(df, x=None, threshold_list=None, category_list=None):
    default_x = 'Toplam Puan'
    # default_threshold_list = [2, 10, 15]
    # default_category_list = ['Çok Düşük', 'Düşük', 'Makul', 'Yüksek']
    default_threshold_list = [0, 2, 5, 10, 15, np.inf]
    default_category_list = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek', 'Diğer Banka Takip']

    if x is None:
        x = default_x
    if threshold_list is None:
        threshold_list = default_threshold_list
    if category_list is None:
        category_list = default_category_list

    if df[x] == np.inf: return category_list[-1]
    for t, c in zip(threshold_list, category_list):
        if df[x] <= t: return c
    return category_list[-1]
    
def create_pra_risk_categories(dfp, x='Toplam Puan', threshold_list=None, category_list=None, r='Risk Kategorisi', xx=None, factor=1_000_000):
    m = 'Müşteri No'
    c = 'Nakdi Reeskontlu Toplam'
    nn = 'Adet'
    s = 'Kümülatif Risk'
    ss = 'Kümülatif Adet'
    p = '%'
    pp = '% '
    if xx is None: xx=x

    df = dfp[[x, c]].copy()
    df = df.groupby(x).sum().reset_index()
    df[c] /= factor
    df[s] = df[c].cumsum()
    t = df[s].values[-1]
    df[p] = df[s] / t
    df[r] = df.apply(categorize_pra, x=xx, threshold_list=threshold_list, category_list=category_list, axis=1)
    
    dft = dfp[[m, x]].copy()
    dft = dft.groupby(x).count().reset_index().rename(columns={m: nn})
    dft[ss] = dft[nn].cumsum()
    tt = dft[ss].values[-1]
    dft[pp] = dft[ss] / tt

    df[' '] = np.nan
    df = pd.merge(df, dft, on=x)

    return df


def return_excluded_info(name, df, minus=True, factor=1_000_000):
    sign = -1 if minus else 1
    return [name, sign * df[mn].count(), sign * df[nr].sum() / factor, sign * df[gnr].sum() / factor, sign * df[tr].sum() / factor]

def return_percentages(c):
    percentages = list(dfk.loc[dfk['Küme'] == c].values[0][1:] / dfk.loc[dfk['Küme'] == c0].values[0][1:])
    return [' '] + percentages

## Analiz Fonksiyonları

In [85]:
# puan_column_list = [
#     'Finansal Güçlük Puanı', 'Entegre Derece Puanı',
#     'Risk Sınıfı Puanı', 'Kur Riski Puanı',
#     'KVFB / Net Satışlar Puanı', 'Borçlanma Oranı Puanı',
#     'Kaldıraç Oranı Puanı', 'Faiz Yükü Karşılama Puanı', 'Cari Oran Puanı',
#     'Sektör Riskliliği Puanı', 'Gecikme Puanı',
#     'Diğer Banka Tahakkuk Puanı'
# ]

# puan_fun_list = [
#     fg_score, eds_score,
#     yz_score, kur_riski_score,
#     kvfb_score, borclanma_score,
#     kaldirac_score, faiz_yuku_score, cari_oran_score,
#     sektor_score, gecikme_score,
#     diger_tahakkuk_score
# ]

# extra_puan_column_list = [
#     'Memzuç Payı Puanı', 'Sektör Limit Doluluk Puanı', 'Bankamız Limit Doluluk Puanı',
#     'Banka Sayısı Değişimi Puanı', 'LGD Puanı',
#     'Diğer Banka Takip Puanı', 'Diğer Banka Tazmin Puanı', 'Diğer Banka Yapılandırma Puanı',
#     'TSPS İflas Puanı', 'Karşılıksız Çek Puanı'
    
# ]

# extra_puan_fun_list = [
#     memzuc_payi_score, sektor_limit_score, bankamiz_limit_score,
#     banka_sayisi_score, lgd_score,
#     diger_takip_score, diger_tazmin_score, diger_yapilandirma_score,
#     tsps_iflas_score, cek_score
# ]


# def create_pra_analysis(df_pra):
#     df = df_pra.copy()

#     omd = memzuc_sorted_date_list[-2]
#     dfom = df_memzuc_all[-2][['Müşteri No', 'Banka Sayısı ' + omd]]
#     dfom.columns = ['Müşteri No', 'Banka Sayısı (t-1)']
#     df = pd.merge(df, dfom, on='Müşteri No', how='left')
#     df['Banka Sayısı Değişimi'] = df['Banka Sayısı'] - df['Banka Sayısı (t-1)']

#     df = pd.merge(df, df_karsiliksiz_cek, on='Müşteri No', how='left')
#     df['Karşılıksız Çek (BB+DB)'] = df['Karşılıksız Çek (BB+DB)'].fillna(0)

#     for c, f in zip(puan_column_list, puan_fun_list):
#         df[c] = df.apply(f, axis=1)

#     df['Toplam Puan'] = 0
#     for c in puan_column_list:
#         df['Toplam Puan'] += df[c]

#     dfo = create_pra_risk_categories(df, 'Toplam Puan')
#     df = pd.merge(df, dfo[['Toplam Puan', 'Risk Kategorisi']], on='Toplam Puan', how='left')

#     df['Takıldığı Toplam Kriter Sayısı'] = 0
#     for c in puan_column_list:
#         df['Takıldığı Toplam Kriter Sayısı'] += df[c].apply(lambda x: min(1, x))


#     dfyr = df.copy()
#     dfyr = dfyr.loc[dfyr['Risk Kategorisi'].isin(['Yüksek', 'Çok Yüksek'])]

#     for c, f in zip(extra_puan_column_list, extra_puan_fun_list):
#         dfyr[c] = dfyr.apply(f, axis=1)

#     dfyr['Toplam Ekstra Puan'] = 0
#     for c in extra_puan_column_list:
#         dfyr['Toplam Ekstra Puan'] += dfyr[c]

#     dfoe = create_pra_risk_categories(dfyr, 'Toplam Ekstra Puan', [0.25, 0.50, 0.75], ['I', 'II', 'III', 'IV'], 'Ekstra Risk Kategorisi', '%')
#     dfoe.loc[dfoe['Toplam Ekstra Puan'] == 0, 'Ekstra Risk Kategorisi'] = '0'

#     dfyr = pd.merge(dfyr, dfoe[['Toplam Ekstra Puan', 'Ekstra Risk Kategorisi']], on='Toplam Ekstra Puan', how='left')


#     dfyr['Takıldığı Toplam Ekstra Kriter Sayısı'] = 0
#     for c in extra_puan_column_list:
#         dfyr['Takıldığı Toplam Ekstra Kriter Sayısı'] += dfyr[c].apply(lambda x: min(1, x))

#     df = pd.merge(df, dfyr.loc[:, ['Müşteri No'] + [x for x in list(dfyr.columns) if x not in list(df.columns)]], on='Müşteri No', how='left')
#     dfyr = dfyr.loc[dfyr['Risk Kategorisi'] == 'Çok Yüksek']

#     return df.copy(), dfyr.copy(), dfo.copy(), dfoe.copy()

## Sunum Fonksiyonları

In [86]:
# def create_pra_sunum(df_pra, zombi=False):
#     dfp = df_pra.copy()
#     # dfp = dfp.loc[dfp['Müşteri Sınıfı'] == 1]

#     # n_list = ['Yüksek', 'Makul', 'Düşük', 'Çok Düşük']
#     n_list = ['Diğer Banka Takip', 'Çok Yüksek', 'Yüksek', 'Orta', 'Düşük', 'Çok Düşük', 'Mükemmel']
#     z = 'Risk Kategorisi'
#     yrc = 'Çok Yüksek Riskli Portföy Oranı'
#     m = 'Müşteri No'
#     r = 'Nakdi Reeskontlu Toplam'
#     factor = 1_000_000
#     segment_list = ['KURUMSAL', 'TİCARİ', 'KOBİ', 'İŞLETME']
#     sube_turu_list = ['Karma', 'Kurumsal', 'Ticari']
#     pra_sunum_list = []

#     pscl = ['Değerlendirmeye Esas Segment', 'Şube Türü', 'Entegre Derece Skor', 'Yetki Kodu', 'Tahsis Sektör Adı'] 
#     psil = [segment_list, sube_turu_list, [(1, 5), (6, 7), (8, 12)], None, sektor_list]
#     pstl = [False, False, True, False, False]
#     psol = [False, False, False, False, True]

#     if zombi:
#         n_list = [1, 0]
#         nn_list = ['Zombi', 'Zombi Değil']
#         z = '3_ZOMBI_BELIRTEC'
#         yrc = 'Zombi Firma Oranı'

#     for c, c_list, use_thresholds, reorder in zip(pscl, psil, pstl, psol):
#         pra_sub_sunum_list = []
#         for x, find_sum, cc0 in zip([m, r], [False, True], ['Adede Göre', 'Nakdi Riske Göre (mio TL)']):
#             df = pd.DataFrame()
#             if use_thresholds:
#                 ettl = c_list
#                 etl = [f'{et[0]} - {et[1]}' for et in ettl]
#                 df[c] = etl
#             else:
#                 df[c] = c_list if c_list is not None else sorted(df_pra.loc[df_pra[c].notnull(), c].unique())
#             t_list = ['Genel Toplam']

#             for n in n_list:
#                 dft = dfp.copy()
#                 if find_sum:
#                     dft = dft.loc[dft[z] == n, [x, c]].groupby(c).sum().reset_index().rename(columns={x: n})
#                 else:
#                     dft = dft.loc[dft[z] == n, [x, c]].groupby(c).count().reset_index().rename(columns={x: n})
#                 if use_thresholds:
#                     nx_list = []
#                     for et in ettl:
#                         nx = dft.loc[(dft[c] >= et[0]) & (dft[c] <= et[1]), n].sum()
#                         nx_list.append(nx)
#                     df[n] = nx_list
#                 else:
#                     df = pd.merge(df, dft, on=c, how='outer')
#                 t_list.append(df[n].sum())
        
#             n = 'Toplam'
#             dft = dfp.copy()
#             if find_sum:
#                 dft = dft[[x, c]].groupby(c).sum().reset_index().rename(columns={x: n})
#             else:
#                 dft = dft[[x, c]].groupby(c).count().reset_index().rename(columns={x: n})
#             if use_thresholds:
#                 nx_list = []
#                 for et in ettl:
#                     nx = dft.loc[(dft[c] >= et[0]) & (dft[c] <= et[1]), n].sum()
#                     nx_list.append(nx)
#                 df[n] = nx_list
#             else:
#                 df = pd.merge(df, dft, on=c, how='outer')
#             t_list.append(df[n].sum())

#             df.loc[len(df)] = t_list
#             if find_sum:
#                 df.iloc[:,1:] = df.iloc[:,1:] / factor
#             # df[yrc] = df[n_list[0]] / df['Toplam']
#             df[yrc] = (df[n_list[0]] + df[n_list[1]]) / df['Toplam']
#             df = df.dropna(axis=0, how='all', subset=list(df.columns)[1:], ignore_index=True)
#             if reorder:
#                 df.iloc[:-1, :] = df.iloc[:-1, :].sort_values(yrc, ascending=False)
#             if zombi:
#                 for n, nn in zip(n_list, nn_list):
#                     df = df.rename(columns={n: nn})
#             df = insert_header(df, cc0)
#             pra_sub_sunum_list.append(df)
#         pra_sunum_list.append(h_stack(pra_sub_sunum_list, 2))

#     return v_stack(pra_sunum_list)


# def create_count_risk_dist(dfm, c, zombi=False, factor=1_000_000):
#     m = 'Müşteri No'
#     x = 'Nakdi Reeskontlu Toplam'
#     # k = 'Nakdi Karşılık'
#     l = 'LGD'

#     nc = 'Adet'
#     nx = 'Nakdi Risk'
#     # nk = 'Ortalama Nakdi Karşılık Oranı'
#     nl = 'Ortalama LGD'

#     br = 'Bankamız Risk'
#     bl = 'Bankamız Limit'
#     mr = 'Memzuç Risk'
#     ml = 'Memzuç Limit'

#     # cr_column_list = [x, k, br, bl, mr, ml]
#     cr_column_list = [x, br, bl, mr, ml]
#     if zombi:
#         dfp = dfm[[m, c, x]].copy()
#     else:
#         dfp = dfm[[m, c, l] + cr_column_list].copy()
#     dfp = dfp.loc[dfp[c].notnull()]
#     r_list = sorted(list(dfp[c].unique()))

#     df = pd.DataFrame()
#     df[c] = r_list

#     dft = dfp[[m, c]].groupby(c).count().reset_index().rename(columns={m: nc})
#     df = pd.merge(df, dft, on=c, how='left')
#     sc = df[nc].sum()
#     df['%'] = df[nc] / sc

#     dft = dfp[[x, c]].groupby(c).sum().reset_index().rename(columns={x: nx})
#     df = pd.merge(df, dft, on=c, how='left')
#     sx = df[nx].sum()
#     df['% '] = df[nx] / sx

#     t_list = ['Genel Toplam', sc, 1, sx, 1]


#     if not zombi:
#         for r in cr_column_list:
#             dft = dfp[[r, c]].groupby(c).sum().reset_index()
#             df = pd.merge(df, dft, on=c, how='left')
            
#         # df[nk] = df[k] / df[x]
#         # sk = df[k].sum() / df[x].sum()
        
#         dft = dfp[[l, c]].groupby(c).mean().reset_index().rename(columns={l: nl})
#         df = pd.merge(df, dft, on=c, how='left')
#         sl = dfp[l].mean()

#         df['Bankamız Memzuç Risk Payı'] = df[br] / df[mr]
#         df['Bankamız Limit Doluluk Oranı'] = df[br] / df[bl]
#         df['Memzuç Limit Doluluk Oranı'] = df[mr] / df[ml]

#         bmrp = df[br].sum() / df[mr].sum()
#         bldo = df[br].sum() / df[bl].sum()
#         mldo = df[mr].sum() / df[ml].sum()

#         # t_list += [sk, sl, bmrp, bldo, mldo]
#         t_list += [sl, bmrp, bldo, mldo]

#         df = df.drop(cr_column_list, axis=1)
#     df = df.sort_values('%', ascending=False).reset_index(drop=True)
#     df.loc[len(df)] = t_list

#     df[nx] /= factor

#     return df

## Firma Kümesi

In [87]:
# c0 = 'Bankamız Ticari Portföy ({})'.format(last_date)
# mn = 'Müşteri No'
# nr = 'Nakdi Reeskontlu Toplam'
# gnr = 'G.Nakdi Toplam'
# tr = 'Toplam Reeskontlu Risk'
# c_list = ['Küme', 'Adet', 'Nakdi Risk', 'G.Nakdi Risk', 'Toplam Reeskontlu Risk']

# df = df_master.copy()
# d = 'Çıkarılma Sebebi'
# dfd = pd.DataFrame(columns=list(df_master.columns) + [d])
# dfk = pd.DataFrame(columns=c_list)

# c = c0
# dfk.loc[len(dfk)] = return_excluded_info(c, df, False)

# c = 'Standart Nitelikli Ticari Portföy'
# dfe = df.loc[(df['Müşteri Sınıfı'] != 1) | (df['İlgili Bölüm'] == 'Krediler İzleme Bölümü')]
# df = df.loc[(df['Müşteri Sınıfı'] == 1) & (df['İlgili Bölüm'] != 'Krediler İzleme Bölümü')]
# dfk.loc[len(dfk)] = return_excluded_info(c, df, False)

# c = 'Banka Risk Grubu, Hazine, Finans, PPP & Belediyeler'
# pra_sektor_exclude_list = [
#     'FİNANS',
#     'PPP (Kamu Garantili KÖİ) (HASTANE VE OTOYOLLAR)',
#     'BELEDİYELER VE OSB\'LER',
#     'BELEDİYELER VE OSB LER'
# ]
# dfe = df.loc[(df['Risk Grubu Kodu'] == 'BANKRG') | (df['Tahsis Sektör Adı'].isin(pra_sektor_exclude_list))]
# df = df.loc[(df['Risk Grubu Kodu'] != 'BANKRG') & (~df['Tahsis Sektör Adı'].isin(pra_sektor_exclude_list))]
# dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)

# c = 'Yurtdışı, Kıbrıs & Özel Bankacılık'
# dfe = df.loc[df['Şube Türü'].isin(['Kıbrıs', 'Y.dışı', 'Özel Bankacılık'])]
# df = df.loc[~df['Şube Türü'].isin(['Kıbrıs', 'Y.dışı', 'Özel Bankacılık'])]
# dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)

# c = 'Toplam Risk < 1 mio TL'
# dfe = df.loc[(df['Toplam Reeskontlu Risk'] < 1_000_000) | (df['FST_YR'].isnull())]
# df = df.loc[(df['Toplam Reeskontlu Risk'] >= 1_000_000) & (~df['FST_YR'].isnull())]
# dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)

# dfk.loc[len(dfk)] = [np.nan] + [np.sum([min(x, 0) for x in dfk[x]]) for x in dfk.columns[1:]]

# c = 'İncelenen Firmalar'
# dfe = df.copy()
# dfk.loc[len(dfk)] = return_excluded_info(c, dfe, False)
# dfk.loc[len(dfk)] = ['İncelenen Firmalar/Ticari Portföy'] + return_percentages('İncelenen Firmalar')[1:]

# dfk = add_h_space(dfk, 3)

# dfc = df[['Müşteri No', 'FST_YR']].groupby('FST_YR').count().reset_index()
# dfc.columns = ['Mali Yıl', 'Adet']
# dfc = dfc.sort_values('Mali Yıl', ascending=False).reset_index(drop=True)
# t = dfc['Adet'].sum()
# dfc['%'] = dfc['Adet'] / t
# dfc.loc[len(dfc)] = ['TOPLAM', t, 1]

# dfr = df[['Nakdi Reeskontlu Toplam', 'FST_YR']].groupby('FST_YR').sum().reset_index()
# dfr.columns = ['Mali Yıl', 'Nakdi Risk']
# dfr = dfr.sort_values('Mali Yıl', ascending=False).reset_index(drop=True)
# t = dfr['Nakdi Risk'].sum()
# dfr['% '] = dfr['Nakdi Risk'] / t
# dfr.loc[len(dfr)] = ['TOPLAM', t, 1]
# dfr['Nakdi Risk'] /= 1_000_000

# dfy = pd.merge(dfc, dfr, on='Mali Yıl')
# dfk = v_stack([dfk, dfy])

# df_pra_kume = dfk.copy()
# df_pra_cikarilan = dfd.copy()
# df_pra = df.copy()

## Analiz

In [88]:
# df_pra, df_pra_yr, df_pra_ozet, df_pra_ozet_ekstra = create_pra_analysis(df_pra)
# df_pra_sunum = create_pra_sunum(df_pra)

# df_pra_ard_segment = create_count_risk_dist(df_pra, 'Değerlendirmeye Esas Segment')
# df_pra_ard_sektor = create_count_risk_dist(df_pra, 'Tahsis Sektör Adı')
# df_pra_ard_sektor_yr = create_count_risk_dist(df_pra_yr, 'Tahsis Sektör Adı')

# dfpaas = insert_header(df_pra_ard_sektor, 'İncelenen Portföy İçerisinde')
# dfpaasyl = insert_header(df_pra_ard_sektor_yr, 'Yüksek Riskli Portföy İçerisinde')
# df_pra_ard = h_stack([dfpaas, dfpaasyl], 1)

In [89]:
# # df_zombi_sunum = create_pra_sunum(df_zombi, True)
# # df_zombi_yr = df_zombi.loc[df_zombi['3_ZOMBI_BELIRTEC'] == 1]

# # df_zombi_ard_segment = create_count_risk_dist(df_zombi, 'Değerlendirmeye Esas Segment', True)
# # df_zombi_ard_sektor = create_count_risk_dist(df_zombi, 'Tahsis Sektör Adı', True)
# # df_zombi_ard_sektor_yr = create_count_risk_dist(df_zombi_yr, 'Tahsis Sektör Adı', True)

# # dfzaas = insert_header(df_zombi_ard_sektor, 'İncelenen Portföy İçerisinde')
# # dfzaasyl = insert_header(df_zombi_ard_sektor_yr, 'Zombi Firmalar Arasında')
# # df_zombi_ard = h_stack([dfzaas, dfzaasyl], 1)

## Korelasyon Matrisi

In [90]:
# # show_one_half = False
# # save_figure = True
# # rearrange = True
# # correlation_matrix_file_name  ='pra_correlation_matrix_pra_{}.png'.format(last_date)

# # x = ['Toplam Puan'] + puan_column_list
# # x = [y for y in x if y != 'Diğer Banka Tahakkuk Puanı']
# # dfcmt = df_pra[x].copy()
# # dfcmt.columns = x
# # correlation_matrix = dfcmt.corr()

# # if rearrange:
# #     x = list(correlation_matrix.sort_values('Toplam Puan', ascending=False).index)
# #     dfcmt = df_pra[x].copy()
# #     dfcmt.columns = x
# #     correlation_matrix = dfcmt.corr()

# # upper_half = np.triu(correlation_matrix) if show_one_half else None
# # # fig = plt.figure(figsize=(df.shape[1], df.shape[1]), dpi=480)
# # # sns.heatmap(correlation_matrix, annot=True, fmt='.2f', square=True, mask=upper_half)

# # fig = plt.figure(figsize=(12, 12), dpi = 350)
# # sns.heatmap(correlation_matrix, annot=True, fmt='.2f', square=True, mask=upper_half)

# # if save_figure:
# #     plt.savefig(output_folder_path + correlation_matrix_file_name, bbox_inches = 'tight')

# # plt.show()


In [91]:
# show_one_half = False
# save_figure = True
# rearrange = True
# correlation_matrix_file_name  ='pra_correlation_matrix_pra_yr_{}.png'.format(last_date)

# x = ['Toplam Ekstra Puan'] + extra_puan_column_list
# # dfcmt = df_pra.loc[df_pra['Risk Kategorisi'].isin([x].copy()
# dfcmt = df_pra.copy()
# dfcmt = dfcmt.loc[dfcmt['Toplam Ekstra Puan'].notnull(), x]
# dfcmt.columns = x
# correlation_matrix = dfcmt.corr()

# if rearrange:
#     x = list(correlation_matrix.sort_values('Toplam Ekstra Puan', ascending=False).index)
#     dfcmt = df_pra[x].copy()
#     dfcmt.columns = x
#     correlation_matrix = dfcmt.corr()

# upper_half = np.triu(correlation_matrix) if show_one_half else None
# # fig = plt.figure(figsize=(df.shape[1], df.shape[1]), dpi=480)
# # sns.heatmap(correlation_matrix, annot=True, fmt='.2f', square=True, mask=upper_half)

# fig = plt.figure(figsize=(12, 12), dpi = 350)
# sns.heatmap(correlation_matrix, annot=True, fmt='.2f', square=True, mask=upper_half)

# if save_figure:
#     plt.savefig(output_folder_path + correlation_matrix_file_name, bbox_inches = 'tight')

# plt.show()


In [92]:
# ff = []
# fn = []
# tsa = 'Tahsis Sektör Adı'
# dd = df_pra.copy()
# dd = dd.loc[dd[tsa].notnull()]
# ll = sorted(dd[tsa].unique())

# for x in ll:
#     df = dd.copy()
#     df = df.loc[(df['Tahsis Sektör Adı'] == x) & (df['Risk Kategorisi'] == 'Yüksek')]
#     df = df.sort_values('Nakdi Reeskontlu Toplam', ascending=False)
#     ff.append(df)
#     fn.append(x[:15])

# quick_export(ff, 'Yüksek Riskli - Sektörel Dağılım', fn)

In [93]:
# riskli_list = df_pra.loc[df_pra['Risk Kategorisi'] == 'Yüksek Riskli', 'Müşteri No']
# def cross_check(df):
#     if df['Müşteri No'] in riskli_list:
#         return 1
#     return 0

# df_zombi['Cross'] = df_zombi.apply(cross_check, axis = 1)

# zombi_list = df_zombi.loc[df_zombi['3_ZOMBI_BELIRTEC'] == 1, 'Müşteri No']
# def cross_check_z(df):
#     if df['Müşteri No'] in zombi_list:
#         return 1
#     return 0
# df_pra_yl = df_pra.loc[df_pra['Risk Kategorisi'] == 'Yüksek Riskli']
# df_pra_yl['Cross'] = df_pra_yl.apply(cross_check_z, axis = 1)

# Yedek

In [94]:
df_wide_backup = df_wide.copy()
df_kr202_last_backup = df_kr202_last.copy()
df_master_backup = df_master.copy()
cek_df_list_backup = cek_df_list.copy()

# Filtre

In [380]:
df_wide = df_wide_backup.copy()
df_kr202_last = df_kr202_last_backup.copy()
df_master = df_master_backup.copy()
cek_df_list = cek_df_list_backup.copy()

In [420]:
apply_filter = False
filter_column = 'Değerlendirmeye Esas Segment'
filter_value = 'İŞLETME'

if apply_filter:
    wide_columns = []
    for c in df_wide[-1].columns:
        if c.endswith(' ' + last_date):
            x = c[:-(len(last_date) + 1)]
            wide_columns.append(x)

    dfn = pd.DataFrame()
    first = True
    for i, (df, d) in enumerate(zip(df_wide, sorted_date_list)):
        dft = df.loc[df[filter_column + ' ' + d] == filter_value]
        df_wide[i] = dft.copy()

    for i, (df, d) in enumerate(zip(cek_df_list, sorted_date_list)):
        dft = df.loc[df[filter_column + ' ' + d] == filter_value]
        cek_df_list[i] = dft.copy()

    # df_kr202_last = df_kr202_last.loc[df_kr202_last[filter_column] == filter_value]
    df_master = df_master.loc[df_master[filter_column] == filter_value]

# Sunum Fonksiyonları

## create_summary()

In [383]:
def create_summary(r, sort_type=-2, fill_na=False, order_list=None, firma_sayisi=True, firma_sayisi_yuzde=True, nakdi_risk=True, nakdi_risk_yuzde=True, gayri_nakdi_risk=True, gayri_nakdi_risk_yuzde=True, toplam_risk=True, toplam_risk_yuzde=True,tl_yp_risk=True, tl_yp_risk_yuzde=True, memzuc=True, memzuc_payi=True, limit_doluluk=True, ortalama_limit_doluluk=True, yapay_zeka=True, entegre_derece=True, musteri_sinifi=True, finansal_gucluk=True, kroa=True, gecikme=True, include_sum=True, drop_empty=True, df_master=df_master):
    c_list = [
        'Müşteri No',
        'Nakdi TL Reeskontlu',
        'Nakdi YP Reeskontlu',
        'Nakdi Reeskontlu Toplam',
        'G.Nakdi TL Bakiye',
        'G.Nakdi YP Bakiye',
        'G.Nakdi Toplam',
        'Toplam Reeskontlu Risk',
        'Memzuç Risk',
        'Memzuç Limit',
        'Bankamız Risk',
        'Bankamız Limit',
        'Sektör Limit Doluluk Oranı (%)',
        'Bankamız Limit Doluluk Oranı (%)',
        'Yapay Zeka Risk Sınıfı',
        'Entegre Derece Skor',
        'Müşteri Sınıfı',
        'Finansal Güçlük',
        'KRÖA',
        'Gecikme'
    ]

    bl, br = 'Bankamız Limit', 'Bankamız Risk'
    ml, mr = 'Memzuç Limit', 'Memzuç Risk'

    if r not in c_list:
        c_list.append(r)
        
    dfm = df_master[c_list].copy()

    if fill_na:
        dfm[r] = dfm[r].fillna('-')
    else:
        dfm = dfm.loc[dfm[r].notnull()]

    df = pd.DataFrame()

    if order_list is None:
        df[r] = dfm.loc[dfm[r].notnull(), r].unique()
    else:
        df[r] = order_list

    n_list = [r]
    t_list = ['TOPLAM']
    space = 0


    if firma_sayisi:
        c = 'Müşteri No'
        n = 'Firma Sayısı'
        dff = dfm[[r, c]]
        dft = dff.groupby(r).count().rename(columns={c: n}).reset_index()
        df = pd.merge(df, dft, how='left', on=r)
        n_list.append(n)
        t = len(dff)
        t_list.append(t)

        if firma_sayisi_yuzde:
            nn = '%'  + ' ' * space
            space += 1
            df[nn] = df[n] / t * 100
            n_list.append(nn)
            t_list.append(100)


    if nakdi_risk:
        c = 'Nakdi Reeskontlu Toplam'
        n = 'Nakdi Risk'
        dff = dfm[[r, c]]
        dft = dff.groupby(r).sum().rename(columns={c: n}).reset_index()
        df = pd.merge(df, dft, how='left', on=r)
        n_list.append(n)
        t = dff[c].sum()
        t_list.append(t)

        if nakdi_risk_yuzde:
            nn = '%'  + ' ' * space
            space += 1
            df[nn] = df[n] / t * 100
            n_list.append(nn)
            t_list.append(100)


    if gayri_nakdi_risk:
        c = 'G.Nakdi Toplam'
        n = 'Gayrinakdi Risk'
        dff = dfm[[r, c]]
        dft = dff.groupby(r).sum().rename(columns={c: n}).reset_index()
        df = pd.merge(df, dft, how='left', on=r)
        n_list.append(n)
        t = dff[c].sum()
        t_list.append(t)

        if gayri_nakdi_risk_yuzde:
            nn = '%'  + ' ' * space
            space += 1
            df[nn] = df[n] / t * 100
            n_list.append(nn)
            t_list.append(100)
            

    if toplam_risk:
        c = 'Toplam Reeskontlu Risk'
        n = 'Toplam Risk'
        dff = dfm[[r, c]]
        dft = dff.groupby(r).sum().rename(columns={c: n}).reset_index()
        df = pd.merge(df, dft, how='left', on=r)
        n_list.append(n)
        t = dff[c].sum()
        t_list.append(t)

        if toplam_risk_yuzde:
            nn = '%'  + ' ' * space
            space += 1
            df[nn] = df[n] / t * 100
            n_list.append(nn)
            t_list.append(100)


    if tl_yp_risk:
        for c_n, c_g, n in zip(['Nakdi TL Reeskontlu', 'Nakdi YP Reeskontlu'], ['G.Nakdi TL Bakiye', 'G.Nakdi YP Bakiye'], ['Toplam TL Risk', 'Toplam YP Risk']):
            dff = dfm[[r, c_n, c_g]]
            dft = dff.groupby(r).sum().reset_index()
            dft[n] = dft[c_n] + dft[c_g]
            df = pd.merge(df, dft, how='left', on=r)
            n_list.append(n)
            t = df[n].sum()
            t_list.append(t)

            if tl_yp_risk_yuzde:
                nn = '%'  + ' ' * space
                space += 1
                df[nn] = df[n] / df['Toplam Risk'] * 100
                t = df[n].sum() / df['Toplam Risk'].sum() * 100
                n_list.append(nn)
                t_list.append(t)


    if memzuc:
        for c in [mr, ml, br, bl]:
            n = c
            dff = dfm[[r, c]]
            dft = dff.groupby(r).sum().rename(columns={c: n}).reset_index()
            df = pd.merge(df, dft, how='left', on=r)
            n_list.append(n)
            t = dff[c].sum()
            t_list.append(t)


        if memzuc_payi:
            for c_1, c_2, n in zip([bl, br], [ml, mr], ['Bankamız Memzuç Limit Payı (%)', 'Bankamız Memzuç Risk Payı (%)']):
                df[n] = df[c_1] / df[c_2] * 100
                t = df[c_1].sum() / df[c_2].sum() * 100
                n_list.append(n)
                t_list.append(t)


        if limit_doluluk:
            for c_1, c_2, n in zip([mr, br], [ml, bl], ['Sektör Limit Doluluk Oranı (%)', 'Bankamız Limit Doluluk Oranı (%)']):
                df[n] = df[c_1] / df[c_2] * 100
                t = df[c_1].sum() / df[c_2].sum() * 100
                n_list.append(n)
                t_list.append(t)

        if ortalama_limit_doluluk:
            c = 'Sektör Limit Doluluk Oranı (%)'
            n = 'Sektör Ortalama Limit Doluluk Oranı (%)'
            dff = dfm[[r, c]]
            dft = dff.groupby(r).mean().rename(columns={c: n}).reset_index()
            df = pd.merge(df, dft, how='left', on=r)
            df[n] = df[n]
            n_list.append(n)
            t = dff[c].mean()
            t_list.append(t)
        
            c = 'Bankamız Limit Doluluk Oranı (%)'
            n = 'Bankamız Ortalama Limit Doluluk Oranı (%)'
            dff = dfm[[r, c]]
            dft = dff.groupby(r).mean().rename(columns={c: n}).reset_index()
            df = pd.merge(df, dft, how='left', on=r)
            df[n] = df[n]
            n_list.append(n)
            t = dff[c].mean()
            t_list.append(t)


    if yapay_zeka:
        c = 'Yapay Zeka Risk Sınıfı'
        x = 'Nakdi Reeskontlu Toplam'
        n = 'Yapay Zeka Risk Sınıfı Ortalaması'
        dff = dfm[[r, c, x]]
        dff = dff.loc[dff[c].notnull()]
        dft = dff[[r, c]].groupby(r).mean().rename(columns={c: n}).reset_index()
        df = pd.merge(df, dft, how='left', on=r)
        n_list.append(n)
        t = dff[c].mean()
        t_list.append(t)

        n = 'Nakdi Risk Ağırlıklı Yapay Zeka Risk Sınıfı Ortalaması'
        cx = 'Yapay Zeka Risk Sınıfı * Nakdi Risk'
        dff = dfm[[r, c, x]]
        dff = dff.loc[dff[c].notnull()]
        dff[cx] = dff[c] * dff[x] 
        dft = dff[[r, cx, x]].groupby(r).sum().reset_index()
        dft[n] = dft[cx] / dft[x]
        dft = dft[[r, n]]
        df = pd.merge(df, dft, how='left', on=r)
        n_list.append(n)
        t = dff[cx].sum() / dff[x].sum()
        t_list.append(t)

        n = 'Yapay Zeka Risk Sınıfı >= 4 (Adede Göre %)'
        df1 = dff[[r, x]].groupby(r).count().reset_index()
        df1.columns = [r, 'Tüm']
        df2 = dff.loc[dff[c] >= 4, [r, x]].groupby(r).count().reset_index()
        df2.columns = [r, 'Riskli']
        dft = pd.merge(df1, df2, on=r)
        dft[n] = dft['Riskli'] / dft['Tüm'] * 100
        df = pd.merge(df, dft[[r, n]], how='left', on=r)
        n_list.append(n)
        t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
        t_list.append(t)

        n = 'Yapay Zeka Risk Sınıfı >= 4 (Riske Göre %)'
        df1 = dff[[r, x]].groupby(r).sum().reset_index()
        df1.columns = [r, 'Tüm']
        df2 = dff.loc[dff[c] >= 4, [r, x]].groupby(r).sum().reset_index()
        df2.columns = [r, 'Riskli']
        dft = pd.merge(df1, df2, on=r)
        dft[n] = dft['Riskli'] / dft['Tüm'] * 100
        df = pd.merge(df, dft[[r, n]], how='left', on=r)
        n_list.append(n)
        t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
        t_list.append(t)


    if entegre_derece:
        c = 'Entegre Derece Skor'
        x = 'Nakdi Reeskontlu Toplam'
        n = 'Entegre Derece Skor Ortalaması'
        dff = dfm[[r, c, x]]
        dff = dff.loc[dff[c].notnull()]
        dft = dff[[r, c]].groupby(r).mean().rename(columns={c: n}).reset_index()
        df = pd.merge(df, dft, how='left', on=r)
        n_list.append(n)
        t = dff[c].mean()
        t_list.append(t)

        n = 'Nakdi Risk Ağırlıklı Entegre Derece Skor Ortalaması'
        cx = 'Entegre Derece Skor * Nakdi Risk'
        dff = dfm[[r, c, x]]
        dff = dff.loc[dff[c].notnull()]
        dff[cx] = dff[c] * dff[x] 
        dft = dff[[r, cx, x]].groupby(r).sum().reset_index()
        dft[n] = dft[cx] / dft[x]
        dft = dft[[r, n]]
        df = pd.merge(df, dft, how='left', on=r)
        n_list.append(n)
        t = dff[cx].sum() / dff[x].sum()
        t_list.append(t)

        n = 'Entegre Derece Skor >= 8 (Adede Göre %)'
        df1 = dff[[r, x]].groupby(r).count().reset_index()
        df1.columns = [r, 'Tüm']
        df2 = dff.loc[dff[c] >= 8, [r, x]].groupby(r).count().reset_index()
        df2.columns = [r, 'Riskli']
        dft = pd.merge(df1, df2, on=r)
        dft[n] = dft['Riskli'] / dft['Tüm'] * 100
        df = pd.merge(df, dft[[r, n]], how='left', on=r)
        n_list.append(n)
        t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
        t_list.append(t)

        n = 'Entegre Derece Skor >= 8 (Riske Göre %)'
        df1 = dff[[r, x]].groupby(r).sum().reset_index()
        df1.columns = [r, 'Tüm']
        df2 = dff.loc[dff[c] >= 8, [r, x]].groupby(r).sum().reset_index()
        df2.columns = [r, 'Riskli']
        dft = pd.merge(df1, df2, on=r)
        dft[n] = dft['Riskli'] / dft['Tüm'] * 100
        df = pd.merge(df, dft[[r, n]], how='left', on=r)
        n_list.append(n)
        t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
        t_list.append(t)


    if musteri_sinifi:
        c = 'Müşteri Sınıfı'
        x = 'Nakdi Reeskontlu Toplam'
        n = 'Yakın İzleme (Adede Göre %)'
        dff = dfm[[r, c, x]]
        dff = dff.loc[dff[c].notnull()]

        df1 = dff[[r, x]].groupby(r).count().reset_index()
        df1.columns = [r, 'Tüm']
        df2 = dff.loc[dff[c] >= 2, [r, x]].groupby(r).count().reset_index()
        df2.columns = [r, 'Riskli']
        dft = pd.merge(df1, df2, on=r)
        dft[n] = dft['Riskli'] / dft['Tüm'] * 100
        df = pd.merge(df, dft[[r, n]], how='left', on=r)
        n_list.append(n)
        t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
        t_list.append(t)

        n = 'Yakın İzleme (Riske Göre %)'
        df1 = dff[[r, x]].groupby(r).sum().reset_index()
        df1.columns = [r, 'Tüm']
        df2 = dff.loc[dff[c] >= 2, [r, x]].groupby(r).sum().reset_index()
        df2.columns = [r, 'Riskli']
        dft = pd.merge(df1, df2, on=r)
        dft[n] = dft['Riskli'] / dft['Tüm'] * 100
        df = pd.merge(df, dft[[r, n]], how='left', on=r)
        n_list.append(n)
        t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
        t_list.append(t)


    if finansal_gucluk:        
        c = 'Finansal Güçlük'
        x = 'Nakdi Reeskontlu Toplam'
        n = 'Finansal Güçlük (Adede Göre %)'
        dff = dfm[[r, c, x]]
        dff = dff.loc[dff[c].notnull()]
        
        df1 = dff[[r, x]].groupby(r).count().reset_index()
        df1.columns = [r, 'Tüm']
        df2 = dff.loc[dff[c] == 1, [r, x]].groupby(r).count().reset_index()
        df2.columns = [r, 'Riskli']
        dft = pd.merge(df1, df2, on=r)
        dft[n] = dft['Riskli'] / dft['Tüm'] * 100
        df = pd.merge(df, dft[[r, n]], how='left', on=r)
        n_list.append(n)
        t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
        t_list.append(t)

        n = 'Finansal Güçlük (Riske Göre %)'
        df1 = dff[[r, x]].groupby(r).sum().reset_index()
        df1.columns = [r, 'Tüm']
        df2 = dff.loc[dff[c] == 1, [r, x]].groupby(r).sum().reset_index()
        df2.columns = [r, 'Riskli']
        dft = pd.merge(df1, df2, on=r)
        dft[n] = dft['Riskli'] / dft['Tüm'] * 100
        df = pd.merge(df, dft[[r, n]], how='left', on=r)
        n_list.append(n)
        t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
        t_list.append(t)


    if kroa:        
        c = 'KRÖA'
        x = 'Nakdi Reeskontlu Toplam'
        n = 'KRÖA (Adede Göre %)'
        dff = dfm[[r, c, x]]
        dff = dff.loc[dff[c].notnull()]
        
        df1 = dff[[r, x]].groupby(r).count().reset_index()
        df1.columns = [r, 'Tüm']
        df2 = dff.loc[dff[c] == 1, [r, x]].groupby(r).count().reset_index()
        df2.columns = [r, 'Riskli']
        dft = pd.merge(df1, df2, on=r)
        dft[n] = dft['Riskli'] / dft['Tüm'] * 100
        df = pd.merge(df, dft[[r, n]], how='left', on=r)
        n_list.append(n)
        t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
        t_list.append(t)

        n = 'KRÖA (Riske Göre %)'
        df1 = dff[[r, x]].groupby(r).sum().reset_index()
        df1.columns = [r, 'Tüm']
        df2 = dff.loc[dff[c] == 1, [r, x]].groupby(r).sum().reset_index()
        df2.columns = [r, 'Riskli']
        dft = pd.merge(df1, df2, on=r)
        dft[n] = dft['Riskli'] / dft['Tüm'] * 100
        df = pd.merge(df, dft[[r, n]], how='left', on=r)
        n_list.append(n)
        t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
        t_list.append(t)


    if gecikme:        
        c = 'Gecikme'
        x = 'Nakdi Reeskontlu Toplam'
        n = 'Gecikme (Adede Göre %)'
        dff = dfm[[r, c, x]]
        dff = dff.loc[dff[c].notnull()]
        
        df1 = dff[[r, x]].groupby(r).count().reset_index()
        df1.columns = [r, 'Tüm']
        df2 = dff.loc[dff[c] == 1, [r, x]].groupby(r).count().reset_index()
        df2.columns = [r, 'Riskli']
        dft = pd.merge(df1, df2, on=r)
        dft[n] = dft['Riskli'] / dft['Tüm'] * 100
        df = pd.merge(df, dft[[r, n]], how='left', on=r)
        n_list.append(n)
        t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
        t_list.append(t)

        n = 'Gecikme (Riske Göre %)'
        df1 = dff[[r, x]].groupby(r).sum().reset_index()
        df1.columns = [r, 'Tüm']
        df2 = dff.loc[dff[c] == 1, [r, x]].groupby(r).sum().reset_index()
        df2.columns = [r, 'Riskli']
        dft = pd.merge(df1, df2, on=r)
        dft[n] = dft['Riskli'] / dft['Tüm'] * 100
        df = pd.merge(df, dft[[r, n]], how='left', on=r)
        n_list.append(n)
        t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
        t_list.append(t)
        

    df = df[n_list]
    if fill_na:
        dfna = df.loc[df[r] == '-']
        df = df.loc[df[r] != '-']
    if order_list is None:
        if sort_type == 1:
            df = df.sort_values(r).reset_index(drop=True)
        elif sort_type == -1:
            df = df.sort_values(r, ascending=False).reset_index(drop=True)
        elif sort_type == 2:
            df = df.sort_values('Nakdi Risk').reset_index(drop=True)
        elif sort_type == -2:
            df = df.sort_values('Nakdi Risk', ascending=False).reset_index(drop=True)
        elif sort_type == 3:
            df = df.sort_values('Firma Sayısı').reset_index(drop=True)
        elif sort_type == -3:
            df = df.sort_values('Firma Sayısı', ascending=False).reset_index(drop=True)
    if fill_na:
        df = pd.concat([df, dfna], ignore_index=True)

    if drop_empty:
        df = df.dropna(axis=0, how='all', subset=list(df.columns)[1:], ignore_index=True)
    if include_sum and len(df) > 1:
        df.loc[len(df)] = t_list
    return df

## create_trend()

In [384]:
def create_trend(row, column, find_sum=False, order_list=None, weight_column=None, factor=1, include_sum=True, drop_empty=True):
    df = pd.DataFrame()

    add = ''
    if factor == 1_000_000_000:
        add = ' (Milyar)'
    elif factor == 1_000_000:
        add = ' (Milyon)'
    elif factor == 1_000:
        add = ' (Bin)'

    middle = ''
    if not find_sum:
        middle = 'Ortalama '

    index = row + ' (Satır) / ' + middle + column + add + ' (Sütun)'

    if order_list is not None:
        df[index] = order_list
    else:
        df[index] = unique_dict[row]

    t_list = ['TOPLAM']

    for i, d in enumerate(sorted_date_list):
        if weight_column is None:
            r = row + ' ' + d
            c = column + ' ' + d

            dfm = df_wide[i][[r, c]].copy()
            dfa = dfm.loc[(dfm[r].notnull()) & (dfm[c].notnull())]
        
            if find_sum:
                dft = dfa.groupby(r).sum().reset_index()
                t = dfa[c].sum()
            else:
                dft = dfa.groupby(r).mean().reset_index()
                t = dfa[c].mean()

        else:
            r = row + ' ' + d
            c = column + ' ' + d
            x = weight_column + ' ' + d

            dfm = df_wide[i][[r, c, x]].copy()
            dfa = dfm.loc[(dfm[r].notnull()) & (dfm[c].notnull())]
            cx = column + ' Ağırlıklı'
            dfa[cx] = dfa[c] * dfa[x]
            dft = dfa[[r, cx, x]].groupby(r).sum().reset_index()
            dft[c] = dft[cx] / dft[x]
            dft = dft[[r, c]]
            t = dfa[cx].sum() / dfa[x].sum()


        dft.columns = [index, d]
        df = pd.merge(df, dft, how='left', on=index)
        t_list.append(t)

    if drop_empty:
        df = df.dropna(axis=0, how='all', subset=list(df.columns)[1:], ignore_index=True)
        
    if include_sum and len(df) > 1:
        df.loc[len(df)] = t_list

    if factor != 1:
        for c in df.columns:
            if c != index:
                df[c] = df[c] / factor
            
    return df


def create_mean_trend(row, column, order_list=None, weight_column=None, factor=1, include_sum=True, drop_empty=True):
    return create_trend(row, column, False, order_list, weight_column, factor, include_sum, drop_empty)

def create_sum_trend(row, column, order_list=None, factor=1, include_sum=True, drop_empty=True):
    return create_trend(row, column, True, order_list, None, factor, include_sum, drop_empty)



def create_threshold_count_trend(row, column, threshold, suffix, find_sum, order_list=None, reverse_order=False, include_sum=True, drop_empty=True, fill_null=None, custom_df=None, custom_nn_list=None):
    df = pd.DataFrame()
    index = row + ' / ' + column + suffix

    if order_list is not None:
        index_list = order_list
    else:
        index_list = unique_dict[row]
    df[index] = index_list
    t_list = ['TOPLAM']
    
    for i, d in enumerate(sorted_date_list):
        r = row + ' ' + d
        c = column + ' ' + d

        if custom_df is not None:
            if type(custom_df) is list:
                dfm = custom_df[i][[r, c]].copy()
            else:
                dfm = custom_df[[r, c]].copy()
        else:
            dfm = df_wide[i][[r, c]].copy()

        if custom_nn_list is not None:
            dfa = dfm.loc[(dfm[r].notnull()) & (dfm[c].isin(custom_nn_list))]
        else:
            dfa = dfm.loc[(dfm[r].notnull()) & (dfm[c].notnull())]

        if type(threshold) is list:
            dff = dfa.loc[dfa[c] == threshold[0]]
        elif reverse_order:
            dff = dfa.loc[dfa[c] <= threshold]
        else:
            dff = dfa.loc[dfa[c] >= threshold]

        if find_sum:
            dftf = dff.groupby(r).count()
            dft = dftf.reset_index().rename(columns={r: index, c: d})
            t = dft[d].sum()
        else:
            dftf = dff.groupby(r).count()
            dfta = dfa.groupby(r).count()
            dft = pd.merge(dftf, dfta, how='outer', on=r).reset_index().rename(columns={r: index})
            dft[d] = dft[c + '_x'] / dft[c + '_y'] * 100
            t1 = len(dff)
            t2 = len(dfa)
            t = t1 / t2 * 100 if t2 != 0 else np.nan

        df = pd.merge(df, dft[[index, d]], how='left', on=index)
        t_list.append(t)

    if drop_empty:
        df = df.dropna(axis=0, how='all', subset=list(df.columns)[1:], ignore_index=True)
        
    if include_sum and len(df) > 1:
        df.loc[len(df)] = t_list
    
    if fill_null is not None:
        df = df.fillna(fill_null)

    return df


def create_count_percent_trend(row, column, threshold, suffix=' (Adede Göre %)', order_list=None, reverse_order=False, include_sum=True, drop_empty=True, fill_null=None, custom_df=None, custom_nn_list=None):
    return create_threshold_count_trend(row, column, threshold, suffix, False, order_list, reverse_order, include_sum, drop_empty, fill_null, custom_df, custom_nn_list)

def create_count_sum_trend(row, column, threshold, suffix=' (Adet)', order_list=None, reverse_order=False, include_sum=True, drop_empty=True, fill_null=None, custom_df=None, custom_nn_list=None):
    return create_threshold_count_trend(row, column, threshold, suffix, True, order_list, reverse_order, include_sum, drop_empty, fill_null, custom_df, custom_nn_list)



def create_threshold_risk_trend(row, column, threshold, suffix, find_sum, order_list=None, reverse_order=False, include_sum=True, drop_empty=True, fill_null=None, factor=1_000_000, custom_df=None, custom_nn_list=None, custom_risk=None):
    df = pd.DataFrame()
    
    add = ''
    if factor == 1_000_000_000:
        add = ' (Milyar TL)'
    elif factor == 1_000_000:
        add = ' (Milyon TL)'
    elif factor == 1_000:
        add = ' (Bin TL)'

    index = row + ' / ' + column + suffix + add

    if order_list is not None:
        index_list = order_list
    else:
        index_list = unique_dict[row]
    df[index] = index_list
    t_list = ['TOPLAM']
    
    for i, d in enumerate(sorted_date_list):
        r = row + ' ' + d
        c = column + ' ' + d
        x = 'Nakdi Reeskontlu Toplam ' + d if custom_risk is None else custom_risk + ' ' + d

        if custom_df is not None:
            if type(custom_df) is list:
                dfm = custom_df[i][[r, c, x]].copy()
            else:
                dfm = custom_df[[r, c, x]].copy()
        else:
            dfm = df_wide[i][[r, c, x]].copy()
        if custom_nn_list is not None:
            dfa = dfm.loc[(dfm[r].notnull()) & (dfm[c].isin(custom_nn_list))]
        else:
            dfa = dfm.loc[(dfm[r].notnull()) & (dfm[c].notnull())]

        if type(threshold) is list:
            dff = dfa.loc[dfa[c] == threshold[0]]
        elif reverse_order:
            dff = dfa.loc[dfa[c] <= threshold, [r, x]]
        else:
            dff = dfa.loc[dfa[c] >= threshold, [r, x]]
        
        dff[x] /=  factor
        dfa[x] /= factor
        
        if find_sum:
            dftf = dff.groupby(r).sum()
            dft = dftf.reset_index().rename(columns={r: index, x: d})
            t = dft[d].sum()
        else:
            dftf = dff.groupby(r).sum()
            dfta = dfa.groupby(r).sum()
            dft = pd.merge(dftf, dfta, how='outer', on=r).reset_index().rename(columns={r: index})
            dft[d] = dft[x + '_x'] / dft[x + '_y'] * 100
            t1 = dff[x].sum()
            t2 = dfa[x].sum()
            t = t1 / t2 * 100 if t2 != 0 else np.nan

        df = pd.merge(df, dft[[index, d]], how='left', on=index)
        t_list.append(t)       

    if drop_empty:
        df = df.dropna(axis=0, how='all', subset=list(df.columns)[1:], ignore_index=True)
    
    if include_sum and len(df) > 1:
        df.loc[len(df)] = t_list
    
    if fill_null is not None:
        df = df.fillna(fill_null)
        
    return df


def create_risk_percent_trend(row, column, threshold, suffix=' (Riske Göre %)', order_list=None, reverse_order=False, include_sum=True, drop_empty=True, fill_null=None, factor=1_000_000, custom_df=None, custom_nn_list=None, custom_risk=None):
    return create_threshold_risk_trend(row, column, threshold, suffix, False, order_list, reverse_order, include_sum, drop_empty, fill_null, factor, custom_df, custom_nn_list, custom_risk)

def create_risk_sum_trend(row, column, threshold, suffix=' (Risk)', order_list=None, reverse_order=False, include_sum=True, drop_empty=True, fill_null=None, factor=1_000_000, custom_df=None, custom_nn_list=None, custom_risk=None):
    return create_threshold_risk_trend(row, column, threshold, suffix, True, order_list, reverse_order, include_sum, drop_empty, fill_null, factor, custom_df, custom_nn_list, custom_risk)

## create_multi_trend()

In [385]:
def create_multi_trend(row, column, threshold, order_list=None, reverse_order=False, include_sum=True, suffix_list=[' (Adede Göre %)', ' (Adet)', ' (Riske Göre %)', ' (Risk)'], drop_empty=True, fill_null=None, factor=1_000_000, custom_df=None, custom_nn_list=None, custom_risk=None):
    df1 = create_count_percent_trend(row, column, threshold, suffix_list[0], order_list, reverse_order, include_sum, drop_empty, fill_null, custom_df, custom_nn_list)
    df2 = create_count_sum_trend(row, column, threshold, suffix_list[1], order_list, reverse_order, include_sum, drop_empty, fill_null, custom_df, custom_nn_list)
    df3 = create_risk_percent_trend(row, column, threshold, suffix_list[2], order_list, reverse_order, include_sum, drop_empty, fill_null, factor, custom_df, custom_nn_list, custom_risk)
    df4 = create_risk_sum_trend(row, column, threshold, suffix_list[3], order_list, reverse_order, include_sum, drop_empty, fill_null, factor, custom_df, custom_nn_list, custom_risk)

    return [df1, df2, df3, df4]

def combine_multi_trend(multi_trend):
    df1 = multi_trend[0].copy()
    df2 = multi_trend[1].copy()
    df3 = multi_trend[2].copy()
    df4 = multi_trend[3].copy()

    n_space = h_space if len(df1) < h_limit else l_space
    for i in range(n_space):
        df1.loc[len(df1)] = np.nan
        df2.loc[len(df2)] = np.nan

    df1.loc[len(df1)] = df3.columns
    df3.columns = df1.columns
    df1 = pd.concat([df1, df3], ignore_index=True)

    df2.loc[len(df1)] = df4.columns
    df4.columns = df2.columns
    df2 = pd.concat([df2, df4], ignore_index=True)
    
    for i in range(w_space):
        df1[' ' * i] = np.nan

    df = pd.concat([df1, df2], axis=1)

    return df

## create_matrix()

In [386]:
def create_matrix(column, threshold, ignore_na=True, old_date=None, new_date=None):
    if old_date is None:
        old_date = sorted_date_list[-2]
    if new_date is None:
        new_date = sorted_date_list[-1]

    old_index = sorted_date_list.index(old_date)
    new_index = sorted_date_list.index(new_date)

    c_old = column + ' ' + old_date
    c_new = column + ' ' + new_date
    index = column + ' - ' + old_date + ' (Satır) / ' + new_date + ' (Sütun)'

    df = pd.DataFrame()
    df_old = df_wide[old_index][['Müşteri No', c_old]]
    df_new = df_wide[new_index][['Müşteri No', c_new]]

    dfm = pd.merge(df_old, df_new, on='Müşteri No', how='inner')
    dfo = dfm.copy()
    dfm[c_old] = dfm[c_old].fillna('-')
    dfm[c_new] = dfm[c_new].fillna('-')

    c_list = list(dfm[c_old].unique())
    if '-' in c_list:
        c_list = sorted(list(filter(lambda x: x != '-', c_list)))
        if not ignore_na:
            c_list += ['-']
    else:
        c_list = sorted(c_list)
    df[index] = c_list + ['TOPLAM']

    for c in c_list:
        a_list = []
        a_sum = 0
        for r in c_list:
            a = len(dfm.loc[(dfm[c_old] == r) & (dfm[c_new] == c)])
            a_list.append(a)
            a_sum += a
        a_list.append(a_sum)
        df[c] = a_list

    df['TOPLAM'] = 0
    for c in c_list:
        df['TOPLAM'] += df[c]

    dfb, dfg = pd.DataFrame(), pd.DataFrame()
    dfb['Müşteri No'] = dfo.loc[(dfo[c_old] < threshold) & (dfo[c_new] >= threshold), 'Müşteri No']
    dfg['Müşteri No'] = dfo.loc[(dfo[c_old] >= threshold) & (dfo[c_new] < threshold), 'Müşteri No']
    dfb = pd.merge(dfb, df_master[['Müşteri No', 'Adı', 'Değerlendirmeye Esas Segment', 'Tahsis Sektör Adı']], how='left', on='Müşteri No')
    dfg = pd.merge(dfg, df_master[['Müşteri No', 'Adı', 'Değerlendirmeye Esas Segment', 'Tahsis Sektör Adı']], how='left', on='Müşteri No')
    dfb = dfb.rename(columns={'Müşteri No': 'Müşteri No ({} {} → {})'.format(column, threshold - 1, threshold)})
    dfg = dfg.rename(columns={'Müşteri No': 'Müşteri No ({} {} → {})'.format(column, threshold, threshold - 1)})
    
    return df, dfb, dfg


def create_matrix_condensed(column, threshold, ignore_na=True, old_date=None, new_date=None):
    if old_date is None:
        old_date = sorted_date_list[-2]
    if new_date is None:
        new_date = sorted_date_list[-1]

    old_index = sorted_date_list.index(old_date)
    new_index = sorted_date_list.index(new_date)

    c_old = column + ' ' + old_date
    c_new = column + ' ' + new_date
    index = column + ' - ' + old_date + ' (Satır) / ' + new_date + ' (Sütun)'

    df = pd.DataFrame()
    df_old = df_wide[old_index][['Müşteri No', c_old]]
    df_new = df_wide[new_index][['Müşteri No', c_new]]

    dfm = pd.merge(df_old, df_new, on='Müşteri No', how='inner')

    c_all = list(dfm[c_old].unique())
    c_list = []
    c_min = int(min(c_all))
    c_max = int(max(c_all))
    if c_min == threshold - 1:
        c = '{}'.format(c_min)
    else:
        c = '{} - {}'.format(c_min, threshold - 1)
    c_list.append(c)
    if c_max == threshold:
        c = '{}'.format(c_max)
    else:
        c = '{} - {}'.format(threshold, c_max)
    c_list.append(c)
    df[index] = c_list + ['TOPLAM']

    dft = dfm.loc[(dfm[c_old] < threshold)]
    a1 = len(dft.loc[dft[c_new] < threshold])
    a2 = len(dft.loc[dft[c_new] >= threshold])
    a_list = [a1, a2, a1 + a2]
    df[c_list[0]] = a_list

    dft = dfm.loc[(dfm[c_old] >= threshold)]
    a1 = len(dft.loc[dft[c_new] < threshold])
    a2 = len(dft.loc[dft[c_new] >= threshold])
    a_list = [a1, a2, a1 + a2]
    df[c_list[1]] = a_list

    df['TOPLAM'] = 0
    for c in c_list:
        df['TOPLAM'] += df[c]
        
    return df

## create_performance()

In [387]:
def create_performance(title, multi_trend, start_date, end_date, order_list):
    df = pd.DataFrame()
    df[title] = order_list
    suffix_list = ['Risk (Milyon TL)', 'Riske Göre %', 'Adet', 'Adede Göre %']

    for i in range(4):
        j = 3 - i
        dft = multi_trend[j].copy()
        x = dft.columns[0]
        c = suffix_list[i] + ' ' + start_date
        dft = dft.rename(columns={x: title, start_date: c})
        df = pd.merge(df, dft[[title, c]], how='left', on=title)

    for i in range(4):
        j = 3 - i
        dft = multi_trend[j].copy()
        x = dft.columns[0]
        c = suffix_list[i] + ' ' + end_date
        dft = dft.rename(columns={x: title, end_date: c})
        df = pd.merge(df, dft[[title, c]], how='left', on=title)

    reordered_indices = [title]

    for d in [start_date, end_date]:
        for s in suffix_list[:2]:
            c = s + ' ' + d
            reordered_indices.append(c)

    for d in [start_date, end_date]:
        for s in suffix_list[2:]:
            c = s + ' ' + d
            reordered_indices.append(c)

    df = df[reordered_indices]
    
    t1 = df['Risk (Milyon TL) ' + start_date].sum()
    t2 = df['Risk (Milyon TL) ' + end_date].sum()

    df['Banka Geneli Riske Göre Pay % ' + start_date] = df['Risk (Milyon TL) ' + start_date] / t1 * 100
    df['Banka Geneli Riske Göre Pay % ' + end_date] = df['Risk (Milyon TL) ' + end_date] / t2 * 100
    df[' '] = np.nan
    indices = [
        'Risk Değişimi (%)',
        'Riske Göre % Değişimi (%)',
        'Adet Değişimi (%)',
        'Adede Göre % Değişimi (%)',
        'Banka Geneli Riske Göre Pay Değişimi (%)'
    ]

    for i, x in enumerate(suffix_list + ['Banka Geneli Riske Göre Pay %']):
        c1 = x + ' ' + start_date
        c2 = x + ' ' + end_date

        df[indices[i]] = (df[c2] - df[c1]) / df[c1] * 100


    return df

## create_score_snapshot()

In [388]:
def create_score_snapshot(column, multi_trend_list, date, order_list=None):
    multi_trend_prefix_list = [
        'Finansal Güçlük',
        'Gecikme',
        'Yakın İzleme',
        'KRÖA',
        'Yapay Zeka Risk Sınıfı >= 4',
        'Entegre Derece Skor >= 8',
        'Karşılıksız Çek'
    ]

    multi_trend_suffix_list = [' (Adet %) ', ' (Risk %) ']

    count_threshold = 10
    risk_threshold = 1_000_000

    if order_list is None:
        order_list = unique_dict[column]
    ons = 'Ortalama Normalize Skor'
    puan = '0-100 Puan'

    df = pd.DataFrame()
    df[column] = order_list
    
    c = column + ' ' + date
    r = 'Müşteri No'
    x = 'Nakdi Reeskontlu Toplam ' + date
    i = sorted_date_list.index(date)

    dfm = df_wide[i][[r, c, x]]
    dft = dfm[[r, c]].groupby(c).count().reset_index()
    dft.columns = [column, 'Firma Sayısı']
    df = pd.merge(df, dft, on=column, how='left')
    dft = dfm[[x, c]].groupby(c).sum().reset_index()
    dft.columns = [column, 'Nakdi Risk']
    df = pd.merge(df, dft, on=column, how='left')

    df = df.loc[(df['Firma Sayısı'] >= count_threshold) & (df['Nakdi Risk'] >= risk_threshold)]

    for mtl, p in zip(multi_trend_list, multi_trend_prefix_list):
        for mt, s in zip([mtl[0], mtl[2]], multi_trend_suffix_list):
            c0 = mt.columns[0]
            dft = mt[[c0, date]].copy().rename(columns={c0: column})
            dft.columns = [column, p + s + d]
            df = pd.merge(df, dft, on=column, how='left')
        
    df = df.fillna(0)

    column_list = list(df.columns[1:])
    n_column_list = ['N-' + c for c in column_list]

    for c in column_list:
        df['N-' + c] = zscore(df[c])

    df[ons] = df[n_column_list].mean(axis = 1)

    n_min = df[ons].min()
    n_max = df[ons].max()

    df[puan] = np.round((df[ons] - n_min) / (n_max - n_min) * 100, 0)

    df[''] = np.nan
    df[' '] = np.nan

    df = df[[column, puan, ons, ''] + column_list + [' '] + n_column_list]
    df = df.sort_values(puan, ascending=False).reset_index(drop=True)

    return df

# Sunumlar

## Özet

In [389]:
df_segment_ozet = create_summary('Değerlendirmeye Esas Segment', order_list=segment_list)
# df_bolum_ozet = create_summary('İlgili Bölüm', order_list=bolum_list)
# df_yetki_ozet = create_summary('Yetki Kodu', 1)
# df_bolge_ozet = create_summary('Bölge Adı', 1)
df_sektor_ozet = create_summary('Tahsis Sektör Adı')
# df_nace_ozet = create_summary('NACE Harf', order_list=nace_list)
# df_sinif_ozet = create_summary('Müşteri Sınıfı', musteri_sinifi=False)
# df_yz_ozet = create_summary('Yapay Zeka Risk Sınıfı', 1, yapay_zeka=False)
# df_eds_ozet = create_summary('Entegre Derece Skor', 1, entegre_derece=False)
# df_sube_ozet = convert_sube_kodu_adi(create_summary('Şube Kodu'))

## Trendler

In [390]:
df_segment_risk_trend = create_sum_trend('Değerlendirmeye Esas Segment', 'Nakdi Reeskontlu Toplam', segment_list, factor=1_000_000)
# df_bolum_risk_trend = create_sum_trend('İlgili Bölüm', 'Nakdi Reeskontlu Toplam', bolum_list, factor=1_000_000)
# df_yetki_risk_trend = create_sum_trend('Yetki Kodu', 'Nakdi Reeskontlu Toplam', factor=1_000_000)
# df_bolge_risk_trend = create_sum_trend('Bölge Adı', 'Nakdi Reeskontlu Toplam', factor=1_000_000)
df_sektor_risk_trend = create_sum_trend('Tahsis Sektör Adı', 'Nakdi Reeskontlu Toplam', sektor_list, factor=1_000_000)
# df_nace_risk_trend = create_sum_trend('NACE Harf', 'Nakdi Reeskontlu Toplam', nace_list, factor=1_000_000)
# df_sinif_risk_trend = create_sum_trend('Müşteri Sınıfı', 'Nakdi Reeskontlu Toplam', factor=1_000_000)
# df_yz_risk_trend = create_sum_trend('Yapay Zeka Risk Sınıfı', 'Nakdi Reeskontlu Toplam', factor=1_000_000)
# df_eds_risk_trend = create_sum_trend('Entegre Derece Skor', 'Nakdi Reeskontlu Toplam', factor=1_000_000)
# df_sube_risk_trend = convert_sube_kodu_adi(create_sum_trend('Şube Kodu', 'Nakdi Reeskontlu Toplam', factor=1_000_000))

In [391]:
df_segment_yz_trend = create_mean_trend('Değerlendirmeye Esas Segment', 'Yapay Zeka Risk Sınıfı', segment_list)
df_segment_eds_trend = create_mean_trend('Değerlendirmeye Esas Segment', 'Entegre Derece Skor', segment_list)
df_segment_sinif_trend = create_mean_trend('Değerlendirmeye Esas Segment', 'Müşteri Sınıfı', segment_list)

df_segment_yz_trend_weighted = create_mean_trend('Değerlendirmeye Esas Segment', 'Yapay Zeka Risk Sınıfı', segment_list, 'Nakdi Reeskontlu Toplam')
df_segment_eds_trend_weighted = create_mean_trend('Değerlendirmeye Esas Segment', 'Entegre Derece Skor', segment_list, 'Nakdi Reeskontlu Toplam')
df_segment_sinif_trend_weighted = create_mean_trend('Değerlendirmeye Esas Segment', 'Müşteri Sınıfı', segment_list, 'Nakdi Reeskontlu Toplam')

# df_bolum_yz_trend = create_mean_trend('İlgili Bölüm', 'Yapay Zeka Risk Sınıfı', bolum_list)
# df_bolum_eds_trend = create_mean_trend('İlgili Bölüm', 'Entegre Derece Skor', bolum_list)
# df_bolum_sinif_trend = create_mean_trend('İlgili Bölüm', 'Müşteri Sınıfı', bolum_list)

# df_yetki_yz_trend = create_mean_trend('Yetki Kodu', 'Yapay Zeka Risk Sınıfı')
# df_yetki_eds_trend = create_mean_trend('Yetki Kodu', 'Entegre Derece Skor')
# df_yetki_sinif_trend = create_mean_trend('Yetki Kodu', 'Müşteri Sınıfı')

# df_bolge_yz_trend = create_mean_trend('Bölge Adı', 'Yapay Zeka Risk Sınıfı')
# df_bolge_eds_trend = create_mean_trend('Bölge Adı', 'Entegre Derece Skor')
# df_bolge_sinif_trend = create_mean_trend('Bölge Adı', 'Müşteri Sınıfı')

df_sektor_yz_trend = create_mean_trend('Tahsis Sektör Adı', 'Yapay Zeka Risk Sınıfı', sektor_list)
df_sektor_eds_trend = create_mean_trend('Tahsis Sektör Adı', 'Entegre Derece Skor', sektor_list)
df_sektor_sinif_trend = create_mean_trend('Tahsis Sektör Adı', 'Müşteri Sınıfı', sektor_list)

df_sektor_yz_trend_weighted = create_mean_trend('Tahsis Sektör Adı', 'Yapay Zeka Risk Sınıfı', sektor_list, 'Nakdi Reeskontlu Toplam')
df_sektor_eds_trend_weighted = create_mean_trend('Tahsis Sektör Adı', 'Entegre Derece Skor', sektor_list, 'Nakdi Reeskontlu Toplam')
df_sektor_sinif_trend_weighted = create_mean_trend('Tahsis Sektör Adı', 'Müşteri Sınıfı', sektor_list, 'Nakdi Reeskontlu Toplam')

# df_nace_yz_trend = create_mean_trend('NACE Harf', 'Yapay Zeka Risk Sınıfı', nace_list)
# df_nace_eds_trend = create_mean_trend('NACE Harf', 'Entegre Derece Skor', nace_list)
# df_nace_sinif_trend = create_mean_trend('NACE Harf', 'Müşteri Sınıfı', nace_list)

# df_sinif_yz_trend = create_mean_trend('Müşteri Sınıfı', 'Yapay Zeka Risk Sınıfı')
# df_sinif_eds_trend = create_mean_trend('Müşteri Sınıfı', 'Entegre Derece Skor')

# df_yz_eds_trend = create_mean_trend('Yapay Zeka Risk Sınıfı', 'Entegre Derece Skor')
# df_yz_sinif_trend = create_mean_trend('Yapay Zeka Risk Sınıfı', 'Müşteri Sınıfı')

# df_eds_yz_trend = create_mean_trend('Entegre Derece Skor', 'Yapay Zeka Risk Sınıfı')
# df_eds_sinif_trend = create_mean_trend('Entegre Derece Skor', 'Müşteri Sınıfı')

# df_sube_yz_trend = convert_sube_kodu_adi(create_mean_trend('Şube Kodu', 'Yapay Zeka Risk Sınıfı'))
# df_sube_eds_trend = convert_sube_kodu_adi(create_mean_trend('Şube Kodu', 'Entegre Derece Skor'))
# df_sube_sinif_trend = convert_sube_kodu_adi(create_mean_trend('Şube Kodu', 'Müşteri Sınıfı'))

## Multi Trendler

In [392]:
segment_yz_multi_trend = create_multi_trend('Değerlendirmeye Esas Segment', 'Yapay Zeka Risk Sınıfı', 4, segment_list)
df_segment_yz_multi_trend = combine_multi_trend(segment_yz_multi_trend)

segment_sinif_multi_trend = create_multi_trend('Değerlendirmeye Esas Segment', 'Müşteri Sınıfı', 2, segment_list)
df_segment_sinif_multi_trend = combine_multi_trend(segment_sinif_multi_trend)

segment_eds_multi_trend = create_multi_trend('Değerlendirmeye Esas Segment', 'Entegre Derece Skor', 8, segment_list)
df_segment_eds_multi_trend = combine_multi_trend(segment_eds_multi_trend)

segment_fg_multi_trend = create_multi_trend('Değerlendirmeye Esas Segment', 'Finansal Güçlük', 1, segment_list)
df_segment_fg_multi_trend = combine_multi_trend(segment_fg_multi_trend)

segment_kroa_multi_trend = create_multi_trend('Değerlendirmeye Esas Segment', 'KRÖA', 1, segment_list)
df_segment_kroa_multi_trend = combine_multi_trend(segment_kroa_multi_trend)

segment_gecikme_multi_trend = create_multi_trend('Değerlendirmeye Esas Segment', 'Gecikme', 1, segment_list)
df_segment_gecikme_multi_trend = combine_multi_trend(segment_gecikme_multi_trend)

segment_cek_multi_trend = create_multi_trend('Değerlendirmeye Esas Segment', 'Durum Kodu', [4], segment_list, factor=1, fill_null=0, custom_df=cek_df_list, custom_nn_list=[2, 3, 4, 5], custom_risk='Çek Tutarı')
df_segment_cek_multi_trend = combine_multi_trend(segment_cek_multi_trend)

In [393]:
# bolum_yz_multi_trend = create_multi_trend('İlgili Bölüm', 'Yapay Zeka Risk Sınıfı', 4, bolum_list)
# df_bolum_yz_multi_trend = combine_multi_trend(bolum_yz_multi_trend)

# bolum_sinif_multi_trend = create_multi_trend('İlgili Bölüm', 'Müşteri Sınıfı', 2, bolum_list)
# df_bolum_sinif_multi_trend = combine_multi_trend(bolum_sinif_multi_trend)

# bolum_eds_multi_trend = create_multi_trend('İlgili Bölüm', 'Entegre Derece Skor', 8, bolum_list)
# df_bolum_eds_multi_trend = combine_multi_trend(bolum_eds_multi_trend)

# bolum_fg_multi_trend = create_multi_trend('İlgili Bölüm', 'Finansal Güçlük', 1, bolum_list)
# df_bolum_fg_multi_trend = combine_multi_trend(bolum_fg_multi_trend)

# bolum_kroa_multi_trend = create_multi_trend('İlgili Bölüm', 'KRÖA', 1, bolum_list)
# df_bolum_kroa_multi_trend = combine_multi_trend(bolum_kroa_multi_trend)

# bolum_gecikme_multi_trend = create_multi_trend('İlgili Bölüm', 'Gecikme', 1, bolum_list)
# df_bolum_gecikme_multi_trend = combine_multi_trend(bolum_gecikme_multi_trend)

# bolum_cek_multi_trend = create_multi_trend('İlgili Bölüm', 'Durum Kodu', [4], bolum_list, factor=1, fill_null=0, custom_df=cek_df_list, custom_nn_list=[2, 3, 4, 5], custom_risk='Çek Tutarı')
# df_bolum_cek_multi_trend = combine_multi_trend(bolum_cek_multi_trend)

In [394]:
# yetki_yz_multi_trend = create_multi_trend('Yetki Kodu', 'Yapay Zeka Risk Sınıfı', 4)
# df_yetki_yz_multi_trend = combine_multi_trend(yetki_yz_multi_trend)

# yetki_sinif_multi_trend = create_multi_trend('Yetki Kodu', 'Müşteri Sınıfı', 2)
# df_yetki_sinif_multi_trend = combine_multi_trend(yetki_sinif_multi_trend)

# yetki_eds_multi_trend = create_multi_trend('Yetki Kodu', 'Entegre Derece Skor', 8)
# df_yetki_eds_multi_trend = combine_multi_trend(yetki_eds_multi_trend)

# yetki_fg_multi_trend = create_multi_trend('Yetki Kodu', 'Finansal Güçlük', 1)
# df_yetki_fg_multi_trend = combine_multi_trend(yetki_fg_multi_trend)

# yetki_kroa_multi_trend = create_multi_trend('Yetki Kodu', 'KRÖA', 1)
# df_yetki_kroa_multi_trend = combine_multi_trend(yetki_kroa_multi_trend)

# yetki_gecikme_multi_trend = create_multi_trend('Yetki Kodu', 'Gecikme', 1)
# df_yetki_gecikme_multi_trend = combine_multi_trend(yetki_gecikme_multi_trend)

# yetki_cek_multi_trend = create_multi_trend('Yetki Kodu', 'Durum Kodu', [4], factor=1, fill_null=0, custom_df=cek_df_list, custom_nn_list=[2, 3, 4, 5], custom_risk='Çek Tutarı')
# df_yetki_cek_multi_trend = combine_multi_trend(yetki_cek_multi_trend)

In [395]:
# bolge_yz_multi_trend = create_multi_trend('Bölge Adı', 'Yapay Zeka Risk Sınıfı', 4)
# df_bolge_yz_multi_trend = combine_multi_trend(bolge_yz_multi_trend)

# bolge_sinif_multi_trend = create_multi_trend('Bölge Adı', 'Müşteri Sınıfı', 2)
# df_bolge_sinif_multi_trend = combine_multi_trend(bolge_sinif_multi_trend)

# bolge_eds_multi_trend = create_multi_trend('Bölge Adı', 'Entegre Derece Skor', 8)
# df_bolge_eds_multi_trend = combine_multi_trend(bolge_eds_multi_trend)

# bolge_fg_multi_trend = create_multi_trend('Bölge Adı', 'Finansal Güçlük', 1)
# df_bolge_fg_multi_trend = combine_multi_trend(bolge_fg_multi_trend)

# bolge_kroa_multi_trend = create_multi_trend('Bölge Adı', 'KRÖA', 1)
# df_bolge_kroa_multi_trend = combine_multi_trend(bolge_kroa_multi_trend)

# bolge_gecikme_multi_trend = create_multi_trend('Bölge Adı', 'Gecikme', 1)
# df_bolge_gecikme_multi_trend = combine_multi_trend(bolge_gecikme_multi_trend)

# bolge_cek_multi_trend = create_multi_trend('Bölge Adı', 'Durum Kodu', [4], bolge_list, factor=1, fill_null=0, custom_df=cek_df_list, custom_nn_list=[2, 3, 4, 5], custom_risk='Çek Tutarı')
# df_bolge_cek_multi_trend = combine_multi_trend(bolge_cek_multi_trend)

In [396]:
sektor_yz_multi_trend = create_multi_trend('Tahsis Sektör Adı', 'Yapay Zeka Risk Sınıfı', 4, sektor_list)
df_sektor_yz_multi_trend = combine_multi_trend(sektor_yz_multi_trend)

sektor_sinif_multi_trend = create_multi_trend('Tahsis Sektör Adı', 'Müşteri Sınıfı', 2, sektor_list)
df_sektor_sinif_multi_trend = combine_multi_trend(sektor_sinif_multi_trend)

sektor_eds_multi_trend = create_multi_trend('Tahsis Sektör Adı', 'Entegre Derece Skor', 8, sektor_list)
df_sektor_eds_multi_trend = combine_multi_trend(sektor_eds_multi_trend)

sektor_fg_multi_trend = create_multi_trend('Tahsis Sektör Adı', 'Finansal Güçlük', 1, sektor_list)
df_sektor_fg_multi_trend = combine_multi_trend(sektor_fg_multi_trend)

sektor_kroa_multi_trend = create_multi_trend('Tahsis Sektör Adı', 'KRÖA', 1, sektor_list)
df_sektor_kroa_multi_trend = combine_multi_trend(sektor_kroa_multi_trend)

sektor_gecikme_multi_trend = create_multi_trend('Tahsis Sektör Adı', 'Gecikme', 1, sektor_list)
df_sektor_gecikme_multi_trend = combine_multi_trend(sektor_gecikme_multi_trend)

sektor_cek_multi_trend = create_multi_trend('Tahsis Sektör Adı', 'Durum Kodu', [4], sektor_list, factor=1, fill_null=0, custom_df=cek_df_list, custom_nn_list=[2, 3, 4, 5], custom_risk='Çek Tutarı')
df_sektor_cek_multi_trend = combine_multi_trend(sektor_cek_multi_trend)

In [397]:
# nace_yz_multi_trend = create_multi_trend('NACE Harf', 'Yapay Zeka Risk Sınıfı', 4, nace_list)
# df_nace_yz_multi_trend = combine_multi_trend(nace_yz_multi_trend)

# nace_sinif_multi_trend = create_multi_trend('NACE Harf', 'Müşteri Sınıfı', 2, nace_list)
# df_nace_sinif_multi_trend = combine_multi_trend(nace_sinif_multi_trend)

# nace_eds_multi_trend = create_multi_trend('NACE Harf', 'Entegre Derece Skor', 8, nace_list)
# df_nace_eds_multi_trend = combine_multi_trend(nace_eds_multi_trend)

# nace_fg_multi_trend = create_multi_trend('NACE Harf', 'Finansal Güçlük', 1, nace_list)
# df_nace_fg_multi_trend = combine_multi_trend(nace_fg_multi_trend)

# nace_kroa_multi_trend = create_multi_trend('NACE Harf', 'KRÖA', 1, nace_list)
# df_nace_kroa_multi_trend = combine_multi_trend(nace_kroa_multi_trend)

# nace_gecikme_multi_trend = create_multi_trend('NACE Harf', 'Gecikme', 1, nace_list)
# df_nace_gecikme_multi_trend = combine_multi_trend(nace_gecikme_multi_trend)

# nace_cek_multi_trend = create_multi_trend('NACE Harf', 'Durum Kodu', [4], nace_list, factor=1, fill_null=0, custom_df=cek_df_list, custom_nn_list=[2, 3, 4, 5], custom_risk='Çek Tutarı')
# df_nace_cek_multi_trend = combine_multi_trend(nace_cek_multi_trend)

In [398]:
# sube_yz_multi_trend_raw = create_multi_trend('Şube Kodu', 'Yapay Zeka Risk Sınıfı', 4, sube_list)
# sube_yz_multi_trend = convert_sube_kodu_adi(sube_yz_multi_trend_raw)
# df_sube_yz_multi_trend = combine_multi_trend(sube_yz_multi_trend)

# sube_sinif_multi_trend_raw = create_multi_trend('Şube Kodu', 'Müşteri Sınıfı', 2, sube_list)
# sube_sinif_multi_trend = convert_sube_kodu_adi(sube_sinif_multi_trend_raw)
# df_sube_sinif_multi_trend = combine_multi_trend(sube_sinif_multi_trend)

# sube_eds_multi_trend_raw = create_multi_trend('Şube Kodu', 'Entegre Derece Skor', 8, sube_list)
# sube_eds_multi_trend = convert_sube_kodu_adi(sube_eds_multi_trend_raw)
# df_sube_eds_multi_trend = combine_multi_trend(sube_eds_multi_trend)

# sube_fg_multi_trend_raw = create_multi_trend('Şube Kodu', 'Finansal Güçlük', 1, sube_list)
# sube_fg_multi_trend = convert_sube_kodu_adi(sube_fg_multi_trend_raw)
# df_sube_fg_multi_trend = combine_multi_trend(sube_fg_multi_trend)

# sube_kroa_multi_trend_raw = create_multi_trend('Şube Kodu', 'KRÖA', 1, sube_list)
# sube_kroa_multi_trend = convert_sube_kodu_adi(sube_kroa_multi_trend_raw)
# df_sube_kroa_multi_trend = combine_multi_trend(sube_kroa_multi_trend)

# sube_gecikme_multi_trend_raw = create_multi_trend('Şube Kodu', 'Gecikme', 1, sube_list)
# sube_gecikme_multi_trend = convert_sube_kodu_adi(sube_gecikme_multi_trend_raw)
# df_sube_gecikme_multi_trend = combine_multi_trend(sube_gecikme_multi_trend)

# sube_cek_multi_trend_raw = create_multi_trend('Şube Kodu', 'Durum Kodu', [4], sube_list, factor=1, fill_null=0, custom_df=cek_df_list, custom_nn_list=[2, 3, 4, 5], custom_risk='Çek Tutarı')
# sube_cek_multi_trend = convert_sube_kodu_adi(sube_cek_multi_trend_raw)
# df_sube_cek_multi_trend = combine_multi_trend(sube_cek_multi_trend)

In [399]:
# il_yz_multi_trend = create_multi_trend('Şube İl', 'Yapay Zeka Risk Sınıfı', 4, il_list)
# df_il_yz_multi_trend = combine_multi_trend(il_yz_multi_trend)

# il_sinif_multi_trend = create_multi_trend('Şube İl', 'Müşteri Sınıfı', 2, il_list)
# df_il_sinif_multi_trend = combine_multi_trend(il_sinif_multi_trend)

# il_eds_multi_trend = create_multi_trend('Şube İl', 'Entegre Derece Skor', 8, il_list)
# df_il_eds_multi_trend = combine_multi_trend(il_eds_multi_trend)

# il_fg_multi_trend = create_multi_trend('Şube İl', 'Finansal Güçlük', 1, il_list)
# df_il_fg_multi_trend = combine_multi_trend(il_fg_multi_trend)

# il_kroa_multi_trend = create_multi_trend('Şube İl', 'KRÖA', 1, il_list)
# df_il_kroa_multi_trend = combine_multi_trend(il_kroa_multi_trend)

# il_gecikme_multi_trend = create_multi_trend('Şube İl', 'Gecikme', 1, il_list)
# df_il_gecikme_multi_trend = combine_multi_trend(il_gecikme_multi_trend)

# il_cek_multi_trend = create_multi_trend('Şube İl', 'Durum Kodu', [4], il_list, factor=1, fill_null=0, custom_df=cek_df_list, custom_nn_list=[2, 3, 4, 5], custom_risk='Çek Tutarı')
# df_il_cek_multi_trend = combine_multi_trend(il_cek_multi_trend)

## Matrisler

In [400]:
# df_yz_matrix, df_yz_3_ten_4, df_yz_4_ten_3 = create_matrix('Yapay Zeka Risk Sınıfı', 4)
# df_sinif_matrix, df_sinif_1_den_2, df_sinif_2_den_1 = create_matrix('Müşteri Sınıfı', 2)
# df_eds_matrix, df_eds_7_den_8, df_eds_8_den_7 = create_matrix('Entegre Derece Skor', 8)

In [401]:
# df_yz_matrix_condensed = create_matrix_condensed('Yapay Zeka Risk Sınıfı', 4)
# df_sinif_matrix_condensed = create_matrix_condensed('Müşteri Sınıfı', 2)
# df_eds_matrix_condensed = create_matrix_condensed('Entegre Derece Skor', 8)

## Skorlama

In [402]:
segment_multi_trend_list = [segment_fg_multi_trend, segment_gecikme_multi_trend, segment_sinif_multi_trend, segment_kroa_multi_trend, segment_yz_multi_trend, segment_eds_multi_trend, segment_cek_multi_trend]
# bolum_multi_trend_list = [bolum_fg_multi_trend, bolum_gecikme_multi_trend, bolum_sinif_multi_trend, bolum_kroa_multi_trend, bolum_yz_multi_trend, bolum_eds_multi_trend, bolum_cek_multi_trend]
# yetki_multi_trend_list = [yetki_fg_multi_trend, yetki_gecikme_multi_trend, yetki_sinif_multi_trend, yetki_kroa_multi_trend, yetki_yz_multi_trend, yetki_eds_multi_trend, yetki_cek_multi_trend]
# bolge_multi_trend_list = [bolge_fg_multi_trend, bolge_gecikme_multi_trend, bolge_sinif_multi_trend, bolge_kroa_multi_trend, bolge_yz_multi_trend, bolge_eds_multi_trend, bolge_cek_multi_trend]
sektor_multi_trend_list = [sektor_fg_multi_trend, sektor_gecikme_multi_trend, sektor_sinif_multi_trend, sektor_kroa_multi_trend, sektor_yz_multi_trend, sektor_eds_multi_trend, sektor_cek_multi_trend]
# nace_multi_trend_list = [nace_fg_multi_trend, nace_gecikme_multi_trend, nace_sinif_multi_trend, nace_kroa_multi_trend, nace_yz_multi_trend, nace_eds_multi_trend, nace_cek_multi_trend]
# sube_multi_trend_list = [sube_fg_multi_trend_raw, sube_gecikme_multi_trend_raw, sube_sinif_multi_trend_raw, sube_kroa_multi_trend_raw, sube_yz_multi_trend_raw, sube_eds_multi_trend_raw, sube_cek_multi_trend_raw]
# il_multi_trend_list = [il_fg_multi_trend, il_gecikme_multi_trend, il_sinif_multi_trend, il_kroa_multi_trend, il_yz_multi_trend, il_eds_multi_trend, il_cek_multi_trend]

In [403]:
df_segment_skor = create_score_snapshot('Değerlendirmeye Esas Segment', segment_multi_trend_list, last_date, segment_list)
# df_bolum_skor = create_score_snapshot('İlgili Bölüm', bolum_multi_trend_list, last_date, bolum_list)
# df_yetki_skor = create_score_snapshot('Yetki Kodu', yetki_multi_trend_list, last_date)
# df_bolge_skor = create_score_snapshot('Bölge Adı', bolge_multi_trend_list, last_date, bolge_list)
df_sektor_skor = create_score_snapshot('Tahsis Sektör Adı', sektor_multi_trend_list, last_date, filtered_sektor_list)
# df_nace_skor = create_score_snapshot('NACE Harf', nace_multi_trend_list, last_date, nace_list)
# df_sube_skor = convert_sube_kodu_adi(create_score_snapshot('Şube Kodu', sube_multi_trend_list, last_date, sube_list))
# df_il_skor = create_score_snapshot('Şube İl', il_multi_trend_list, last_date, il_list)

## Korelasyon Matrisi

In [404]:
# show_one_half = False
# save_figure = True

# x = [p + s + d for s in multi_trend_suffix_list for p in multi_trend_prefix_list]
# y = [x[:-8] for x in x]
# dfcmt = df_sube_skor[x].copy()
# dfcmt.columns = y
# correlation_matrix = dfcmt.corr()
# upper_half = np.triu(correlation_matrix) if show_one_half else None
# # fig = plt.figure(figsize=(df.shape[1], df.shape[1]), dpi=480)
# # sns.heatmap(correlation_matrix, annot=True, fmt='.2f', square=True, mask=upper_half)

# fig = plt.figure(figsize=(15, 15), dpi = 300)
# sns.heatmap(correlation_matrix, annot=True, fmt='.2f', square=True, mask=upper_half)

# if save_figure:
#     plt.savefig(output_folder_path + 'correlation_matrix.png', bbox_inches = 'tight')

# plt.show()


# Sunum Formatlama

## Color Scale

In [405]:
def add_color_scale(i, row_based=True, multi=False, reverse=True, vertical=False, mid_value=None, percentile=None, gray_empty=True):
    green, yellow, red = '#63BE7B', '#FFEB84', '#F8696B'
    if reverse:
        color_scale_format = {
            'type': '3_color_scale',
            'min_color': green,
            'mid_color': yellow,
            'max_color': red
        }
    else:
        color_scale_format = {
            'type': '3_color_scale',
            'min_color': red,
            'mid_color': yellow,
            'max_color': green
        }

    if mid_value is not None:
        color_scale_format = {**color_scale_format, **{
            'mid_type': 'num',
            'mid_value': mid_value
        }}
        
    if type(percentile) is tuple:
        color_scale_format = {**color_scale_format, **{
            'min_type': 'percentile',
            'max_type': 'percentile',
            'min_value': percentile[0],
            'max_value': percentile[1]
        }}

    if multi:
        df = multi_df_list[multi_index_list.index(i)][0]
    else:
        df = df_list[i]
    h = df.shape[0]
    w = df.shape[1]      
    n_space = h_space if len(df) < h_limit else l_space

    worksheet = writer.sheets[str(i)]
    gray_format = workbook.add_format({'bg_color': 'E5E1E6'})
    blank_format = {'type':'blanks', 'format': gray_format}

    if multi:
        c_list = [(letters[1], letters[w - 1]), (letters[w + w_space + 1], letters[w * 2 + w_space - 1])]
        r_list = [2, h + n_space + 3]
    elif not vertical:
        c_list = [(letters[1], letters[w - 1])]
        r_list = [2]
    else:
        c_list = [1]
        r_list = [(2, h + 1)]

    if row_based:
        if not vertical:
            for c1, c2 in c_list:
                for r in r_list:
                    for i in range(h):
                        worksheet.conditional_format('{}{}:{}{}'.format(c1, r + i, c2, r + i), color_scale_format)
                    if gray_empty:
                        worksheet.conditional_format('{}{}:{}{}'.format(c1, r, c2, r + h - 1), blank_format)
        else:
            for r1, r2 in r_list:
                for c in c_list:
                    for i in range(w - 1):
                        worksheet.conditional_format('{}{}:{}{}'.format(letters[c + i], r1, letters[c + i], r2), color_scale_format)
                    if gray_empty:
                        worksheet.conditional_format('{}{}:{}{}'.format(letters[c], r1, letters[c + w - 2], r2), blank_format)
        
    else:
        for c1, c2 in c_list:
            for r in r_list:
                worksheet.conditional_format('{}{}:{}{}'.format(c1, r, c2, r + h - 1), color_scale_format)
                if gray_empty:
                    worksheet.conditional_format('{}{}:{}{}'.format(c1, r, c2, r + h - 1), blank_format)


## Line Chart

In [406]:
def add_line_chart(df, sheet_name, y_label, title, chart_place=None, chart_row = 'A', chart_start = 'B', chart_end = 'M', chart_offset = 0, int_format = False, log_scale = False, data_labels=False):
    length = len(df)
    if chart_place is None:
        chart_place = 'B{}'.format(length + 3)

    chart = workbook.add_chart({'type': 'line'})
    for i in range(len(df)):
        chart.add_series({
        'categories': '={}!${}${}:${}${}'.format(sheet_name, chart_start, 1 + chart_offset, chart_end, 1 + chart_offset),
        'values': '={}!${}${}:${}${}'.format(sheet_name, chart_start, i + 2 + chart_offset, chart_end, i + 2 + chart_offset),
        'name': '={}!${}${}'.format(sheet_name, chart_row, i + 2 + chart_offset),
        'data_labels': {'value': data_labels}
        })



    x_format = {'name': 'Aylar', 'name_font': {'size': 11}, 'num_font': {'rotation': -45}}
    chart.set_x_axis(x_format)

    y_format = {'name': y_label, 'name_font': {'size': 11}, 'major_gridlines': {'visible': True, 'line':{'color': 'gray', 'transparency': 80}}}
    if int_format:
        y_format = {**y_format, **{'num_format': '#,##0'}}


    l_min = df.iloc[:,1:].min().min()
    l_max = df.iloc[:,1:].max().max()

    if log_scale:
        zero = False
        e_min = math.log(l_min, 10) if l_min > 0 else 0
        l_min = 10 ** np.floor(e_min)
        l_max = 10 ** np.ceil(math.log(l_max, 10))
        y_format = {**y_format, **{'log_base': 10}}
    else:
        l_range = l_max - l_min
        l_min -= l_range * 0.1
        l_max += l_range * 0.1
        l_min = np.floor(l_min * 10) / 10
        l_max = np.ceil(l_max * 10) / 10

        if l_min < 0:
            l_min = 0 

    y_format = {**y_format, **{'min': l_min, 'max': l_max}}
    chart.set_y_axis(y_format)
    
    chart.set_title({'name': title, 'name_font': {'size': 15}})

    chart.set_size({'x_scale': 1.88, 'y_scale': 1.2})
    worksheet = writer.sheets[sheet_name]
    worksheet.insert_chart(chart_place, chart)

In [407]:
def add_multi_line_chart(multi_trend, sheet_name, y_label, title):
    h = multi_trend[0].shape[0]
    w = multi_trend[0].shape[1]      

    r1 = 'A'
    r2 = letters[w + w_space]

    s1 = 'B'
    s2 = letters[w + w_space + 1]

    e1 = letters[w - 1]
    e2 = letters[2 * w + w_space - 1]

    o1 = 0
    o2 = h + h_space + 1


    chart_places, chart_rows, chart_starts, chart_ends, chart_offsets, int_formats, log_scales = [], [], [], [], [], [], []

    chart_places.append(f'{s1}{h + 3}')
    chart_places.append(f'{s2}{h + 3}')
    chart_places.append(f'{s1}{2 * h + h_space + 4}')
    chart_places.append(f'{s2}{2 * h + h_space + 4}')

    
    chart_offsets.append(o1)
    chart_offsets.append(o1)
    chart_offsets.append(o2)
    chart_offsets.append(o2)

    for i in range(2):
        chart_rows.append(r1)
        chart_rows.append(r2)
        
        chart_starts.append(s1)
        chart_starts.append(s2)
        
        chart_ends.append(e1)
        chart_ends.append(e2)

        int_formats.append(False)
        int_formats.append(True)

        log_scales.append(False)
        log_scales.append(True)


    for i, df in enumerate(multi_trend):
        add_line_chart(df, sheet_name, y_label[i], title[i], chart_places[i], chart_rows[i], chart_starts[i], chart_ends[i], chart_offsets[i], int_formats[i], log_scales[i])


## Pie Chart

In [408]:
def add_pie_chart(df, sheet_name, title, chart_place, value_columns=['B'], num_format='#,##0,, "mio" ₺', category_column = 'A'):
    chart = workbook.add_chart({'type': 'pie'})
    show_labels = True if len(df) <= 6 else False
    
    end = len(df)
    if end < 2: end = 2

    for value_column in value_columns:
        chart.add_series({
        'categories': '={}!${}${}:${}${}'.format(sheet_name, category_column, 2, category_column, end),
        'values': '={}!${}${}:${}${}'.format(sheet_name, value_column, 2, value_column, end),
        'name': '={}!${}${}'.format(sheet_name, value_column, 1),
        'data_labels': {
            'category': show_labels,
            'value': show_labels,
            'num_format': num_format,
            'separator': '\n',
            'position': 'outside_end',
            'border': {'color': 'gray'},
            'fill': {'color': 'white'},
            'font': {'size': 9}
        }
        })

    chart.set_legend({'position': 'bottom'})
    chart.set_size({'x_scale': 0.91, 'y_scale': 1.24})
    chart.set_title({'name': title, 'name_font': {'bold': False, 'size': 15}})


    worksheet = writer.sheets[sheet_name]
    worksheet.insert_chart(chart_place, chart)

## Bar Chart

In [409]:
def add_bar_chart(df, sheet_name, y_label, title, chart_place, value_columns=['J', 'K'], category_column='A', label_columns=None, include_sum=False, stacked=False):
    data_label_style = {
            'value': True,
            'num_format': '0.00" %"',
            'font': {'size': 9}
        }
    
    if stacked:
        chart = workbook.add_chart({'type': 'bar', 'subtype': 'percent_stacked'})
        data_label_style = {**data_label_style, **{
                            'border': {'color': 'gray'},
                            'fill': {'color': 'white'}}
                            }
    else:
        chart = workbook.add_chart({'type': 'column'})

    if len(df) >= 7: data_label_style = False

    end = len(df) + 1 if include_sum else len(df)
    if end < 2: end = 2
    
    if label_columns is None:
        label_columns = value_columns

    for value_column, label_column in zip(value_columns, label_columns):
        chart.add_series({
        'categories': '={}!${}${}:${}${}'.format(sheet_name, category_column, 2, category_column, end),
        'values': '={}!${}${}:${}${}'.format(sheet_name, value_column, 2, value_column, end),
        'name': '={}!${}${}'.format(sheet_name, label_column, 1),
        'data_labels': data_label_style
        })

    chart.set_legend({'position': 'bottom'})
    chart.set_size({'x_scale': 1.09, 'y_scale': 1.24})
    chart.set_title({'name': title, 'name_font': {'bold': False, 'size': 15}})
    chart.set_y_axis({'name': y_label, 'name_font': {'bold': False}, 'num_format': '0', 'major_gridlines': {'visible': True, 'line':{'color': 'gray', 'transparency': 80}}})


    worksheet = writer.sheets[sheet_name]
    worksheet.insert_chart(chart_place, chart)

## Combined Chart

In [410]:
def add_combined_chart(df, sheet_name, y_label, y_label_2, title, chart_place, value_columns = ['M', 'N'], line_columns = ['L'], category_column = 'A'):
    chart = workbook.add_chart({'type': 'column'})
    
    end = len(df)
    if end < 2: end = 2
    
    for value_column in value_columns:
        chart.add_series({
        'categories': '={}!${}${}:${}${}'.format(sheet_name, category_column, 2, category_column, end),
        'values': '={}!${}${}:${}${}'.format(sheet_name, value_column, 2, value_column, end),
        'name': '={}!${}${}'.format(sheet_name, value_column, 1),
        })

    line_chart = workbook.add_chart({'type': 'line'})
    for value_column in line_columns:
        line_chart.add_series({
        'categories': '={}!${}${}:${}${}'.format(sheet_name, category_column, 2, category_column, end),
        'values': '={}!${}${}:${}${}'.format(sheet_name, value_column, 2, value_column, end),
        'name': '={}!${}${}'.format(sheet_name, value_column, 1),
        'y2_axis': True,
        'marker': {
        'type': 'square',
        'size': 9
        }})
        
    line_chart.set_y2_axis({'name': y_label_2, 'name_font': {'bold': False}, 'num_format': '0.0'})

    chart.combine(line_chart)
    
    chart.set_legend({'position': 'bottom'})
    chart.set_size({'x_scale': 1.09, 'y_scale': 1.24})
    chart.set_title({'name': title, 'name_font': {'bold': False, 'size': 15}})
    chart.set_y_axis({'name': y_label, 'name_font': {'bold': False}, 'num_format': '0', 'major_gridlines': {'visible': True, 'line':{'color': 'gray', 'transparency': 80}}})


    worksheet = writer.sheets[sheet_name]
    worksheet.insert_chart(chart_place, chart)

## Summary Charts

In [411]:
def add_summary_charts(i):
    df = df_list[i]

    yz, eds, sinif, fg, kroa, gecikme = True, True, True, True, True, True
    if 'Yapay Zeka Risk Sınıfı' in list(df.columns)[0]:
        yz = False
    elif 'Entegre Derece Skor' in list(df.columns)[0]:
        eds = False
    elif 'Müşteri Sınıfı' in list(df.columns)[0]:
        sinif = False
        
    r1 = len(df) + 3
    r2 = r1 + 18
    r3 = r2 + 18

    c1 = 'A'
    c2 = 'E'
    c3 = 'J'
    c4 = 'P'
    
    sheet_name = str(i)
    if len(df) >= 2:
        add_pie_chart(df, sheet_name, 'Firma Sayısı', f'{c1}{r1}', ['B'], '#,##0')
        add_pie_chart(df, sheet_name, 'Nakdi Risk', f'{c1}{r2}', ['D'])
        add_pie_chart(df, sheet_name, 'Gayrinakdi Risk', f'{c2}{r2}', ['F'])
        add_pie_chart(df, sheet_name, 'Toplam Risk', f'{c2}{r1}', ['H'])
    add_bar_chart(df, sheet_name, 'Toplam Risk', 'Toplam Risk', f'{c3}{r1}', ['K', 'M'], 'A', ['J', 'L'], False, True)
    add_bar_chart(df, sheet_name, 'Bankamız Memzuç Payı (%)', 'Bankamız Memzuç Payı (%)', f'{c4}{r1}', ['R', 'S'])
    add_bar_chart(df, sheet_name, 'Limit Doluluk Oranı (%)', 'Limit Doluluk Oranı (%)', f'{c4}{r2}', ['T', 'U'])

    i1, i2 = 23, 24
    n = r1
    c5 = c6 = c7 = c8 = 'V'
    
    if yz:
        add_bar_chart(df, sheet_name, 'Yapay Zeka Risk Sınıfı Ortalaması', 'Yapay Zeka Risk Sınıfı Ortalaması', f'{c5}{r1}', [letters[i1], letters[i2]])
        i1 += 2
        i2 += 2
        add_bar_chart(df, sheet_name, 'Yapay Zeka Risk Sınıfı >= 4 Oranı (%)', 'Yapay Zeka Risk Sınıfı >= 4 Oranı (%)', f'{c5}{r2}', [letters[i1], letters[i2]])
        c6 = c7 = c8 = 'AC'
        i1 += 2
        i2 += 2

    if eds:
        add_bar_chart(df, sheet_name, 'Entegre Derece Skor Ortalaması', 'Entegre Derece Skor Ortalaması', f'{c6}{r1}', [letters[i1], letters[i2]])
        i1 += 2
        i2 += 2
        add_bar_chart(df, sheet_name, 'Entegre Derece Skor >= 8 Oranı (%)', 'Entegre Derece Skor >= 8 Oranı (%)', f'{c6}{r2}', [letters[i1], letters[i2]])
        c7 = c8 = 'AJ'
        i1 += 2
        i2 += 2

    
    n = r1
    if sinif:
        add_bar_chart(df, sheet_name, 'Yakın İzleme Oranı (%)', 'Yakın İzleme Oranı (%)', f'{c7}{n}', [letters[i1], letters[i2]])
        c8 = 'AR'
        i1 += 2
        i2 += 2
        n = r2
    
    if fg:
        add_bar_chart(df, sheet_name, 'Finansal Güçlük Oranı (%)', 'Finansal Güçlük Oranı (%)', f'{c7}{n}', [letters[i1], letters[i2]])
        c8 = 'AR'
        i1 += 2
        i2 += 2
    
    n = r1
    if kroa:
        add_bar_chart(df, sheet_name, 'KRÖA Oranı (%)', 'KRÖA Oranı (%)', f'{c8}{n}', [letters[i1], letters[i2]])
        n = r2
        i1 += 2
        i2 += 2

    if gecikme:
        add_bar_chart(df, sheet_name, 'Gecikme Oranı (%)', 'Gecikme Oranı (%)', f'{c8}{n}', [letters[i1], letters[i2]])

# AAAAAAAAAA

In [412]:
# df_wide = df_wide_backup.copy()
# df_kr202_last = df_kr202_last_backup.copy()
# df_master = df_master_backup.copy()

# apply_filter = True
# tsa = 'Tahsis Sektör Adı'
# ppp = 'PPP (Kamu Garantili KÖİ) (HASTANE VE OTOYOLLAR)'
# ms = 'Müşteri Sınıfı'
# pk = 'Proje Kredisi'
# des = 'Değerlendirmeye Esas Segment'
# g = 'Gecikme'
# # filter_column, filter_value = ms, 2
# # filter_column, filter_value = des, s
# # segment_value = 'TİCARİ'
# filter_column, filter_value = 'EU', 'MS1 PK0'
# #  + ' - ' + segment_value
# musteri_value = 1
# gecikme_value = 0
# proje_value = 0

# wide_columns = []
# for c in df_wide[-1].columns:
#     if c.endswith(' ' + last_date):
#         x = c[:-(len(last_date) + 1)]
#         wide_columns.append(x)

# dfn = pd.DataFrame()
# first = True
# for i, (dfw, d) in enumerate(zip(df_wide, sorted_date_list)):
#     df = dfw.copy()
#     df = df.loc[df[tsa + ' ' + d] != ppp]
#     df = df.loc[df[ms + ' ' + d] <= musteri_value]
#     df = df.loc[df[pk + ' ' + d] <= proje_value]
#     # df = df.loc[df[des + ' ' + d] == segment_value]
#     df = df.loc[df[g + ' ' + d] <= gecikme_value]
#     df_wide[i] = df.copy()

# df = df_master.copy()
# df = df.loc[df[tsa] != ppp]
# df = df.loc[df[ms] <= musteri_value]
# df = df.loc[df[pk] <= proje_value]
# # df = df.loc[df[des] == segment_value]
# df = df.loc[df[g] <= gecikme_value]
# df_master = df.copy()

In [413]:
# def fix_aaa(df):
#     aacl = ['ENERJİ', 'METAL ÜRÜNLER', 'İNŞAAT TAAHHÜT', 'ULAŞTIRMA DEPOLAMA', 'GIDA', 'OTOMOTİV ULAŞIM ARAÇLARI', 'TİCARET', 'TURİZM', 'TEKSTİL', 'TAŞA TOPRAĞA DAYALI SANAYİ', 'KİMYA', 'DİĞER HİZMET FAALİYETLERİ', 'TARIM VE HAYVANCILIK', 'RAFİNERİ VE YAKIT TİCARETİ (AKARYAKIT, PETROL, KÖMÜR, GAZ)', 'ELEKTRONİK BİLİŞİM', 'TOPLAM']
#     aarcl = ['ENERJİ', 'METAL ÜRÜNLER', 'İNŞAAT VE TAAHHÜT', 'ULAŞTIRMA VE DEPOLAMA', 'GIDA', 'OTOMOTİV VE ULAŞIM ARAÇLARI', 'TİCARET', 'TURİZM', 'TEKSTİL', 'TAŞA TOPRAĞA DAYALI SANAYİ', 'KİMYA*', 'DİĞER HİZMET FAALİYETLERİ', 'TARIM VE HAYVANCILIK', 'RAFİNERİ VE YAKIT TİCARETİ', 'ELEKTRONİK BİLİŞİM', 'GENEL ORTALAMA']
#     c0 = list(df.columns)[0]
#     cf = list(df.columns)[2]
#     cl = list(df.columns)[-1]
#     df = df.loc[df[c0].isin(aacl), [c0] + list(df.columns)[2:]].reset_index(drop=True)
#     df[c0] = aarcl
#     df['10 Aylık Değişim'] = df[cl] - df[cf]
#     return df


# def fix_aaas(df):
#     c0 = list(df.columns)[0]
#     cf = list(df.columns)[2]
#     cl = list(df.columns)[-1]
#     df = df.loc[:, [c0] + list(df.columns)[2:]].reset_index(drop=True)
#     df['10 Aylık Değişim'] = df[cl] - df[cf]
#     return df

In [414]:
# def create_summary(r, sort_type=-2, fill_na=False, order_list=None, firma_sayisi=True, firma_sayisi_yuzde=True, nakdi_risk=True, nakdi_risk_yuzde=True, gayri_nakdi_risk=True, gayri_nakdi_risk_yuzde=True, toplam_risk=True, toplam_risk_yuzde=True,tl_yp_risk=True, tl_yp_risk_yuzde=True, memzuc=True, memzuc_payi=True, limit_doluluk=True, ortalama_limit_doluluk=True, yapay_zeka=True, entegre_derece=True, musteri_sinifi=True, finansal_gucluk=True, kroa=True, gecikme=True, include_sum=True, drop_empty=True, df_master=df_master):
#     c_list = [
#         'Müşteri No',
#         'Nakdi TL Reeskontlu',
#         'Nakdi YP Reeskontlu',
#         'Nakdi Reeskontlu Toplam',
#         'G.Nakdi TL Bakiye',
#         'G.Nakdi YP Bakiye',
#         'G.Nakdi Toplam',
#         'Toplam Reeskontlu Risk',
#         'Memzuç Risk',
#         'Memzuç Limit',
#         'Bankamız Risk',
#         'Bankamız Limit',
#         'Sektör Limit Doluluk Oranı (%)',
#         'Bankamız Limit Doluluk Oranı (%)',
#         'Yapay Zeka Risk Sınıfı',
#         'Entegre Derece Skor',
#         'Müşteri Sınıfı',
#         'Finansal Güçlük',
#         'KRÖA',
#         'Gecikme'
#     ]

#     bl, br = 'Bankamız Limit', 'Bankamız Risk'
#     ml, mr = 'Memzuç Limit', 'Memzuç Risk'

#     if r not in c_list:
#         c_list.append(r)
        
#     dfm = df_master[c_list].copy()

#     if fill_na:
#         dfm[r] = dfm[r].fillna('-')
#     else:
#         dfm = dfm.loc[dfm[r].notnull()]

#     df = pd.DataFrame()

#     if order_list is None:
#         df[r] = dfm.loc[dfm[r].notnull(), r].unique()
#     else:
#         df[r] = order_list

#     n_list = [r]
#     t_list = ['TOPLAM']
#     space = 0


#     if firma_sayisi:
#         c = 'Müşteri No'
#         n = 'Firma Sayısı'
#         dff = dfm[[r, c]]
#         dft = dff.groupby(r).count().rename(columns={c: n}).reset_index()
#         df = pd.merge(df, dft, how='left', on=r)
#         n_list.append(n)
#         t = len(dff)
#         t_list.append(t)

#         if firma_sayisi_yuzde:
#             nn = '%'  + ' ' * space
#             space += 1
#             df[nn] = df[n] / t * 100
#             n_list.append(nn)
#             t_list.append(100)


#     if nakdi_risk:
#         c = 'Nakdi Reeskontlu Toplam'
#         n = 'Nakdi Risk'
#         dff = dfm[[r, c]]
#         dft = dff.groupby(r).sum().rename(columns={c: n}).reset_index()
#         df = pd.merge(df, dft, how='left', on=r)
#         n_list.append(n)
#         t = dff[c].sum()
#         t_list.append(t)

#         if nakdi_risk_yuzde:
#             nn = '%'  + ' ' * space
#             space += 1
#             df[nn] = df[n] / t * 100
#             n_list.append(nn)
#             t_list.append(100)


#     if gayri_nakdi_risk:
#         c = 'G.Nakdi Toplam'
#         n = 'Gayrinakdi Risk'
#         dff = dfm[[r, c]]
#         dft = dff.groupby(r).sum().rename(columns={c: n}).reset_index()
#         df = pd.merge(df, dft, how='left', on=r)
#         n_list.append(n)
#         t = dff[c].sum()
#         t_list.append(t)

#         if gayri_nakdi_risk_yuzde:
#             nn = '%'  + ' ' * space
#             space += 1
#             df[nn] = df[n] / t * 100
#             n_list.append(nn)
#             t_list.append(100)
            

#     if toplam_risk:
#         c = 'Toplam Reeskontlu Risk'
#         n = 'Toplam Risk'
#         dff = dfm[[r, c]]
#         dft = dff.groupby(r).sum().rename(columns={c: n}).reset_index()
#         df = pd.merge(df, dft, how='left', on=r)
#         n_list.append(n)
#         t = dff[c].sum()
#         t_list.append(t)

#         if toplam_risk_yuzde:
#             nn = '%'  + ' ' * space
#             space += 1
#             df[nn] = df[n] / t * 100
#             n_list.append(nn)
#             t_list.append(100)


#     if tl_yp_risk:
#         for c_n, c_g, n in zip(['Nakdi TL Reeskontlu', 'Nakdi YP Reeskontlu'], ['G.Nakdi TL Bakiye', 'G.Nakdi YP Bakiye'], ['Toplam TL Risk', 'Toplam YP Risk']):
#             dff = dfm[[r, c_n, c_g]]
#             dft = dff.groupby(r).sum().reset_index()
#             dft[n] = dft[c_n] + dft[c_g]
#             df = pd.merge(df, dft, how='left', on=r)
#             n_list.append(n)
#             t = df[n].sum()
#             t_list.append(t)

#             if tl_yp_risk_yuzde:
#                 nn = '%'  + ' ' * space
#                 space += 1
#                 df[nn] = df[n] / df['Toplam Risk'] * 100
#                 t = df[n].sum() / df['Toplam Risk'].sum() * 100
#                 n_list.append(nn)
#                 t_list.append(t)


#     if memzuc:
#         for c in [mr, ml, br, bl]:
#             n = c
#             dff = dfm[[r, c]]
#             dft = dff.groupby(r).sum().rename(columns={c: n}).reset_index()
#             df = pd.merge(df, dft, how='left', on=r)
#             n_list.append(n)
#             t = dff[c].sum()
#             t_list.append(t)


#         if memzuc_payi:
#             for c_1, c_2, n in zip([bl, br], [ml, mr], ['Bankamız Memzuç Limit Payı (%)', 'Bankamız Memzuç Risk Payı (%)']):
#                 df[n] = df[c_1] / df[c_2] * 100
#                 t = df[c_1].sum() / df[c_2].sum() * 100
#                 n_list.append(n)
#                 t_list.append(t)


#         if limit_doluluk:
#             for c_1, c_2, n in zip([mr, br], [ml, bl], ['Sektör Limit Doluluk Oranı (%)', 'Bankamız Limit Doluluk Oranı (%)']):
#                 df[n] = df[c_1] / df[c_2] * 100
#                 t = df[c_1].sum() / df[c_2].sum() * 100
#                 n_list.append(n)
#                 t_list.append(t)

#         if ortalama_limit_doluluk:
#             c = 'Sektör Limit Doluluk Oranı (%)'
#             n = 'Sektör Ortalama Limit Doluluk Oranı (%)'
#             dff = dfm[[r, c]]
#             dft = dff.groupby(r).mean().rename(columns={c: n}).reset_index()
#             df = pd.merge(df, dft, how='left', on=r)
#             df[n] = df[n]
#             n_list.append(n)
#             t = dff[c].mean()
#             t_list.append(t)
        
#             c = 'Bankamız Limit Doluluk Oranı (%)'
#             n = 'Bankamız Ortalama Limit Doluluk Oranı (%)'
#             dff = dfm[[r, c]]
#             dft = dff.groupby(r).mean().rename(columns={c: n}).reset_index()
#             df = pd.merge(df, dft, how='left', on=r)
#             df[n] = df[n]
#             n_list.append(n)
#             t = dff[c].mean()
#             t_list.append(t)


#     if yapay_zeka:
#         c = 'Yapay Zeka Risk Sınıfı'
#         n = 'Nakdi Risk Ağırlıklı Yapay Zeka Risk Sınıfı Ortalaması'
#         x = 'Nakdi Reeskontlu Toplam'
#         cx = 'Yapay Zeka Risk Sınıfı * Nakdi Risk'
#         dff = dfm[[r, c, x]]
#         dff = dff.loc[dff[c].notnull()]
#         dff[cx] = dff[c] * dff[x] 
#         dft = dff[[r, cx, x]].groupby(r).sum().reset_index()
#         dft[n] = dft[cx] / dft[x]
#         dft = dft[[r, n]]
#         df = pd.merge(df, dft, how='left', on=r)
#         n_list.append(n)
#         t = dff[cx].sum() / dff[x].sum()
#         t_list.append(t)

        
#         n = 'Yapay Zeka Risk Sınıfı Ortalaması'
#         dff = dfm[[r, c, x]]
#         dff = dff.loc[dff[c].notnull()]
#         dft = dff[[r, c]].groupby(r).mean().rename(columns={c: n}).reset_index()
#         df = pd.merge(df, dft, how='left', on=r)
#         n_list.append(n)
#         t = dff[c].mean()
#         t_list.append(t)

#         n = 'Yapay Zeka Risk Sınıfı >= 4 (Adede Göre %)'
#         df1 = dff[[r, x]].groupby(r).count().reset_index()
#         df1.columns = [r, 'Tüm']
#         df2 = dff.loc[dff[c] >= 4, [r, x]].groupby(r).count().reset_index()
#         df2.columns = [r, 'Riskli']
#         dft = pd.merge(df1, df2, on=r)
#         dft[n] = dft['Riskli'] / dft['Tüm'] * 100
#         df = pd.merge(df, dft[[r, n]], how='left', on=r)
#         n_list.append(n)
#         t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
#         t_list.append(t)

#         n = 'Yapay Zeka Risk Sınıfı >= 4 (Riske Göre %)'
#         df1 = dff[[r, x]].groupby(r).sum().reset_index()
#         df1.columns = [r, 'Tüm']
#         df2 = dff.loc[dff[c] >= 4, [r, x]].groupby(r).sum().reset_index()
#         df2.columns = [r, 'Riskli']
#         dft = pd.merge(df1, df2, on=r)
#         dft[n] = dft['Riskli'] / dft['Tüm'] * 100
#         df = pd.merge(df, dft[[r, n]], how='left', on=r)
#         n_list.append(n)
#         t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
#         t_list.append(t)


#     if entegre_derece:
#         c = 'Entegre Derece Skor'
#         n = 'Nakdi Risk Ağırlıklı Entegre Derece Skor Ortalaması'
#         x = 'Nakdi Reeskontlu Toplam'
#         cx = 'Entegre Derece Skor * Nakdi Risk'
#         dff = dfm[[r, c, x]]
#         dff = dff.loc[dff[c].notnull()]
#         dff[cx] = dff[c] * dff[x] 
#         dft = dff[[r, cx, x]].groupby(r).sum().reset_index()
#         dft[n] = dft[cx] / dft[x]
#         dft = dft[[r, n]]
#         df = pd.merge(df, dft, how='left', on=r)
#         n_list.append(n)
#         t = dff[cx].sum() / dff[x].sum()
#         t_list.append(t)

        
#         n = 'Entegre Derece Skor Ortalaması'
#         dff = dfm[[r, c, x]]
#         dff = dff.loc[dff[c].notnull()]
#         dft = dff[[r, c]].groupby(r).mean().rename(columns={c: n}).reset_index()
#         df = pd.merge(df, dft, how='left', on=r)
#         n_list.append(n)
#         t = dff[c].mean()
#         t_list.append(t)

#         n = 'Entegre Derece Skor >= 8 (Adede Göre %)'
#         df1 = dff[[r, x]].groupby(r).count().reset_index()
#         df1.columns = [r, 'Tüm']
#         df2 = dff.loc[dff[c] >= 8, [r, x]].groupby(r).count().reset_index()
#         df2.columns = [r, 'Riskli']
#         dft = pd.merge(df1, df2, on=r)
#         dft[n] = dft['Riskli'] / dft['Tüm'] * 100
#         df = pd.merge(df, dft[[r, n]], how='left', on=r)
#         n_list.append(n)
#         t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
#         t_list.append(t)

#         n = 'Entegre Derece Skor >= 8 (Riske Göre %)'
#         df1 = dff[[r, x]].groupby(r).sum().reset_index()
#         df1.columns = [r, 'Tüm']
#         df2 = dff.loc[dff[c] >= 8, [r, x]].groupby(r).sum().reset_index()
#         df2.columns = [r, 'Riskli']
#         dft = pd.merge(df1, df2, on=r)
#         dft[n] = dft['Riskli'] / dft['Tüm'] * 100
#         df = pd.merge(df, dft[[r, n]], how='left', on=r)
#         n_list.append(n)
#         t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
#         t_list.append(t)


#     if musteri_sinifi:
#         c = 'Müşteri Sınıfı'
#         x = 'Nakdi Reeskontlu Toplam'
#         n = 'Yakın İzleme (Adede Göre %)'
#         dff = dfm[[r, c, x]]
#         dff = dff.loc[dff[c].notnull()]

#         df1 = dff[[r, x]].groupby(r).count().reset_index()
#         df1.columns = [r, 'Tüm']
#         df2 = dff.loc[dff[c] >= 2, [r, x]].groupby(r).count().reset_index()
#         df2.columns = [r, 'Riskli']
#         dft = pd.merge(df1, df2, on=r)
#         dft[n] = dft['Riskli'] / dft['Tüm'] * 100
#         df = pd.merge(df, dft[[r, n]], how='left', on=r)
#         n_list.append(n)
#         t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
#         t_list.append(t)

#         n = 'Yakın İzleme (Riske Göre %)'
#         df1 = dff[[r, x]].groupby(r).sum().reset_index()
#         df1.columns = [r, 'Tüm']
#         df2 = dff.loc[dff[c] >= 2, [r, x]].groupby(r).sum().reset_index()
#         df2.columns = [r, 'Riskli']
#         dft = pd.merge(df1, df2, on=r)
#         dft[n] = dft['Riskli'] / dft['Tüm'] * 100
#         df = pd.merge(df, dft[[r, n]], how='left', on=r)
#         n_list.append(n)
#         t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
#         t_list.append(t)


#     if finansal_gucluk:        
#         c = 'Finansal Güçlük'
#         x = 'Nakdi Reeskontlu Toplam'
#         n = 'Finansal Güçlük (Adede Göre %)'
#         dff = dfm[[r, c, x]]
#         dff = dff.loc[dff[c].notnull()]
        
#         df1 = dff[[r, x]].groupby(r).count().reset_index()
#         df1.columns = [r, 'Tüm']
#         df2 = dff.loc[dff[c] == 1, [r, x]].groupby(r).count().reset_index()
#         df2.columns = [r, 'Riskli']
#         dft = pd.merge(df1, df2, on=r)
#         dft[n] = dft['Riskli'] / dft['Tüm'] * 100
#         df = pd.merge(df, dft[[r, n]], how='left', on=r)
#         n_list.append(n)
#         t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
#         t_list.append(t)

#         n = 'Finansal Güçlük (Riske Göre %)'
#         df1 = dff[[r, x]].groupby(r).sum().reset_index()
#         df1.columns = [r, 'Tüm']
#         df2 = dff.loc[dff[c] == 1, [r, x]].groupby(r).sum().reset_index()
#         df2.columns = [r, 'Riskli']
#         dft = pd.merge(df1, df2, on=r)
#         dft[n] = dft['Riskli'] / dft['Tüm'] * 100
#         df = pd.merge(df, dft[[r, n]], how='left', on=r)
#         n_list.append(n)
#         t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
#         t_list.append(t)


#     if kroa:        
#         c = 'KRÖA'
#         x = 'Nakdi Reeskontlu Toplam'
#         n = 'KRÖA (Adede Göre %)'
#         dff = dfm[[r, c, x]]
#         dff = dff.loc[dff[c].notnull()]
        
#         df1 = dff[[r, x]].groupby(r).count().reset_index()
#         df1.columns = [r, 'Tüm']
#         df2 = dff.loc[dff[c] == 1, [r, x]].groupby(r).count().reset_index()
#         df2.columns = [r, 'Riskli']
#         dft = pd.merge(df1, df2, on=r)
#         dft[n] = dft['Riskli'] / dft['Tüm'] * 100
#         df = pd.merge(df, dft[[r, n]], how='left', on=r)
#         n_list.append(n)
#         t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
#         t_list.append(t)

#         n = 'KRÖA (Riske Göre %)'
#         df1 = dff[[r, x]].groupby(r).sum().reset_index()
#         df1.columns = [r, 'Tüm']
#         df2 = dff.loc[dff[c] == 1, [r, x]].groupby(r).sum().reset_index()
#         df2.columns = [r, 'Riskli']
#         dft = pd.merge(df1, df2, on=r)
#         dft[n] = dft['Riskli'] / dft['Tüm'] * 100
#         df = pd.merge(df, dft[[r, n]], how='left', on=r)
#         n_list.append(n)
#         t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
#         t_list.append(t)


#     if gecikme:        
#         c = 'Gecikme'
#         x = 'Nakdi Reeskontlu Toplam'
#         n = 'Gecikme (Adede Göre %)'
#         dff = dfm[[r, c, x]]
#         dff = dff.loc[dff[c].notnull()]
        
#         df1 = dff[[r, x]].groupby(r).count().reset_index()
#         df1.columns = [r, 'Tüm']
#         df2 = dff.loc[dff[c] == 1, [r, x]].groupby(r).count().reset_index()
#         df2.columns = [r, 'Riskli']
#         dft = pd.merge(df1, df2, on=r)
#         dft[n] = dft['Riskli'] / dft['Tüm'] * 100
#         df = pd.merge(df, dft[[r, n]], how='left', on=r)
#         n_list.append(n)
#         t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
#         t_list.append(t)

#         n = 'Gecikme (Riske Göre %)'
#         df1 = dff[[r, x]].groupby(r).sum().reset_index()
#         df1.columns = [r, 'Tüm']
#         df2 = dff.loc[dff[c] == 1, [r, x]].groupby(r).sum().reset_index()
#         df2.columns = [r, 'Riskli']
#         dft = pd.merge(df1, df2, on=r)
#         dft[n] = dft['Riskli'] / dft['Tüm'] * 100
#         df = pd.merge(df, dft[[r, n]], how='left', on=r)
#         n_list.append(n)
#         t = dft['Riskli'].sum() / dft['Tüm'].sum() * 100
#         t_list.append(t)
        

#     df = df[n_list]
#     if fill_na:
#         dfna = df.loc[df[r] == '-']
#         df = df.loc[df[r] != '-']
#     if order_list is None:
#         if sort_type == 1:
#             df = df.sort_values(r).reset_index(drop=True)
#         elif sort_type == -1:
#             df = df.sort_values(r, ascending=False).reset_index(drop=True)
#         elif sort_type == 2:
#             df = df.sort_values('Nakdi Risk').reset_index(drop=True)
#         elif sort_type == -2:
#             df = df.sort_values('Nakdi Risk', ascending=False).reset_index(drop=True)
#         elif sort_type == 3:
#             df = df.sort_values('Firma Sayısı').reset_index(drop=True)
#         elif sort_type == -3:
#             df = df.sort_values('Firma Sayısı', ascending=False).reset_index(drop=True)
#     if fill_na:
#         df = pd.concat([df, dfna], ignore_index=True)

#     if drop_empty:
#         df = df.dropna(axis=0, how='all', subset=list(df.columns)[1:], ignore_index=True)
#     if include_sum and len(df) > 1:
#         df.loc[len(df)] = t_list
#     return df

In [415]:
# df_segment_ozet = create_summary('Değerlendirmeye Esas Segment', order_list=segment_list)
# df_segment_yz_trend_weighted = create_mean_trend('Değerlendirmeye Esas Segment', 'Yapay Zeka Risk Sınıfı', segment_list, 'Nakdi Reeskontlu Toplam', drop_empty=True)
# df_segment_yz_trend = create_mean_trend('Değerlendirmeye Esas Segment', 'Yapay Zeka Risk Sınıfı', segment_list, drop_empty=True)

# # df_segment_yz_trend_weighted = fix_aaas(df_segment_yz_trend_weighted)
# # df_segment_yz_trend = fix_aaas(df_segment_yz_trend)


# df_sektor_ozet = create_summary('Tahsis Sektör Adı')
# df_sektor_yz_trend_weighted = create_mean_trend('Tahsis Sektör Adı', 'Yapay Zeka Risk Sınıfı', sektor_list, 'Nakdi Reeskontlu Toplam', drop_empty=True)
# df_sektor_yz_trend = create_mean_trend('Tahsis Sektör Adı', 'Yapay Zeka Risk Sınıfı', sektor_list, drop_empty=True)

# # df_sektor_yz_trend_weighted = fix_aaa(df_sektor_yz_trend_weighted)
# # df_sektor_yz_trend = fix_aaa(df_sektor_yz_trend)

 
# # df_segment_eds_trend_weighted = create_mean_trend('Değerlendirmeye Esas Segment', 'Entegre Derece Skor', segment_list, 'Nakdi Reeskontlu Toplam')
# # df_sektor_eds_trend_weighted = create_mean_trend('Tahsis Sektör Adı', 'Entegre Derece Skor', sektor_list, 'Nakdi Reeskontlu Toplam')
# # df_segment_eds_trend = create_mean_trend('Değerlendirmeye Esas Segment', 'Entegre Derece Skor', segment_list)
# # df_sektor_eds_trend = create_mean_trend('Tahsis Sektör Adı', 'Entegre Derece Skor', sektor_list)


# Başlıklar

In [416]:
firm_sheet_dict = {
    'Firma Listesi': df_master,
}
    
summary_sheet_dict = {
    'Segment Bazlı Özet': df_segment_ozet,
    # 'İlgili Bölüm Bazlı Özet': df_bolum_ozet,
    # 'Yetki Kodu Bazlı Özet': df_yetki_ozet,
    # 'Bölge Adı Bazlı Özet': df_bolge_ozet,
    'Sektör Bazlı Özet': df_sektor_ozet,
    # 'NACE Harf Bazlı Özet': df_nace_ozet,
    # 'Müşteri Sınıfı Bazlı Özet': df_sinif_ozet,
    # 'Yapay Zeka Risk Sınıfı Bazlı Özet': df_yz_ozet,
    # 'Entegre Derece Skor Bazlı Özet': df_eds_ozet,
    # 'Şube Bazlı Özet': df_sube_ozet,
}


risk_sheet_dict =  {
    'Segment * Nakdi Risk (Milyon TL)': df_segment_risk_trend,
    # 'İlgili Bölüm * Nakdi Risk (Milyon TL)': df_bolum_risk_trend,
    # 'Yetki Kodu * Nakdi Risk (Milyon TL)': df_yetki_risk_trend,
    # 'Bölge Adı * Nakdi Risk (Milyon TL)': df_bolge_risk_trend,
    'Sektör * Nakdi Risk (Milyon TL)': df_sektor_risk_trend,
    # 'NACE Harf * Nakdi Risk (Milyon TL)': df_nace_risk_trend,
    # 'Müşteri Sınıfı * Nakdi Risk (Milyon TL)': df_sinif_risk_trend,
    # 'Yapay Zeka Risk Sınıfı * Nakdi Risk (Milyon TL)': df_yz_risk_trend,
    # 'Entegre Derece Skor * Nakdi Risk (Milyon TL)': df_eds_risk_trend,
    # 'Şube * Nakdi Risk (Milyon TL)': df_sube_risk_trend,
}


trend_sheet_dict = {
    'Segment * Ortalama Yapay Zeka Risk Sınıfı': df_segment_yz_trend,
    'Segment * Ortalama Müşteri Sınıfı': df_segment_sinif_trend,
    'Segment * Ortalama Entegre Derece Skor': df_segment_eds_trend,
    'Segment * Ortalama Yapay Zeka Risk Sınıfı (Nakdi Risk Ağırlıklı)': df_segment_yz_trend_weighted,
    'Segment * Ortalama Müşteri Sınıfı (Nakdi Risk Ağırlıklı)': df_segment_sinif_trend_weighted,
    'Segment * Ortalama Entegre Derece Skor (Nakdi Risk Ağırlıklı)': df_segment_eds_trend_weighted,
    # 'İlgili Bölüm * Ortalama Yapay Zeka Risk Sınıfı': df_bolum_yz_trend,
    # 'İlgili Bölüm * Ortalama Müşteri Sınıfı': df_bolum_sinif_trend,
    # 'İlgili Bölüm * Ortalama Entegre Derece Skor': df_bolum_eds_trend,
    # 'Yetki Kodu * Ortalama Yapay Zeka Risk Sınıfı': df_yetki_yz_trend,
    # 'Yetki Kodu * Ortalama Müşteri Sınıfı': df_yetki_sinif_trend,
    # 'Yetki Kodu * Ortalama Entegre Derece Skor': df_yetki_eds_trend,
    # 'Bölge Adı * Ortalama Yapay Zeka Risk Sınıfı': df_bolge_yz_trend,
    # 'Bölge Adı * Ortalama Müşteri Sınıfı': df_bolge_sinif_trend,
    # 'Bölge Adı * Ortalama Entegre Derece Skor': df_bolge_eds_trend,
    'Sektör * Ortalama Yapay Zeka Risk Sınıfı': df_sektor_yz_trend,
    'Sektör * Ortalama Müşteri Sınıfı': df_sektor_sinif_trend,
    'Sektör * Ortalama Entegre Derece Skor': df_sektor_eds_trend,
    'Sektör * Ortalama Yapay Zeka Risk Sınıfı (Nakdi Risk Ağırlıklı)': df_sektor_yz_trend_weighted,
    'Sektör * Ortalama Müşteri Sınıfı (Nakdi Risk Ağırlıklı)': df_sektor_sinif_trend_weighted,
    'Sektör * Ortalama Entegre Derece Skor (Nakdi Risk Ağırlıklı)': df_sektor_eds_trend_weighted,
    # 'NACE Harf * Ortalama Yapay Zeka Risk Sınıfı': df_nace_yz_trend,
    # 'NACE Harf * Ortalama Müşteri Sınıfı': df_nace_sinif_trend,
    # 'NACE Harf * Ortalama Entegre Derece Skor': df_nace_eds_trend,
    # 'Müşteri Sınıfı * Ortalama Yapay Zeka Risk Sınıfı': df_sinif_yz_trend,
    # 'Müşteri Sınıfı * Ortalama Entegre Derece Skor': df_sinif_eds_trend,
    # 'Yapay Zeka Risk Sınıfı * Ortalama Müşteri Sınıfı': df_yz_sinif_trend,
    # 'Yapay Zeka Risk Sınıfı * Ortalama Entegre Derece Skor': df_yz_eds_trend,
    # 'Entegre Derece Skor * Ortalama Yapay Zeka Risk Sınıfı': df_eds_yz_trend,
    # 'Entegre Derece Skor * Ortalama Müşteri Sınıfı': df_eds_sinif_trend,
    # 'Şube * Ortalama Yapay Zeka Risk Sınıfı': df_sube_yz_trend,
    # 'Şube * Ortalama Müşteri Sınıfı': df_sube_sinif_trend,
    # 'Şube * Ortalama Entegre Derece Skor': df_sube_eds_trend,
}


multi_trend_sheet_dict = {
    'Segment * Yapay Zeka Risk Sınıfı >= 4': df_segment_yz_multi_trend,
    'Segment * Müşteri Sınıfı = 2': df_segment_sinif_multi_trend,
    'Segment * Entegre Derece Skor >= 8': df_segment_eds_multi_trend,
    'Segment * Finansal Güçlük': df_segment_fg_multi_trend,
    'Segment * KRÖA': df_segment_kroa_multi_trend,
    'Segment * Gecikme': df_segment_gecikme_multi_trend,
    'Segment * Karşılıksız Çek': df_segment_cek_multi_trend,
    # 'İlgili Bölüm * Yapay Zeka Risk Sınıfı >= 4': df_bolum_yz_multi_trend,
    # 'İlgili Bölüm * Müşteri Sınıfı = 2': df_bolum_sinif_multi_trend,
    # 'İlgili Bölüm * Entegre Derece Skor >= 8': df_bolum_eds_multi_trend,
    # 'İlgili Bölüm * Finansal Güçlük': df_bolum_fg_multi_trend,
    # 'İlgili Bölüm * KRÖA': df_bolum_kroa_multi_trend,
    # 'İlgili Bölüm * Gecikme': df_bolum_gecikme_multi_trend,
    # 'İlgili Bölüm * Karşılıksız Çek': df_bolum_cek_multi_trend,
    # 'Yetki Kodu * Yapay Zeka Risk Sınıfı >= 4': df_yetki_yz_multi_trend,
    # 'Yetki Kodu * Müşteri Sınıfı = 2': df_yetki_sinif_multi_trend,
    # 'Yetki Kodu * Entegre Derece Skor >= 8': df_yetki_eds_multi_trend,
    # 'Yetki Kodu * Finansal Güçlük': df_yetki_fg_multi_trend,
    # 'Yetki Kodu * KRÖA': df_yetki_kroa_multi_trend,
    # 'Yetki Kodu * Gecikme': df_yetki_gecikme_multi_trend,
    # 'Yetki Kodu * Karşılıksız Çek': df_yetki_cek_multi_trend,
    # 'Bölge Adı * Yapay Zeka Risk Sınıfı >= 4': df_bolge_yz_multi_trend,
    # 'Bölge Adı * Müşteri Sınıfı = 2': df_bolge_sinif_multi_trend,
    # 'Bölge Adı * Entegre Derece Skor >= 8': df_bolge_eds_multi_trend,
    # 'Bölge Adı * Finansal Güçlük': df_bolge_fg_multi_trend,
    # 'Bölge Adı * KRÖA': df_bolge_kroa_multi_trend,
    # 'Bölge Adı * Gecikme': df_bolge_gecikme_multi_trend,
    # 'Bölge Adı * Karşılıksız Çek': df_bolge_cek_multi_trend,
    'Sektör * Yapay Zeka Risk Sınıfı >= 4': df_sektor_yz_multi_trend,
    'Sektör * Müşteri Sınıfı = 2': df_sektor_sinif_multi_trend,
    'Sektör * Entegre Derece Skor >= 8': df_sektor_eds_multi_trend,
    'Sektör * Finansal Güçlük': df_sektor_fg_multi_trend,
    'Sektör * KRÖA': df_sektor_kroa_multi_trend,
    'Sektör * Gecikme': df_sektor_gecikme_multi_trend,
    'Sektör * Karşılıksız Çek': df_sektor_cek_multi_trend,
    # 'NACE Harf * Yapay Zeka Risk Sınıfı >= 4': df_nace_yz_multi_trend,
    # 'NACE Harf * Müşteri Sınıfı = 2': df_nace_sinif_multi_trend,
    # 'NACE Harf * Entegre Derece Skor >= 8': df_nace_eds_multi_trend,
    # 'NACE Harf * Finansal Güçlük': df_nace_fg_multi_trend,
    # 'NACE Harf * KRÖA': df_nace_kroa_multi_trend,
    # 'NACE Harf * Gecikme': df_nace_gecikme_multi_trend,
    # 'NACE Harf * Karşılıksız Çek': df_nace_cek_multi_trend,
    # 'Şube * Yapay Zeka Risk Sınıfı >= 4': df_sube_yz_multi_trend,
    # 'Şube * Müşteri Sınıfı = 2': df_sube_sinif_multi_trend,
    # 'Şube * Entegre Derece Skor >= 8': df_sube_eds_multi_trend,
    # 'Şube * Finansal Güçlük': df_sube_fg_multi_trend,
    # 'Şube * KRÖA': df_sube_kroa_multi_trend,
    # 'Şube * Gecikme': df_sube_gecikme_multi_trend,
    # 'Şube * Karşılıksız Çek': df_sube_cek_multi_trend,
    # 'İl * Yapay Zeka Risk Sınıfı >= 4': df_il_yz_multi_trend,
    # 'İl * Müşteri Sınıfı = 2': df_il_sinif_multi_trend,
    # 'İl * Entegre Derece Skor >= 8': df_il_eds_multi_trend,
    # 'İl * Finansal Güçlük': df_il_fg_multi_trend,
    # 'İl * KRÖA': df_il_kroa_multi_trend,
    # 'İl * Gecikme': df_il_gecikme_multi_trend,
    # 'İl * Karşılıksız Çek' : df_il_cek_multi_trend,
}

matrix_sheet_dict = {
    # 'Yapay Zeka Risk Sınıfı Değişimi': df_yz_matrix,
    # 'Müşteri Sınıfı Değişimi': df_sinif_matrix,
    # 'Entegre Derece Skor Değişimi': df_eds_matrix,
    # 'Yapay Zeka Risk Sınıfı Değişimi Özet': df_yz_matrix_condensed,
    # 'Müşteri Sınıfı Değişimi Özet': df_sinif_matrix_condensed,
    # 'Entegre Derece Skor Değişimi Özet': df_eds_matrix_condensed,
}

switch_sheet_dict = {
    # 'Yapay Zeka Risk Sınıfı 3 → 4': df_yz_3_ten_4,
    # 'Yapay Zeka Risk Sınıfı 4 → 3': df_yz_4_ten_3,
    # 'Müşteri Sınıfı 1 → 2': df_sinif_1_den_2,
    # 'Müşteri Sınıfı 2 → 1': df_sinif_2_den_1,
    # 'Entegre Derece Skor 7 → 8': df_eds_7_den_8,
    # 'Entegre Derece Skor 8 → 7': df_eds_8_den_7,
}

scoring_sheet_dict = {
    'Segment Skorlama': df_segment_skor,
    # 'İlgili Bölüm Skorlama': df_bolum_skor,
    # 'Yetki Kodu Skorlama': df_yetki_skor,
    # 'Bölge Adı Skorlama': df_bolge_skor,
    'Sektör Skorlama': df_sektor_skor,
    # 'NACE Harf Skorlama': df_nace_skor,
    # 'Şube Skorlama': df_sube_skor,
    # 'İl Skorlama': df_il_skor,
}

zombie_sheet_dict = {
    # 'Zombi Firmalar - Özet': df_zombi_sunum,
    # 'Zombi Firmalar - Sektör Kırılımı': df_zombi_ard,
    # 'Zombi Firmalar - Segment Kırılımı': df_zombi_ard_segment,
    # 'Zombi Firmalar - Yetki Kodu Kırılımı': df_zombi_ard_yetki,
}

pra_sheet_dict = {
    # 'Portföy Risklilik Analizi - Küme': df_pra_kume,
    # 'Portföy Risklilik Analizi - Toplam Puan': df_pra_ozet,
    # 'Portföy Risklilik Analizi - Toplam Ekstra Puan': df_pra_ozet_ekstra,
    # 'Portföy Risklilik Analizi - Sunum': df_pra_sunum,
    # 'Portföy Risklilik Analizi - Sektör Kırılımı': df_pra_ard,
    # 'Portföy Risklilik Analizi - Segment Kırılımı': df_pra_ard_segment,
    # 'Portföy Risklilik Analizi - Yüksek Riskli Firmalar': df_pra_yr,
    # 'Portföy Risklilik Analizi - Tüm Firmalar': df_pra,
}



sheet_dict_list = [firm_sheet_dict, summary_sheet_dict, risk_sheet_dict, trend_sheet_dict, multi_trend_sheet_dict, matrix_sheet_dict, switch_sheet_dict, scoring_sheet_dict, zombie_sheet_dict, pra_sheet_dict]

page_list = [item for sublist in [list(x.keys()) for x in sheet_dict_list] for item in sublist]
df_list = [item for sublist in [list(x.values()) for x in sheet_dict_list] for item in sublist]
sheet_list = list(range(len(page_list)))

sheet_list_list = []
ll = [len(x.keys()) for x in sheet_dict_list]
c = 0
for i, l in enumerate(ll):
    sheet_list_list.append(list(range(c, c + l)))
    c += l

firm_sheet_list, summary_sheet_list, risk_sheet_list, trend_sheet_list, multi_trend_sheet_list, matrix_sheet_list, switch_sheet_list, scoring_sheet_list, zombie_sheet_list, pra_sheet_list = sheet_list_list


df_index = pd.DataFrame()
sheet_list = [x for x in range(len(page_list))]
df_index['Sayfa'] = sheet_list
df_index['Başlık'] = df_index['Sayfa'].apply(lambda x: '=HYPERLINK("#{}!A1", "{}")'.format(str(x), page_list[x]))
sheet_color_list = ['E5E1E6', 'C7CEEA', 'B2CBF7', 'B5EAD7', 'E2F0CB', 'F7ECC8', 'FFDAC1', 'FFB7B2', 'FF9AA2', 'DCB5CB', 'AF8FC1']

In [417]:
multi_df_list = [
    segment_yz_multi_trend, segment_sinif_multi_trend, segment_eds_multi_trend, segment_fg_multi_trend, segment_kroa_multi_trend, segment_gecikme_multi_trend, segment_cek_multi_trend,
    # bolum_yz_multi_trend, bolum_sinif_multi_trend, bolum_eds_multi_trend, bolum_fg_multi_trend, bolum_kroa_multi_trend, bolum_gecikme_multi_trend, bolum_cek_multi_trend,
    # yetki_yz_multi_trend, yetki_sinif_multi_trend, yetki_eds_multi_trend, yetki_fg_multi_trend, yetki_kroa_multi_trend, yetki_gecikme_multi_trend, yetki_cek_multi_trend,
    # bolge_yz_multi_trend, bolge_sinif_multi_trend, bolge_eds_multi_trend, bolge_fg_multi_trend, bolge_kroa_multi_trend, bolge_gecikme_multi_trend, bolge_cek_multi_trend,
    sektor_yz_multi_trend, sektor_sinif_multi_trend, sektor_eds_multi_trend, sektor_fg_multi_trend, sektor_kroa_multi_trend, sektor_gecikme_multi_trend, sektor_cek_multi_trend,
    # nace_yz_multi_trend, nace_sinif_multi_trend, nace_eds_multi_trend, nace_fg_multi_trend, nace_kroa_multi_trend, nace_gecikme_multi_trend, nace_cek_multi_trend,
    # sube_yz_multi_trend, sube_sinif_multi_trend, sube_eds_multi_trend, sube_fg_multi_trend, sube_kroa_multi_trend, sube_gecikme_multi_trend, sube_cek_multi_trend,
    # il_yz_multi_trend, il_sinif_multi_trend, il_eds_multi_trend, il_fg_multi_trend, il_kroa_multi_trend, il_gecikme_multi_trend, il_cek_multi_trend
]

multi_title_prefix_list = [
    'Yapay Zeka Risk Sınıfı >= 4',
    'Müşteri Sınıfı = 2',
    'Entegre Derece Skor >= 8',
    'Finansal Güçlük',
    'KRÖA',
    'Gecikme',
    'Karşılıksız Çek'
]

multi_title_suffix_list = [
    'Oranı (Adede Göre %)',
    'Firma Adedi',
    'Oranı (Riske Göre %)',
    'Toplam Nakdi Risk (Milyon TL)'
]

multi_title_list = []
for i in range(int(len(multi_df_list)/len(multi_title_prefix_list))):
    for p in multi_title_prefix_list:
        t = []
        for s in multi_title_suffix_list:
            t.append(p + ' ' + s)
        multi_title_list.append(t)

multi_label_list = [[
    'Adede Göre %',
    'Firma Adedi',
    'Riske Göre %',
    'Toplam Nakdi Risk (Milyon TL)'
]] * len(multi_title_list)


multi_index_list = []
for x in multi_title_prefix_list:
    for i, p in enumerate(page_list):
        if x in p and 'Performans' not in p:
            multi_index_list.append(i)
            
multi_index_list = sorted(multi_index_list)

multi_row = []
for df in multi_df_list:
    n_space = h_space if len(df[0]) < h_limit else l_space
    x = df[0].shape[0] + n_space + 1
    multi_row.append(x)

multi_column = [df[0].shape[1] + w_space for df in multi_df_list]
multi_width = [df[0].shape[1] for df in multi_df_list]

# Final XLSX

In [419]:
output_file_name = 'Panel v64'
selective = None
skip_first = True
# skip_first = apply_filter
# selective = summary_sheet_list + trend_sheet_list
# selective = pra_sheet_list

open_file = True

if apply_filter:
    output_file_name += f' - {filter_column} - {filter_value}'

color_long_multi_trend, color_multi_trend, color_trend, row_based = True, True, True, True
output_file_path = output_folder_path + output_file_name + '.xlsx'
writer = pd.ExcelWriter(output_file_path, engine = 'xlsxwriter')

workbook = writer.book
bold_format = workbook.add_format({'bold': True, 'text_wrap': True})
row_format = workbook.add_format({'align': 'left'})
link_format = workbook.add_format({'underline': True, 'color': 'blue'})
float_format = workbook.add_format()
float_format.set_num_format('#,##0.00')
int_format = workbook.add_format()
int_format.set_num_format('#,##0')

color_format_list = []
for i in range(len(sheet_list_list)):
    color = sheet_color_list[i]
    cf = workbook.add_format()
    cf.set_bg_color(color)
    color_format_list.append(cf)

colored_title_format = workbook.add_format({'bold': True})
colored_title_format.set_bg_color('C9C4CF')


df_index.to_excel(writer, sheet_name = 'Başlıklar', index = False)
for i, df in enumerate(df_list):
    if skip_first and i == 0:
        continue
    if selective is not None and i not in selective:
        continue
    sheet = str(i)
    df.to_excel(writer, sheet_name = sheet, index = False)

    worksheet = writer.sheets[sheet]

    if i in multi_index_list:
        index = multi_index_list.index(i)
        row = multi_row[index]
        column = multi_column[index]
        end = multi_width[index]
        worksheet.set_row(0, 60, bold_format)
        worksheet.set_row(row, 60, bold_format)
        if 'Sektör' in multi_df_list[multi_index_list.index(i)][0].columns[0]:
            worksheet.set_column(0, 0, 60, row_format)
            worksheet.set_column(column, column, 60, row_format)
        else:
            worksheet.set_column(0, 0, 18, row_format)
            worksheet.set_column(column, column, 18, row_format)
        worksheet.set_column(1, end - 1, 10, float_format)
        worksheet.set_column(column + 1, df.shape[1] - 1, 10, int_format)

    else:
        all_int = False
        all_float = False
        if 'Nakdi Reeskontlu Toplam' in df.columns[0]:
            all_int = True
        for j, c in enumerate(df.columns):
            worksheet.set_row(0, 60, bold_format)
            worksheet.set_column(0, 0, 18, row_format)
            if all_int:
                worksheet.set_column(j, j, 15, int_format)
            elif all_float:
                worksheet.set_column(j, j, 15, float_format)
            elif any(x in str(c).lower() for x in ['%', 'ortalama', '2022', '2023', 'n-']):
                worksheet.set_column(j, j, 10, float_format)
            else:
                worksheet.set_column(j, j, 16, int_format)


worksheet = writer.sheets['Başlıklar']
worksheet.set_row(0, None, bold_format)
worksheet.set_column(1, 1, 70, link_format)
worksheet.write('A1', df_index.columns[0], colored_title_format)
worksheet.write('B1', df_index.columns[1], colored_title_format)

for x, cf in enumerate(color_format_list):
    for i in sheet_list_list[x]:
        worksheet.write(f'A{i + 2}', i, cf)
        worksheet.write(f'B{i + 2}', df_index.loc[i, 'Başlık'], cf)

for i in risk_sheet_list + trend_sheet_list:
    if selective is not None and i not in selective:
        continue
    add_line_chart(df_list[i], str(i), page_list[i], page_list[i])
    if color_trend:
        add_color_scale(i, row_based)

for i, label, title, df in zip(multi_index_list, multi_label_list, multi_title_list, multi_df_list):
    if selective is not None and i not in selective:
        continue
    if len(df[0]) < h_limit:
        add_multi_line_chart(df, str(i), label, title)
        if color_multi_trend:
            add_color_scale(i, row_based, True)
    elif color_long_multi_trend:
        add_color_scale(i, row_based, True)

for i in summary_sheet_list:
    if selective is not None and i not in selective:
        continue
    add_summary_charts(i)

for i in scoring_sheet_list:
    if selective is not None and i not in selective:
        continue
    add_color_scale(i, row_based, vertical=True, gray_empty=False)

writer.close()

if open_file:
    os.startfile(output_file_path)

# PLAYGROUND

In [ ]:
# dfss = []
# sss = []
# for i in [-3, -2, -1]:
#     d = sorted_date_list[i]
#     df = pd.merge(df_kr202_all[i], df_eds_all[i], on='Müşteri No', how='left')
#     df = pd.merge(df, kr202_df_raw_list[i][['Musteri No', 'Adi']].rename(columns={'Musteri No': 'Müşteri No'}), on='Müşteri No', how='left')
#     df = df.loc[df['Tahsis Sektör Adı ' + d] == 'DİĞER İMALAT']
#     # quick_export(df, 'Diğer İmalat ' + d)
#     dfss.append(df)
#     sss.append(d)
# quick_export(dfss, 'Diğer İmalat 2023.08-10', sss)

In [ ]:
# fn = 'NACE KR202.csv'
# dfnk = pd.read_csv(base_path + 'ML\\' + fn, sep=';', encoding='ANSI')
# fn = 'NACE Eski.csv'
# dfne = pd.read_csv(base_path + 'ML\\' + fn, sep=';', encoding='ANSI')
# fn = 'NACE Yeni.csv'
# dfny = pd.read_csv(base_path + 'ML\\' + fn, sep=';', encoding='ANSI')

In [ ]:
# def checknace(df):
#     return df['NACE KR202'] == df['NACE Eski'] == df['NACE Yeni']

# dfnn = pd.merge(dfnk, dfne, on='Müşteri No', how='left')
# dfnn = pd.merge(dfnn, dfny, on='Müşteri No', how='left')
# dfnn = dfnn.loc[~dfnn['Müşteri No'].isnull()]
# ns = ['NACE KR202', 'NACE Eski', 'NACE Yeni']
# dfnn = dfnn.dropna(subset = ns)
# for c in ns:
#     dfnn[c] = dfnn[c].apply(lambda x: x.strip())

# dfnn['Kontrol'] = dfnn.apply(checknace, axis=1)
# # dfnn['Kontrol'] = dfnn['NACE KR202'] == dfnn['NACE Eski'] == dfnn['NACE Yeni']
# # dfnn['K-E'] = dfnn['NACE KR202'] == dfnn['NACE Eski']
# # dfnn['K-Y'] = dfnn['NACE KR202'] == dfnn['NACE Yeni']
# # dfnn['E-Y'] = dfnn['NACE Eski'] == dfnn['NACE Yeni']

# dfnn = dfnn.loc[dfnn['Kontrol'] == False]

# dfnn['Kontrol'].value_counts()
# # dfnn['K-Y'].value_counts()
# quick_export(dfnn, 'NACE Check')

In [ ]:
# quick_export(dfnn, 'NACE Check')

In [ ]:
# df_master['Tahsis Sektör Adı'].value_counts()
# a = 'Adı'
# x = 'Bankamız Risk'
# y = 'Bankamız Memzuç Risk Payı (%)'
# f = 'Tahsis Sektör Adı'
# ff = 'DERİ VE DERİ ÜRÜNLERİ'
# # size = 50

# df = df_master.copy()
# df = df[[a, x, y, f]]
# df = df.loc[df[f] == ff, [a, x, y]]
# df = df.sort_values('Bankamız Risk', ascending=False).reset_index(drop=True)
# # for c in [x, y]:
# #     df[c] = df[c] / 1_000_000
# # fig = plt.figure(figsize=(56,56), dpi = 480)

# plt.scatter(df[x], df[y])
# plt.xlabel(x)
# plt.ylabel(y)
# plt.xscale('log')
# # plt.yscale('log')

In [ ]:
# tsa = 'Tahsis Sektör Adı'
# x = 'Nakdi Reeskontlu Toplam'
# m = 'Müşteri No'

# des = 'Değerlendirmeye Esas Segment'
# eds = 'Entegre Derece Skor'
# yz = 'Yapay Zeka Risk Sınıfı'

# br = 'Bankamız Risk'
# bl = 'Bankamız Limit'
# mr = 'Memzuç Risk'
# ml = 'Memzuç Limit'

# bmp = 'Bankamız Memzuç Risk Payı'
# ldo = 'Sektör Limit Doluluk Oranı'

# dfc = pd.DataFrame()
# dfr = pd.DataFrame()
# dfeds = pd.DataFrame()
# dfyz = pd.DataFrame()
# dfmp = pd.DataFrame()
# dfldo = pd.DataFrame()

# dfc[tsa] = sorted(sektor_list)
# dfr[tsa] = sorted(sektor_list)
# dfeds[tsa] = sorted(sektor_list)
# dfyz[tsa] = sorted(sektor_list)
# dfmp[tsa] = sorted(sektor_list)
# dfldo[tsa] = sorted(sektor_list)

# ct = ['TOPLAM']
# rt = ['TOPLAM']
# edst = ['ORTALAMA']
# yzt = ['ORTALAMA']
# mpt = ['ORTALAMA']
# ldot = ['ORTALAMA']

# dfm = df_master[[m, x, tsa, des, eds, yz, br, bl, mr, ml]].copy()
# for s in segment_list:
#     df = dfm.copy()
#     df = df.loc[df[des] == s]

#     dft = df[[tsa, m]].groupby(tsa).count().reset_index()
#     dft.columns = [tsa, s]
#     dfc = pd.merge(dfc, dft, on=tsa)
#     ct.append(dfc[s].sum())

#     dft = df[[tsa, x]].groupby(tsa).sum().reset_index()
#     dft.columns = [tsa, s]
#     dfr = pd.merge(dfr, dft, on=tsa)
#     rt.append(dfr[s].sum())

#     dft = df[[tsa, eds]].groupby(tsa).mean().reset_index()
#     dft.columns = [tsa, s]
#     dfeds = pd.merge(dfeds, dft, on=tsa)
#     edst.append(dft[s].mean())
    
#     dft = df[[tsa, yz]].groupby(tsa).mean().reset_index()
#     dft.columns = [tsa, s]
#     dfyz = pd.merge(dfyz, dft, on=tsa)
#     yzt.append(dft[s].mean())

#     dft = df[[tsa, br, bl, mr, ml]].groupby(tsa).sum().reset_index()
#     dft[bmp] = dft[br] / dft[mr]
#     dft[ldo] = dft[mr] / dft[ml]

#     dfz = dft[[tsa, bmp]].copy()
#     dfz.columns = [tsa, s]
#     dfmp = pd.merge(dfmp, dfz, on=tsa)
#     mpt.append(dft[br].sum() / dft[mr].sum())
    
#     dfz = dft[[tsa, ldo]].copy()
#     dfz.columns = [tsa, s]
#     dfldo = pd.merge(dfldo, dfz, on=tsa)
#     ldot.append(dft[mr].sum() / dft[ml].sum())
    


# dft = dfm[[tsa, eds]].groupby(tsa).mean().reset_index()
# dft.columns = [tsa, 'ORTALAMA']
# dfeds = pd.merge(dfeds, dft, on=tsa)
# edst.append(dfm[eds].mean())

# dft = dfm[[tsa, yz]].groupby(tsa).mean().reset_index()
# dft.columns = [tsa, 'ORTALAMA']
# dfyz = pd.merge(dfyz, dft, on=tsa)
# yzt.append(dfm[yz].mean())


# dft = dfm[[tsa, br, bl, mr, ml]].groupby(tsa).sum().reset_index()
# dft[bmp] = dft[br] / dft[mr]
# dft[ldo] = dft[mr] / dft[ml]

# dfz = dft[[tsa, bmp]].copy()
# dfz.columns = [tsa, 'ORTALAMA']
# dfmp = pd.merge(dfmp, dfz, on=tsa)
# mpt.append(dfm[br].sum() / dfm[mr].sum())

# dfz = dft[[tsa, ldo]].copy()
# dfz.columns = [tsa, 'ORTALAMA']
# dfldo = pd.merge(dfldo, dfz, on=tsa)
# ldot.append(dfm[mr].sum() / dfm[ml].sum())






# dfc.loc[len(dfc)] = ct
# dfr.loc[len(dfr)] = rt    
# dfeds.loc[len(dfeds)] = edst
# dfyz.loc[len(dfyz)] = yzt
# dfmp.loc[len(dfmp)] = mpt
# dfldo.loc[len(dfldo)] = ldot
# dfc['TOPLAM'] = 0
# dfr['TOPLAM'] = 0

# for s in segment_list:
#     dfc['TOPLAM'] += dfc[s]
#     dfr['TOPLAM'] += dfr[s]


# dfr.iloc[:, 1:] /= 1_000_000

# dfrp = dfr.copy()
# dfcp = dfc.copy()

# for c in list(dfcp.columns)[1:]:
#     dfcp[c] /= dfcp['TOPLAM']
#     dfrp[c] /= dfrp['TOPLAM']


# dfc.columns = ['Adet'] + list(dfc.columns)[1:]
# dfr.columns = ['Nakdi Risk (mio TL)'] + list(dfr.columns)[1:]
# dfcp.columns = ['Adet %'] + list(dfcp.columns)[1:]
# dfrp.columns = ['Nakdi Risk %'] + list(dfrp.columns)[1:]
# dfeds.columns = ['Entegre Derece Skor Ortalaması'] + list(dfeds.columns)[1:]
# dfyz.columns = ['Yapay Zeka Risk Sınıfı Ortalaması'] + list(dfyz.columns)[1:]
# dfmp.columns = ['Bankamız Memzuç Risk Payı'] + list(dfmp.columns)[1:]
# dfldo.columns = ['Sektör Limit Doluluk Oranı'] + list(dfldo.columns)[1:]


# b1 = h_stack([dfc, dfr], 2)
# b2 = h_stack([dfcp, dfrp], 2)
# b3 = h_stack([dfeds, dfyz], 2)
# b4 = h_stack([dfmp, dfldo], 2)
# b = v_stack([b1, b2, b3, b4], 2)
# quick_export(b, 'Sektör - Segment Dağılımı v4')


In [ ]:
# xls = pd.ExcelFile(base_path + 'ML\\ML KR202.xlsx')
# df_ml_kr202 = pd.read_excel(xls)
# save_pickle(df_ml_kr202, 'df_ml_kr202')
# xls.close()

In [ ]:
# xls = pd.ExcelFile(base_path + 'ML\\ML İzleme.xlsx')
# df_ml_izleme = pd.read_excel(xls)
# save_pickle(df_ml_izleme, 'df_ml_izleme')
# xls.close()

In [ ]:

# for df in [df_kr202_all, df_proje_all, df_kur_all, df_izleme_all, df_memzuc_all, df_fg_all]:
#     print(list(df.columns))
#     print()

In [ ]:
# dfm = df_master.copy()
# # dfm = dfm.loc[dfm['FST_YR'].notnull()]
# dfss = []
# for s in segment_list:
#     df = dfm.copy()
#     df = df.loc[df['Değerlendirmeye Esas Segment'] == s]
#     dfx = df['NACE Harf'].value_counts().reset_index()
#     dfx = dfx.sort_values('NACE Harf')
#     dfss.append(dfx)

# quick_export(dfss, 'Banka NACE Harf All', segment_list)
# # quick_export(dfss, 'Banka NACE Harf MT Var', segment_list)

# # quick_export(df['NACE Harf'].value_counts().reset_index(), 'Banka NACE Harf MT')
# # quick_export(df_nace, 'NACE')

# Grup Firmasız Limit Risk

In [ ]:
# if os.path.exists(pickle_folder_path + 'df_gflr'):
#     df_gflr = load_pickle('df_gflr')
# else:
#     fn = 'Grup_Firmasız_Limit_Risk- 31.10.2023.xlsx'
#     xls = pd.ExcelFile(base_path + 'ML\\' + fn)
#     df = pd.DataFrame()
    # sheet_list = xls.sheet_names
#     for sheet in sheet_list:
#         print('\t', colored(sheet, 'light_cyan'), 'okunuyor')
#         dfs = pd.read_excel(xls, sheet_name=sheet)
#         df = pd.concat([df, dfs], ignore_index=True) if len(sheet_list) > 1 else dfs

#     df_gflr = df.copy()
#     save_pickle(df_gflr, 'df_gflr')
#     xls.close()

In [ ]:
# df = df_gflr.copy()
# df = df.loc[df['Müşteri Numarası'] == 316753699]
# quick_export(df, 'ASIR DIŞ')

In [ ]:
# dfx = df_gflr[['Müşteri Numarası', 'Firma Unvanı', 'VKN', 'TCKN']]
# dfx.columns = ['Müşteri No', 'Adı', 'VKN', 'TCKN']
# # for c in dfx.columns:
# #     if c!= 'Adı':
# #         dfx[c] = dfx[c].astype(int)
# dfx.to_csv(output_folder_path + 'MNVT.csv', index=False, sep=';', float_format='%d')

In [ ]:
# df_gflr_backup = df_gflr.copy()
# df_gflr['NACE Harf'] = df_gflr[' Nace Kodu'].apply(lambda x: x[:1] if type(x) is str else x)
# df_gflr = df_gflr.rename(columns = {'Müşteri Numarası' :'Müşteri No', 'Firma Unvanı': 'Adı'})

In [ ]:
# fn = '2023.10 Tüm Firmalar - DES NACE 1.csv'
# df_gflr_nace_1 = pd.read_csv(base_path + 'ML\\' + fn, encoding='ANSI', low_memory=False)
# fn = '2023.10 Tüm Firmalar - DES NACE 2.csv'
# df_gflr_nace_2 = pd.read_csv(base_path + 'ML\\' + fn, encoding='ANSI', low_memory=False)

# fn = '2023.10 Tüm Firmalar - N Memzuç 1.csv'
# df_gflr_memzuc_n_1 = pd.read_csv(base_path + 'ML\\' + fn, encoding='ANSI', low_memory=False)
# fn = '2023.10 Tüm Firmalar - N Memzuç 2.csv'
# df_gflr_memzuc_n_2 = pd.read_csv(base_path + 'ML\\' + fn, encoding='ANSI', low_memory=False)

# fn = '2023.10 Tüm Firmalar - GN Memzuç 1.csv'
# df_gflr_memzuc_gn_1 = pd.read_csv(base_path + 'ML\\' + fn, encoding='ANSI', low_memory=False)
# fn = '2023.10 Tüm Firmalar - GN Memzuç 2.csv'
# df_gflr_memzuc_gn_2 = pd.read_csv(base_path + 'ML\\' + fn, encoding='ANSI', low_memory=False)

# fn = 'Ekim EDS.csv'
# df_gflr_eds = pd.read_csv(base_path + 'ML\\' + fn, sep=';', encoding='ANSI', low_memory=False)

# # fn = '2023.10 Tüm Firmalar MT.csv'
# # df_gflr_mali = pd.read_csv(base_path + 'ML\\' + fn, sep=';', encoding='ANSI')
# fn = '2023.10 Tüm Firmalar MT Hatasız.csv'
# df_gflr_mali_h = pd.read_csv(base_path + 'ML\\' + fn, sep=';', encoding='ANSI')
# # fn = '2023.10 Tüm Firmalar MT.csv'
# # df_gflr_mali = pd.read_csv(base_path + 'ML\\' + fn, sep=';', encoding='ANSI')
# # fn = '2023.10 Tüm Firmalar MT Hatasız 2021.csv'
# # # df_gflr_mali_h21 = pd.read_csv(base_path + 'ML\\' + fn, sep=';', encoding='ANSI')
# fn = 'Tüm Firmalar Tüzel.csv'
# df_gflr_tuzel = pd.read_csv(base_path + 'ML\\' + fn, sep=';', encoding='ANSI')

In [ ]:
# fn = 'lgd2export.csv'
# df_memz = pd.read_csv(base_path + 'ML\\' + fn, sep=';', encoding='ANSI')

In [ ]:
# mn = 'Müşteri No'
# des = 'Değerlendirmeye Esas Segment'
# eds = 'Entegre Derece Skor'
# nk = 'NACE Kodu'
# nh = 'NACE Harf'

# sl, sr = 'Sektör Limit', 'Sektör Risk'
# bl, br = 'Bankamız Limit', 'Bankamız Risk'

# bldo = 'Bankamız Limit Doluluk Oranı'
# sldo = 'Sektör Limit Doluluk Oranı'
# bmrp = 'Bankamız Memzuç Risk Payı'
# bmlp = 'Bankamız Memzuç Limit Payı'

# df = pd.concat([df_gflr_nace_1, df_gflr_nace_2], ignore_index = True)
# df = df[['MUSTERI_NO', 'DES', 'NACE']]
# df.columns = [mn, des, nk]
# df = df.loc[df[mn].notnull()]
# df = df.drop_duplicates(keep='first')
# df[des] = df[des].map(lambda x: {'MİKRO': 'İŞLETME'}.get(x, x))
# df['NACE Harf'] = df['NACE Kodu'].apply(lambda x: x[:1] if type(x) is str else np.nan)
# df = df.sort_values(mn).reset_index(drop=True)
# df_gflr_des_nace = df.copy()


# df = pd.concat([df_gflr_memzuc_n_1, df_gflr_memzuc_n_2], ignore_index = True)
# df = df.drop('SNPST_DT', axis=1)
# df.columns = [mn] + [c + ' N' for c in [sl, sr, bl, br]]
# df = df.loc[df[mn].notnull()]
# df = df.sort_values(mn).reset_index(drop=True)
# df = df.drop_duplicates(keep='first')
# df = df.fillna(0)
# df_gflr_memzuc_n = df.copy()


# df = pd.concat([df_gflr_memzuc_gn_1, df_gflr_memzuc_gn_2], ignore_index = True)
# df = df.drop('SNPST_DT', axis=1)
# df.columns = [mn] + [c + ' GN' for c in [sl, sr, bl, br]]
# df = df.loc[df[mn].notnull()]
# df = df.sort_values(mn).reset_index(drop=True)
# df = df.drop_duplicates(keep='first')
# df = df.fillna(0)
# df_gflr_memzuc_gn = df.copy()


# df = df_gflr_memzuc_n.copy()
# df = pd.merge(df, df_gflr_memzuc_gn, on='Müşteri No', how='outer')
# df = df.fillna(0)
# for c in [sl, sr, bl, br]:
#     df[c] = df[c + ' N'] + df[c + ' GN']

# for s in ['', ' N', ' GN']:
#     df[bldo + s] = df[br + s] / df[bl + s]
#     df[sldo + s] = df[sr + s] / df[sl + s]
#     df[bmrp + s] = df[br + s] / df[sr + s]
#     df[bmlp + s] = df[bl + s] / df[sl + s]

# # df = df.loc[df[bl] != 0]

# df_gflr_memzuc = df.copy()

In [ ]:
# # LİMİT VAR
# df = df_gflr.copy()
# df = pd.merge(df, df_gflr_memzuc, on='Müşteri No', how='inner')
# df = pd.merge(df, df_gflr_eds, on='Müşteri No', how='inner')
# df = pd.merge(df.loc[:, ~df.columns.isin([des, nk, nh])], df_gflr_des_nace, on='Müşteri No', how='inner')
# df = df.loc[df[bl] != 0]
# df_gflr_ozet_l = df.copy()

# # LİMİT VAR, NACE DOLU
# df = df.loc[df[nh].notnull()]
# df_gflr_ozet_ln = df.copy()

# # LİMİT VAR, NACE DOLU, HATASIZ 2020+ MT VAR
# df = pd.merge(df, df_gflr_mali_h, on='Müşteri No', how='inner')
# df_gflr_ozet_lnm = df.copy()

# # LİMİT VAR, HATASIZ 2020+ MT VAR
# df = pd.merge(df_gflr_ozet_l, df_gflr_mali_h, on='Müşteri No', how='inner')
# df_gflr_ozet_lm = df.copy()

# # df = pd.merge(df_gflr, df_gflr_mali_h, on='Müşteri No', how='inner')
# # df

In [ ]:
# dga = []
# dgas = ['NACE + LİMİT VAR', 'NACE + LİMİT + MT VAR', '0 RİSK İLK 20', '0 RİSK İLK 20 MT VAR']
# dfn = df_nace[['NACE_KODU', 'NACE_ADI']]
# dfn.columns = ['NACE Harf', 'NACE Adı']

# for dfm in [df_gflr_ozet_ln, df_gflr_ozet_lnm]:
#     df = pd.DataFrame()
#     df[nh] = sorted(dfm[nh].unique())
    
#     df = pd.merge(df, dfn, on=nh, how='left')

#     for s in ['', ' N', ' GN']:
#         cl = [nh] + [c + s for c in [sl, sr, bl, br]]
#         dft = dfm[cl].groupby(nh).sum().reset_index()
#         dft.iloc[:, 1:] /= 1_000_000
#         df = pd.merge(df, dft, on=nh, how='left')
    
#     df.loc[len(df)] = ['GENEL TOPLAM', ''] + list(df.sum())[2:]

#     for s in ['', ' N', ' GN']:
#         df[bldo + s] = df[br + s] / df[bl + s]
#         df[sldo + s] = df[sr + s] / df[sl + s]
#         df[bmrp + s] = df[br + s] / df[sr + s]
#         df[bmlp + s] = df[bl + s] / df[sl + s]

#     tl = []
#     for s, p in zip(['', ' N', ' GN'], ['', 'N. ', 'GN. ']):
#         dft = dfm[[nh, eds, bldo + s]].copy()
#         dft = dft.loc[dft[bldo + s] < 0.5, [nh, eds]]
#         tl.append(dft[eds].mean())
#         dft.columns = [nh, f'Ortalama {eds} ({p}LDO < 50%)']
#         dft = dft.groupby(nh).mean().reset_index()
#         df = pd.merge(df, dft, on=nh, how='left')
#     df.iloc[-1:, -3:] = tl
#     dga.append(df)

# for dfm in df_gflr_ozet_l, df_gflr_ozet_lm:
#     df = dfm.copy()
#     df = df.loc[df[br] == 0]
#     df = df.sort_values(bl, ascending=False).reset_index(drop=True)
#     dga.append(df.head(20))

In [ ]:
# quick_export(dga, 'Tüm Firmalar NACE Bazlı Risk Limit Raw v1', dgas)

In [ ]:
# quick_export(dga)

In [ ]:
# df_gflr = pd.merge(df_gflr, df_gflr_mali, on='Müşteri No', how='inner')

In [ ]:
# fn = '2023.10 Tüm Firmalar MT - DES NACE.xlsx'
# xls = pd.ExcelFile(base_path + 'ML\\' + fn)
# df_gflr_desnace = pd.read_excel(xls)

In [ ]:
# dgdn = df_gflr_desnace[['MUSTERI_NO', 'DES', 'NACE']].copy()
# dgdn.columns = ['Müşteri No', 'Değerlendirmeye Esas Segment', 'NACE Kodu']

# dgdn = dgdn.loc[dgdn['Müşteri No'].isin(df_gflr_mali_h['Müşteri No'])]
# dgdn = dgdn.loc[(dgdn['Değerlendirmeye Esas Segment'].notnull()) & (dgdn['NACE Kodu'].notnull())]
# dgdn['Değerlendirmeye Esas Segment'] = dgdn['Değerlendirmeye Esas Segment'].replace('MİKRO', 'İŞLETME')
# dgdn = dgdn.drop_duplicates(keep='first')
# dgdn['NACE Harf'] = dgdn['NACE Kodu'].apply(lambda x: x[:1])

In [ ]:
# df = dgdn.copy()
# df = df.loc[df['Müşteri No'].isin(df_gflr_mali_h21['Müşteri No'])]
# # df = df.drop_duplicates(keep='first')
# df.shape

In [ ]:
# dfm = dgdn.copy()
# # dfm = dfm.loc[dfm['FST_YR'].notnull()]
# dfsx = []

# dfx = dfm['NACE Harf'].value_counts().reset_index().sort_values('NACE Harf')
# dfsx.append(dfx)

# for s in segment_list:
#     df = dfm.copy()
#     df = df.loc[df['Değerlendirmeye Esas Segment'] == s]
#     dfx = df['NACE Harf'].value_counts().reset_index()
#     dfx = dfx.sort_values('NACE Harf')
#     dfsx.append(dfx)

# quick_export(dfsx, 'Banka NACE Harf All - Tüm (Hatasız)', ['GENEL'] + segment_list)

# PRA Validasyon

In [ ]:
# # Puan Artış Azalış

# cl = ['Yapay Zeka Risk Sınıfı', 'Entegre Derece Skor']
# crs = [4, 11]


# def find_max_cv(df, cs):
#     return max([df[c] for c in cs])

# r = 'Risk Kategorisi'
# rl = ['Çok Düşük', 'Düşük', 'Makul', 'Yüksek']

# mn = 'Müşteri No'
# a = 'Adı'
# c0 = 'Geçen Adet'
# ct = 'Toplam Adet' 
# dfv_list_list = []

# dfpn = df_pra.copy()
# end_term = last_date


# for terms in ['2023.09', '2023.06', '2023.03', '2022.12']:
#     dfpo = load_pickle(f'df_pra_v3_{terms[2:4]}{terms[-2:]}')
#     start_term = terms

#     si = sorted_date_list.index(start_term) + 1
#     ei = sorted_date_list.index(end_term) + 1

#     sorted_date_list_v = sorted_date_list[si: ei]
#     df_wide_v = df_wide[si: ei]

#     dfm = dfpo[[mn, a, r] + cl].copy()
#     dfv_list =[]

#     for dfw, d in zip(df_wide_v, sorted_date_list_v):
#         dfwf = dfw[['Müşteri No'] + [c + ' ' + d for c in cl]]
#         dfm = pd.merge(dfm, dfwf, on='Müşteri No', how='left')


#     for c in cl:
#         dfm = dfm.rename(columns={c: c + ' ' + start_term})
#         cs = [c + ' ' + d for d in sorted_date_list_v]
#         dfm[c + ' Max'] = dfm.apply(find_max_cv, cs=cs, axis=1)
#         dfm[c + ' Fark'] = dfm[c + ' Max'] - dfm[c + ' ' + start_term]

#     for c, cr in zip(cl, crs):
#         cn = c + ' Fark'
#         dfv = pd.DataFrame()
#         dfv[r]  = rl
#         tl = ['GENEL TOPLAM']
#         # for n in sorted(dfm.loc[dfm[cn].notnull(), cn].unique()):
#         for n in [x for x in range(-cr, cr + 1)]:
#             df = dfm[[mn, r, cn]].copy()
#             df = df.loc[df[cn] == n]
#             dft = df[[r, mn]].groupby(r).count().reset_index()
#             dft.columns = [r, n]
#             dfv = pd.merge(dfv, dft, on=r, how='left')
#             tl.append(dfv[n].sum())

#         dfv.loc[len(dfv)] = tl

#         dfv.columns = [c + ' Değişimi'] + list(dfv.columns)[1:]
#         dfv['GENEL TOPLAM'] = [dfv.iloc[i, 1:].sum() for i in range(len(dfv))]
#         dfv_list.append(dfv)
#         dfvp = dfv.copy()
#         # dfvp.iloc[:, 1:] /= dfvp.iloc[-1, -1]
#         for x in list(dfvp.columns)[1:]:
#             dfvp[x] /= dfvp.iloc[:, -1]
#         dfv_list.append(dfvp)
#     dfvv = insert_header(v_stack(dfv_list), f'{start_term} → {end_term}')
#     dfv_list_list.append(dfvv)
# dfv_ozet = h_stack(dfv_list_list)
# quick_export(dfv_ozet, f'PRA v3 Validasyon Fark2')

In [ ]:
# # Bracket

# cl = ['Entegre Derece Skor']
# c = 'Entegre Derece Skor'
# oldss = [[1, 5], [1, 5], [6, 7]]
# newss = [[6, 7], [8, 12], [8, 12]]

# def find_max_cv(df, cs):
#     return max([df[c] for c in cs])

# r = 'Risk Kategorisi'
# rl = ['Çok Düşük', 'Düşük', 'Makul', 'Yüksek']

# mn = 'Müşteri No'
# a = 'Adı'
# c0 = 'Geçen Adet'
# ct = 'Toplam Adet' 
# dfv_list_list = []

# dfpn = df_pra.copy()
# end_term = last_date


# for terms in ['2023.09', '2023.06', '2023.03', '2022.12']:
#     dfpo = load_pickle(f'df_pra_v3_{terms[2:4]}{terms[-2:]}')
#     start_term = terms

#     si = sorted_date_list.index(start_term) + 1
#     ei = sorted_date_list.index(end_term) + 1

#     sorted_date_list_v = sorted_date_list[si: ei]
#     df_wide_v = df_wide[si: ei]

#     dfm = dfpo[[mn, a, r] + cl].copy()
#     dfv_list =[]

#     for dfw, d in zip(df_wide_v, sorted_date_list_v):
#         dfwf = dfw[['Müşteri No'] + [c + ' ' + d for c in cl]]
#         dfm = pd.merge(dfm, dfwf, on='Müşteri No', how='left')


#     for c in cl:
#         dfm = dfm.rename(columns={c: c + ' ' + start_term})
#         cs = [c + ' ' + d for d in sorted_date_list_v]
#         dfm[c + ' Max'] = dfm.apply(find_max_cv, cs=cs, axis=1)

#     for olds, news in zip(oldss, newss):
#         cos, coe = olds[0], olds[1]
#         cns, cne = news[0], news[1]
#         co = c + ' ' + start_term
#         cn = c + ' Max'
#         dfv = pd.DataFrame()
#         dfv[r]  = rl
#         tl = ['GENEL TOPLAM']

#         df = dfm[[mn, r, co, cn]].copy()
#         dfa = df.loc[
#             (df[co] >= cos) &
#             (df[co] <= coe) &
#             (df[cn].notnull()),
#             [mn, r, cn]
#         ]
#         dff = dfa.loc[
#             (dfa[cn] >= cns) &
#             (dfa[cn] <= cne),
#             [mn, r]
#         ]

#         dft = dff[[r, mn]].groupby(r).count().reset_index()
#         dft.columns = [r, c0]
#         dfv = pd.merge(dfv, dft, on=r, how='left')
#         tl.append(dfv[c0].sum())

#         dft = dfa[[r, mn]].groupby(r).count().reset_index()
#         dft.columns = [r, ct]
#         dfv = pd.merge(dfv, dft, on=r, how='left')
#         tl.append(dfv[ct].sum())

#         dfv['%'] = dfv[c0] / dfv[ct]
#         tl.append(dfv[c0].sum() / dfv[ct].sum())


#         dfv.loc[len(dfv)] = tl

#         dfv.columns = [f'{c} {cos}-{coe} → {cns}-{cne}'] + list(dfv.columns)[1:]
#         dfv_list.append(dfv)
#     dfvv = insert_header(v_stack(dfv_list), f'{start_term} → {end_term}')
#     dfv_list_list.append(dfvv)
# dfv_ozet = h_stack(dfv_list_list)
# quick_export(dfv_ozet, f'PRA v3 Validasyon xr2')

In [ ]:
# # Threshold Geçiş

# cl = ['Müşteri Sınıfı', 'Gecikme', 'Gecikme Gün', 'Finansal Güçlük', 'Yapay Zeka Risk Sınıfı', 'Entegre Derece Skor']
# ctol = [1, 0, 0, 0, 1, 5]
# ctnl = [2, 1, 10, 1, 2, 6]
# reverse = False

# def find_max_cv(df, cs):
#     return max([df[c] for c in cs])

# r = 'Risk Kategorisi'
# rl = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek', 'Diğer Banka Takip']

# mn = 'Müşteri No'
# a = 'Adı'
# c0 = 'Geçen Adet'
# ct = 'Toplam Adet' 
# dfv_list_list = []

# end_term = last_date

# if reverse:
#     ctol, ctnl = ctnl, ctol

# for terms in ['2023.09', '2023.06', '2023.03', '2022.12']:
#     dfpo = load_pickle(f'df_pra_v4_{terms[2:4]}{terms[-2:]}')
#     start_term = terms

#     si = sorted_date_list.index(start_term) + 1
#     ei = sorted_date_list.index(end_term) + 1

#     sorted_date_list_v = sorted_date_list[si: ei]
#     df_wide_v = df_wide[si: ei]

#     dfm = dfpo[[mn, a, r] + cl].copy()
#     dfv_list =[]

#     for dfw, d in zip(df_wide_v, sorted_date_list_v):
#         dfwf = dfw[['Müşteri No'] + [c + ' ' + d for c in cl]]
#         dfm = pd.merge(dfm, dfwf, on='Müşteri No', how='left')


#     for c in cl:
#         dfm = dfm.rename(columns={c: c + ' ' + start_term})
#         cs = [c + ' ' + d for d in sorted_date_list_v]
#         dfm[c + ' Max'] = dfm.apply(find_max_cv, cs=cs, axis=1)

#     for c, cto, ctn in zip(cl, ctol, ctnl):
#         co = c + ' ' + start_term
#         cn = c + ' Max'
#         dfv = pd.DataFrame()
#         dfv[r]  = rl
#         tl = ['GENEL TOPLAM']

#         df = dfm[[mn, r, co, cn]].copy()
#         if reverse:
#             dfa = df.loc[(df[co] >= cto) & (df[cn].notnull()), [mn, r, cn]]
#             dff = dfa.loc[dfa[cn] <= ctn, [mn, r]]
#         else:
#             dfa = df.loc[(df[co] <= cto) & (df[cn].notnull()), [mn, r, cn]]
#             dff = dfa.loc[dfa[cn] >= ctn, [mn, r]]

#         dft = dff[[r, mn]].groupby(r).count().reset_index()
#         dft.columns = [r, c0]
#         dfv = pd.merge(dfv, dft, on=r, how='left')
#         tl.append(dfv[c0].sum())

#         dft = dfa[[r, mn]].groupby(r).count().reset_index()
#         dft.columns = [r, ct]
#         dfv = pd.merge(dfv, dft, on=r, how='left')
#         tl.append(dfv[ct].sum())

#         dfv['%'] = dfv[c0] / dfv[ct]
#         tl.append(dfv[c0].sum() / dfv[ct].sum())


#         dfv.loc[len(dfv)] = tl

#         dfv.columns = [f'{c} {cto} → {ctn}'] + list(dfv.columns)[1:]
#         dfv_list.append(dfv)
#     dfvv = insert_header(v_stack(dfv_list), f'{start_term} → {end_term}')
#     dfv_list_list.append(dfvv)
# dfv_ozet = h_stack(dfv_list_list)

# quick_export(dfv_ozet, f'PRA v4 Validasyon {now()}')

In [ ]:
# # Diğer banka tahakkuk

# dfl = []

# c = 'Toplam Puan'
# mn = 'Müşteri No'
# x = 'Nakdi Reeskontlu Toplam'
# df = pd.DataFrame()
# tl = ['GENEL TOPLAM']
# df[c] = sorted(dbt[c].unique())
# dft = dbt[[c, mn]].groupby(c).count().reset_index()
# tl.append(dft[mn].sum())
# dft.columns = [c, 'Adet']
# df = pd.merge(df, dft, on=c, how='left')
# dft = dbt[[c, x]].groupby(c).sum().reset_index()
# dft[x] /= 1_000_000
# tl.append(dft[x].sum())
# dft.columns = [c, 'Nakdi Risk (mio TL)']
# df = pd.merge(df, dft, on=c, how='left')
# df.loc[len(df)] = tl
# dfl.append(df)

# c = 'Risk Kategorisi'
# mn = 'Müşteri No'
# x = 'Nakdi Reeskontlu Toplam'
# df = pd.DataFrame()
# tl = ['GENEL TOPLAM']
# df[c] = rl
# dft = dbt[[c, mn]].groupby(c).count().reset_index()
# tl.append(dft[mn].sum())
# dft.columns = [c, 'Adet']
# df = pd.merge(df, dft, on=c, how='left')
# dft = dbt[[c, x]].groupby(c).sum().reset_index()
# dft[x] /= 1_000_000
# tl.append(dft[x].sum())
# dft.columns = [c, 'Nakdi Risk (mio TL)']
# df = pd.merge(df, dft, on=c, how='left')
# df.loc[len(df)] = tl
# dfl.append(df)


# dfq = h_stack(dfl, 2)
# quick_export(dfq, 'DBT Özet')

In [ ]:
# # Aylık Detay

# cl = ['Müşteri Sınıfı', 'Gecikme', 'Gecikme Gün', 'Finansal Güçlük', 'Yapay Zeka Risk Sınıfı', 'Entegre Derece Skor']
# ctol = [1, 0, 0, 0, 1, 5]
# ctnl = [2, 1, 10, 1, 2, 6]
# # cl = ['Müşteri Sınıfı']
# # ctol = [1]
# # ctnl = [2]

# def find_max_cv(df, cs):
#     return max([df[c] for c in cs])

# r = 'Risk Kategorisi'
# rl = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek', 'Diğer Banka Takip']

# mn = 'Müşteri No'
# a = 'Adı'
# c0 = 'Geçen Adet'
# ct = 'Toplam Adet' 
# dfv_list_list = []

# dfpn = df_pra.copy()
# end_term = last_date


# dfv_list =[]
# dfp_list =[]

# for terms in ['2022.12']:
#     dfpo = load_pickle(f'df_pra_v4_{terms[2:4]}{terms[-2:]}')
#     start_term = terms

#     si = sorted_date_list.index(start_term) + 1
#     ei = sorted_date_list.index(end_term) + 1

#     sorted_date_list_v = sorted_date_list[si: ei]
#     df_wide_v = df_wide[si: ei]

#     dfm = dfpo[[mn, a, r] + cl].copy()

#     for dfw, d in zip(df_wide_v, sorted_date_list_v):
#         dfwf = dfw[['Müşteri No'] + [c + ' ' + d for c in cl]]
#         dfm = pd.merge(dfm, dfwf, on='Müşteri No', how='left')
    
#     for c in cl:
#         dfm = dfm.rename(columns = {c: c + ' ' + start_term})

#     for c, cto, ctn in zip(cl, ctol, ctnl):
#         co = c + ' ' + start_term
#         dfv = pd.DataFrame()
#         dfv[r]  = rl
#         tl = ['GENEL TOPLAM']

#         dfa = dfm.loc[(dfm[co] <= cto), [mn, r]]
#         dft = dfa.groupby(r).count().reset_index()
#         dft.columns = [r, ct]
#         dfv = pd.merge(dfv, dft, on=r, how='left')
#         tl.append(dfv[ct].sum())

#         for i, d in enumerate(sorted_date_list_v):
#             cn = c + ' ' + d + ' Max'
#             sorted_date_list_v_c = sorted_date_list_v[:i+1]
#             dfm = dfm.rename(columns={c: c + ' ' + start_term})
#             cs = [c + ' ' + d for d in sorted_date_list_v_c]
#             dfm[cn] = dfm.apply(find_max_cv, cs=cs, axis=1)

#             dfa = dfm.loc[(dfm[co] <= cto) & (dfm[cn] >= ctn), [mn, r]]
#             dft = dfa.groupby(r).count().reset_index()
#             dft.columns = [r, d]
#             dfv = pd.merge(dfv, dft, on=r, how='left')
#             tl.append(dfv[d].sum())


#         dfv.loc[len(dfv)] = tl
#         dfv.columns = [f'{c} {cto} → {ctn}'] + list(dfv.columns)[1:]

#         dfp = dfv.copy()
#         for x in list(dfp.columns)[2:]:
#             dfp[x] /= dfp[ct]

#         dfv_list.append(dfv)
#         dfp_list.append(dfp)

# dfvh = insert_header(v_stack(dfv_list), f'{start_term} → {end_term}')
# dfvp = insert_header(v_stack(dfp_list), f'{start_term} → {end_term}')
# dfv_ozet = h_stack([dfvp, dfvh])

# quick_export(dfv_ozet, f'PRA v4 Validasyon {now()}')